# Laparoscopic Semantic Segmentation - Training

**Purpose**: train and checkpoint the segmentation model used throughout the dissertation.

**Hypothesis this feeds**: baseline failures are concentrated in identifiable conditions (small/thin tools, smoke/blood/poor exposure) rather than spread evenly, and targeted interventions in those conditions should outperform optimising the overall average. See `01_baseline_analysis/01a_baseline_failure_analysis.ipynb` for the evidence base.

**Model**: U-Net + ResNet-34 encoder (ImageNet-pretrained), binary instrument-vs-background segmentation.

**Data**: Pascal VOC segmentation layout (CVAT export), 5-fold cross-validation with video-level (group-aware) splits to prevent leakage.

**Framework**: PyTorch + `segmentation_models_pytorch`.

**Input**: GCS buckets (`trainingFrames/`) and a Google Sheet (`SHEET_ID`) for metadata.
**Output**: checkpoints, predicted masks, and metrics saved to `training-checkpoints-<KCL_ID>/`. See `README.md` for full schema.

## Key configuration
This notebook is config-driven: settings most likely to change between runs live in the config cell. Important toggles:

- `ARCHITECTURE` / `ENCODER` - model swap (`Unet`/`resnet34` default; also `UnetPlusPlus`, `FPN`, `DeepLabV3Plus` / `resnet50`, `efficientnet-b4`, `mit_b2`)
- `LOSS` - `bce` | `bce_dice` | `bce_tversky`
- `RUN_CV` / `RUN_SWEEP` / `RUN_POWER` / `RUN_THRESHOLD_REEVAL` / `RUN_AUG_SWEEP` / `RUN_AUG_ABLATION` / `RUN_STATIC_TILING` / `RUN_SYNTHETIC_AUG` / `RUN_TRACKING_CASCADE` / `RUN_ROI_FINETUNE` - which stages this execution runs
- `CASCADE_VARIANT` - which cascade arm runs when `RUN_TRACKING_CASCADE` is on: `padded` | `protected` | `fused` | `fused_redetect`
- `CASCADE_WEIGHTS_FROM` - which finished run's checkpoints a test-time arm re-scores; `FOLDS_JSON` keeps every arm on identical splits and should not be changed
- `EXPORT_PRED_MASKS` - write per-frame predicted masks to GCS for `01b`'s gallery
- `MANUAL_TAG` - human-readable tag folded into `RUN_NAME`/`RUN_ID` for this arm

^ Don't forget to update `MANUAL_TAG` in the config cell before running a new experiment!

### Baseline configuration

Baseline settings:

- `ARCHITECTURE = "Unet"`, `ENCODER = "resnet34"`
- `LOSS = "bce"`
- No tracking-based crop - resized to `IMAGE_HEIGHT` × `IMAGE_WIDTH` (512×512)
- No targeted synthetic augmentation - some generic augmentation is applied (flip, brightness/contrast, Gaussian noise, elastic transform). Without this, the baseline would not be a fair or strong model given the limited number of frames.
- `MANUAL_TAG = "baseline"` - this does not affect the model; look for `baseline-seed2026_cv_Unet_resnet34_bce_lr1e-04_p20/20260812-205137/` in the GCS bucket.

### Experiment table

Record of arms tested in this notebook. Keep it up to date as new arms are added.

The "Config toggle" column is the config variable that switches that arm on.

Before running an experiment, update `MANUAL_TAG` in the config cell to match the arm name.

| Arm | Loss | Key change | Config toggle | Status |
|---|---|---|---|---|
| Baseline | BCE | - | default config, no toggle needed | complete |
| Tracking-based crop | BCE | crop to tracked ROI for effective resolution | `RUN_TRACKING_CASCADE` + `CASCADE_VARIANT` | complete |
| Weighted BCE-Tversky | BCE-Tversky | explicit recall/precision balance | `LOSS = "bce_tversky"` | complete |
| Synthetic augmentation | BCE | smoke/blood/exposure augmentation injected in training | `RUN_SYNTHETIC_AUG` + `RUN_AUG_SWEEP` (mini-sweep) | complete |
| Static tiling | BCE | native-resolution 512x512 tiles instead of a whole-frame resize | `RUN_STATIC_TILING` | complete |
| ROI-crop fine-tune | BCE | short fine-tune on GT-centred ROI crops, so the encoder has seen the zoomed geometry the cascade feeds it | `RUN_ROI_FINETUNE` (section 12e) | complete |



**Fix for CUDA path (only needed for Colab Enterprise)**


If PyTorch reports no CUDA device even though an NVIDIA GPU is present, the driver libraries may not be on the library path. Run the cell below to add the standard NVIDIA library locations before importing torch.


In [ ]:
!LD_LIBRARY_PATH=/usr/local/nvidia/lib64 /usr/local/nvidia/bin/nvidia-smi


In [ ]:
!echo 'export LD_LIBRARY_PATH=/usr/local/nvidia/lib64:$LD_LIBRARY_PATH' >> ~/.bashrc
!echo '/usr/local/nvidia/lib64' >> /etc/ld.so.conf.d/nvidia.conf
!ldconfig


In [ ]:
import torch
print(torch.cuda.is_available())


**Cleanup**

Due to cost limits, notebook storage is tight and frequent reruns fill it quickly. This cell clears the notebook's temporary files.

!! DO NOT LEAVE `WIPE_DATA_CACHE` OR `WIPE_CHECKPOINTS` ON !!


In [ ]:
import shutil, os, glob

WIPE_DATA_CACHE  = False   # /content/lap_data      - costs a full re-download from GCS
WIPE_CHECKPOINTS = False   # /content/checkpoints   - costs the folds already trained

if WIPE_DATA_CACHE:
    shutil.rmtree("/content/lap_data", ignore_errors=True)
if WIPE_CHECKPOINTS:
    shutil.rmtree("/content/checkpoints", ignore_errors=True)
shutil.rmtree("last_cell_figs", ignore_errors=True)

# To add more files to wipe, add them to the list below. Supports wildcards!
for pattern in [
    "/tmp/fg_fraction*.np[yz]",
    "/tmp/*.json",
    "/tmp/*.png",
    "/tmp/*.csv",
    "last_cell_figs.zip",
]:
    for f in glob.glob(pattern):
        try:
            os.remove(f)
        except FileNotFoundError:
            pass

print(f"Cleaned notebook temp files "
      f"(wipe data={WIPE_DATA_CACHE}, wipe checkpoints={WIPE_CHECKPOINTS}).")


## 1. Install dependencies
Run the cell below to install the packages required for this notebook.

Note: `torch` and `opencv` are not installed here because Google Colab Enterprise runtimes ship with both. The CUDA path cell at the top already configures the driver libraries; reinstalling torch can make the notebook silently fall back to CPU.


In [ ]:
%pip install -q \
    segmentation-models-pytorch \
    "albumentations>=2.0" \
    google-cloud-storage \
    matplotlib tqdm

import albumentations, segmentation_models_pytorch, torch, cv2
import numpy as _np
print("=" * 60)
print(f"albumentations : {albumentations.__version__}")
print(f"smp            : {segmentation_models_pytorch.__version__}")
print(f"torch          : {torch.__version__}")
print(f"cv2            : {cv2.__version__}")
print(f"numpy          : {_np.__version__}")
print(f"cuda available : {torch.cuda.is_available()}")
print("=" * 60)


## 2. Configuration
Change model settings in the config cell below.

The final cell locks the RNG so the GPU run is deterministic given `SEED`.


In [ ]:
# === Run types ===
# == Standard ==
RUN_SWEEP = False
RUN_CV = True
RUN_POWER = False

# == Experimental ==
RUN_LOSS_BALANCE_SWEEP = False
RUN_AUG_SWEEP = False
RUN_AUG_ABLATION = False
RUN_SYNTHETIC_AUG = False
RUN_THRESHOLD_REEVAL = False
RUN_STATIC_TILING = False
RUN_TRACKING_CASCADE = False
RUN_ROI_FINETUNE = True
EXPORT_PRED_MASKS = True

# === Data ===
SHEET_ID = "REPLACE_SHEET_ID"
GCS_BUCKET = "laparoscopic-seg-REPLACE_K_NUMBER"
DATA_ROOT = "/content"
FOREGROUND_LABEL = "instrument"
IMAGE_HEIGHT = 512
IMAGE_WIDTH = 512

# === Model ===
ARCHITECTURE = "Unet"  # "Unet" | "UnetPlusPlus" | "FPN" | "DeepLabV3Plus"
ENCODER = "resnet34"  # "resnet34" | "resnet50" | "efficientnet-b4" | "mit_b2"
ENCODER_WEIGHTS = "imagenet"  # "imagenet" | None

# === Training ===
BATCH_SIZE = 8
LR = 1e-4
NUM_EPOCHS = 100
PATIENCE = 20
MIN_DELTA = 1e-4

NUM_WORKERS = 8
USE_AMP = True

NOTOOL_SAMPLE_SHARE = 0.20

# === Loss ===
LOSS = "bce"  # "bce" | "bce_dice" | "bce_tversky"

BCE_DICE_LAMBDA = 0.5

TVERSKY_ALPHA = 0.3
TVERSKY_BETA  = 0.7
BCE_TVERSKY_LAMBDA = 0.5

# === Tracking cascade (sections 7b, 7c, 13b) ===
CASCADE_VARIANT = "fused_redetect"  # "padded" | "protected" | "fused" | "fused_redetect"
CASCADE_PAD_FRAC = 0.18
TRACK_HORIZON_S = 2.0
TRACK_DOWNSCALE = 0.5
TEMPLATE_MATCH_MIN = 0.55
CASCADE_SQUARE_ROI = True
CASCADE_MIN_BOX_PX = 16
CASCADE_MIN_TRACK_PX = 12
CASCADE_MAX_ROI_FRAC = 0.90
CASCADE_COMBINE = "max"  # "replace" | "max" | "blend"
CASCADE_BLEND_W = 0.5
CASCADE_MAX_ZOOM = 2.5
CASCADE_MAX_ROIS = 3
CASCADE_MIN_COMPONENT_PX = 64
CLIPS_GCS = f"gs://{GCS_BUCKET}/temporalTrainingFrames"
ROI_FT_EPOCHS = 50
ROI_FT_P = 0.5
ROI_FT_PAD_RANGE = (0.15, 0.60)
ROI_FT_NEG_P = 0.3
ROI_FT_SMALL_W = 1.0

# === Evaluation ===
PRIMARY_THRESHOLD = 0.50
EVAL_THRESHOLDS = [0.10, 0.20, 0.30, 0.40, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95]

# === Figure palette ===
FIG_PRIMARY = "red"
FIG_SECONDARY = "#6c757d"
FIG_RULE = "#333333"
FIG_SERIES = ["red", "#457b9d", "#f4a261", "#2a9d8f", "#9c6ade", "#c45c26", "#1d3557"]

# === LR sweep ===
LR_GRID = [3e-3, 1e-3, 3e-4, 1e-4, 3e-5]
LR_SOURCE = "fixed to baseline sweep winner, RUN_ID 20260812-205137, fold 0, not re-swept here"

# === Loss balance sweep ===
TVERSKY_BETA_GRID = [0.5, 0.6, 0.7, 0.8, 0.9]

# === Synthetic augmentation (section 12c) ===
AUG_LEVELS = {
	"safe": {
		"motion_blur": 25, "saturation": 1.2, "fog_coef": 0.5, "alpha_coef": 0.2,
		"brightness_limit": 0.3, "contrast_limit": 0.2,
		"elastic_alpha": 20.0, "elastic_sigma": 25.0,
	},
	"extreme": {
		"motion_blur": 45, "saturation": 1.5, "fog_coef": 0.9, "alpha_coef": 0.2,
		"brightness_limit": 0.4, "contrast_limit": 0.2,
		"elastic_alpha": 160.0, "elastic_sigma": 25.0,
	},
}
AUG_P_LOW = 0.05
AUG_P_STD = 0.15
AUG_PRESETS = {
	"low":    {"level": "safe",    "p": AUG_P_LOW},
	"medium": {"level": "safe",    "p": AUG_P_STD},
	"high":   {"level": "extreme", "p": AUG_P_STD},

	"fatal":  {"level": "safe", "p": AUG_P_LOW, "augs": ("RandomFog", "RandomBrightnessContrast")},

	"low_additive": {"level": "safe", "p": AUG_P_LOW, "extend": True,
	                 "augs": ("MotionBlur", "ColorJitter", "RandomFog")},
}
AUG_PRESET = "low_additive"
AUG_ABLATION_LEVEL = "safe"
AUG_ABLATION_P = AUG_P_LOW
AUG_ABLATION_VARIANTS = {
	"baseline": {
		"label": "baseline",
		"augs": (),
	},
	"high_vol": {
		"label": "high-vol risk",
		"augs": ("MotionBlur", "ColorJitter"),
	},
	"fatal": {
		"label": "fatal risk",
		"augs": ("RandomFog", "RandomBrightnessContrast"),
	},
}

# === Threshold re-eval (section 15) ===
THRESHOLD_REEVAL_ARMS = {
	"bce_baseline":  "gs://training-checkpoints-REPLACE_K_NUMBER/experiments/baseline-seed2026_cv_Unet_resnet34_bce_lr1e-04_p20/20260812-205137",
	"tversky_b0.70": "gs://training-checkpoints-REPLACE_K_NUMBER/experiments/tversky2_cv_balsweep_Unet_resnet34_bce_tversky_lr1e-04_p20/20260808-013552",
}
THRESHOLD_REEVAL_REFERENCE = "bce_baseline"
REEVAL_THRESHOLDS = sorted(set(EVAL_THRESHOLDS + [0.80, 0.90, 0.95]))

# === Reproducibility ===
SEED = 2026

GCS_RUN_BASELINE = "gs://training-checkpoints-REPLACE_K_NUMBER/experiments/baseline-seed2026_cv_Unet_resnet34_bce_lr1e-04_p20/20260812-205137"
GCS_RUN_ROIFT = "gs://training-checkpoints-REPLACE_K_NUMBER/experiments/roift-neg-clean_cv_roift_Unet_resnet34_bce_lr1e-04_p20/20260814-105438"
GCS_RUN_TVERSKY = "gs://training-checkpoints-REPLACE_K_NUMBER/experiments/tversky2_cv_balsweep_Unet_resnet34_bce_tversky_lr1e-04_p20/20260808-013552"
FOLDS_JSON = f"{GCS_RUN_BASELINE}/folds.json"

TOP_N = 3

# === Power analysis ===
POWER_SIZES = [500, 1000, 1500, 2000, 2774]
POWER_FOLD = 0

# === Discord progress pings ===
DISCORD_WEBHOOK_URL = "REPLACE_DISCORD_WEBHOOK_URL"

print("=" * 60)
_lr_note = " (will be replaced by the sweep winner)" if (RUN_SWEEP and RUN_CV) else ""
print(f"LR             : {LR:.0e} ({LR_SOURCE}){_lr_note}")
print(f"DISCORD pings  : {'on' if DISCORD_WEBHOOK_URL else 'off'}")
_thr_note = (f"  ({len(THRESHOLD_REEVAL_ARMS)} arm(s), ref={THRESHOLD_REEVAL_REFERENCE})"
             if RUN_THRESHOLD_REEVAL else "")
print(f"THRESH re-eval : {'on' if RUN_THRESHOLD_REEVAL else 'off'}{_thr_note}")
_aug_note = (f"  (preset={AUG_PRESET}, sweep={'on' if RUN_AUG_SWEEP else 'off'},"
             f" ablation={'on' if RUN_AUG_ABLATION else 'off'})"
             if RUN_SYNTHETIC_AUG or RUN_AUG_SWEEP or RUN_AUG_ABLATION else "")
print(f"SYNTH aug      : {'on' if RUN_SYNTHETIC_AUG else 'off'}{_aug_note}")
_tile_note = f"  (tile={IMAGE_HEIGHT}x{IMAGE_WIDTH})" if RUN_STATIC_TILING else ""
print(f"STATIC TILING  : {'on' if RUN_STATIC_TILING else 'off'}{_tile_note}")
_casc_note = (f"  (variant={CASCADE_VARIANT}, pad={CASCADE_PAD_FRAC:.0%}, "
              f"horizon={TRACK_HORIZON_S:.1f}s, fallback=resize"
              + (f", combine={CASCADE_COMBINE}, max_zoom={CASCADE_MAX_ZOOM:.1f}x, "
                 f"max_rois={CASCADE_MAX_ROIS}" if CASCADE_VARIANT.startswith("fused") else "")
              + ")"
              if RUN_TRACKING_CASCADE else "")
print(f"TRACK CASCADE  : {'on' if RUN_TRACKING_CASCADE else 'off'}{_casc_note}")
if RUN_TRACKING_CASCADE and RUN_STATIC_TILING:
    print("WARNING        : tiling and the cascade are both on, so training uses native tiles "
          "while evaluation crops ROIs - two interventions at once, not a clean contrast")
print("=" * 60)


In [ ]:
# === Checkpointing ===
import os, time
CHECKPOINT_DIR = f"{DATA_ROOT}/checkpoints"	# a per-run subdirectory is assigned below, once RUN_NAME and RUN_ID exist
CKPT_BUCKET = "training-checkpoints-REPLACE_K_NUMBER"
MANUAL_TAG = "roift-long-seed2026"	# configurable tag folded into RUN_NAME for this arm

active_runs = []
if RUN_CV:
    active_runs.append("cv")
if RUN_SWEEP:
    active_runs.append("sweep")
if RUN_LOSS_BALANCE_SWEEP and LOSS == "bce_tversky":
    active_runs.append("balsweep")
if RUN_AUG_SWEEP:
    active_runs.append("augsweep")
if RUN_AUG_ABLATION:
    active_runs.append("augablate")
if RUN_SYNTHETIC_AUG:
    active_runs.append(f"synaug-{AUG_PRESET}")
if RUN_POWER:
    active_runs.append("power")
if RUN_THRESHOLD_REEVAL:
    active_runs.append("thrreeval")
if RUN_TRACKING_CASCADE:
    active_runs.append(f"cascade-{CASCADE_VARIANT}")
if RUN_ROI_FINETUNE:
    active_runs.append("roift")
run_types_str = "_".join(active_runs) if active_runs else "standard"
full_tag = f"{MANUAL_TAG}_{run_types_str}" if MANUAL_TAG else run_types_str
_lr_tag = "lrsweep" if (RUN_SWEEP and RUN_CV) else f"lr{LR:.0e}"	# if the sweep is being run and the winner is adopted, the LR tag is lrsweep, otherwise it's the LR used in the config
RUN_NAME = f"{full_tag}_{ARCHITECTURE}_{ENCODER}_{LOSS}_{_lr_tag}_p{PATIENCE}"

# Comment out only when deliberately resuming a disconnected run.
RESUME_RUN_ID = None	# pin an existing RUN_ID to carry on in that run folder; None starts a fresh one
os.environ.pop("RUN_ID", None)
RUN_ID = RESUME_RUN_ID or time.strftime("%Y%m%d-%H%M%S")	# pinning is what lets a killed run resume instead of retraining from scratch
GCS_CKPT_DIR = f"gs://{CKPT_BUCKET}/experiments/{RUN_NAME}/{RUN_ID}"
CHECKPOINT_DIR = f"{CHECKPOINT_DIR}/{RUN_NAME}/{RUN_ID}"	# per run, so two notebooks sharing a machine cannot overwrite each other's best_fold{k}.pt; that is what invalidated both 7 Aug baselines
if RESUME_RUN_ID:
    assert os.system(f"gsutil -q ls {GCS_CKPT_DIR}") == 0, (
        f"RESUME_RUN_ID is set but {GCS_CKPT_DIR} does not exist. RUN_NAME is built from "
        f"MANUAL_TAG and the arm toggles, so this usually means the config does not match the "
        f"run being resumed.")
    print(f"RESUMING       : {GCS_CKPT_DIR}")
RESUME_FROM_GCS = False	# True: load all five checkpoints, skip training entirely (how a re-scoring arm runs, see the next cell). MUST be False for anything that trains, and 12e asserts it
RESUME_INCOMPLETE = True	# skip folds already checkpointed at RUN_ID; give smoke tests a throwaway RUN_ID

print("=" * 60)
print(f"RUN_ID         : {RUN_ID}")
print(f"RUN_NAME       : {RUN_NAME}")
print(f"GCS run dir    : {GCS_CKPT_DIR}")
print(f"local ckpts   : {CHECKPOINT_DIR}")
print(f"RESUME         : incomplete-folds={RESUME_INCOMPLETE}  all-from-gcs={RESUME_FROM_GCS}")
print("=" * 60)


In [ ]:
# === Seed this arm with finished weights when it re-scores ===
CASCADE_WEIGHTS_FROM = GCS_RUN_BASELINE	# GCS_RUN_BASELINE | GCS_RUN_ROIFT | GCS_RUN_TVERSKY; the weights this arm re-scores or fine-tunes from

def _count_ckpts(d):
    return len(os.popen(f"gsutil ls {d}/best_fold*.pt 2>/dev/null").read().split())

if RESUME_FROM_GCS:
    if _count_ckpts(GCS_CKPT_DIR) < 5:
        print(f"copying 5 checkpoints into {GCS_CKPT_DIR} ...")
        !gsutil -m cp {CASCADE_WEIGHTS_FROM}/best_fold*.pt {GCS_CKPT_DIR}/
    _n = _count_ckpts(GCS_CKPT_DIR)
    assert _n == 5, f"expected 5 checkpoints in {GCS_CKPT_DIR}, found {_n}"
    print(f"weights seeded : {_n} folds copied from {CASCADE_WEIGHTS_FROM.rsplit('/', 2)[-2]}")
    print("training       : skipped, this arm only re-scores (RESUME_FROM_GCS=True)")
elif RUN_TRACKING_CASCADE:
    print("training       : RESUME_FROM_GCS is False, so this trains a fresh arm instead of "
          "re-scoring the baseline's weights")
else:
    print("weights        : not seeding, this arm trains its own (RESUME_FROM_GCS=False)")


In [ ]:
# === Discord notifications ===
import hashlib, json, urllib.request

_RUN_COLOURS = (0xE74C3C, 0xE67E22, 0xF1C40F, 0x2ECC71, 0x3498DB, 0x9B59B6)
_EVENT_ICONS = {
    "start":  "[+]",  # beginning
    "epoch":  "[-]",  # epoch progress
    "fold":   "[*]",  # fold done
    "finish": "[=]",  # entire runtime finished / shutting down
    "error":  "[!]",  # failure (incl. shutdown failure)
}
RUN_COLOUR = _RUN_COLOURS[int(hashlib.md5(RUN_ID.encode()).hexdigest(), 16) % len(_RUN_COLOURS)]

def notify(kind, title, lines=None):
    """Send embedded message to Discord."""
    if not DISCORD_WEBHOOK_URL:
        return
    icon = _EVENT_ICONS.get(kind, "⬜")
    desc = "\n".join(str(x) for x in (lines or []) if x is not None)[:4000]
    embed = {
        "title": f"{icon} {title}"[:256],
        "description": desc or None,
        "color": RUN_COLOUR,
        "footer": {"text": f"{MANUAL_TAG} · {RUN_ID}"},
    }
    if embed["description"] is None:
        del embed["description"]
    try:
        req = urllib.request.Request(
            DISCORD_WEBHOOK_URL,
            data=json.dumps({"embeds": [embed]}).encode(),
            headers={"Content-Type": "application/json", "User-Agent": "lap-seg-notify"},
            method="POST",
        )
        urllib.request.urlopen(req, timeout=3).read()
    except Exception as e:
        print(f"[notify] skipped: {type(e).__name__}: {e}")

print(f"DISCORD pings  : {'on' if DISCORD_WEBHOOK_URL else 'off'}  colour=#{RUN_COLOUR:06X}")
notify("start", f"run started · {RUN_NAME}")


In [ ]:
# === Reproducibility ===
import os, random
import numpy as np
import torch

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

def seed(seed: int = SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True	# Prefer a repeatable run over picking the "fastest" algorithm
    torch.backends.cudnn.benchmark = False

torch.use_deterministic_algorithms(True, warn_only=True)
seed(SEED)


## 3. Mount GCS bucket with GCSFuse (Colab Enterprise only)

Colab Enterprise is bound to the signed-in Google account, so authentication is automatic.

GCSFuse is a system binary. Outside Colab Enterprise, install it manually: https://docs.cloud.google.com/storage/docs/cloud-storage-fuse/install


In [ ]:
import os

MOUNT_POINT = "/mnt/gcs"
os.makedirs(MOUNT_POINT, exist_ok=True)

!gcsfuse --implicit-dirs {GCS_BUCKET} {MOUNT_POINT}

# Override data dirs to point at the mount
LOCAL_DATA_DIR = f"{MOUNT_POINT}/trainingFrames"
IMAGES_DIR = f"{LOCAL_DATA_DIR}/JPEGImages"
MASKS_DIR = f"{LOCAL_DATA_DIR}/SegmentationClass"
IMAGESETS_DIR = f"{LOCAL_DATA_DIR}/ImageSets/Segmentation"
LABELMAP_FILE = f"{LOCAL_DATA_DIR}/labelmap.txt"
NOTOOLS_DIR = f"{LOCAL_DATA_DIR}/NoTools"

# Lightweight check to confirm the bucket is mounted correctly
print("=" * 60)
print(f"mount point    : {MOUNT_POINT}")
for path in [IMAGES_DIR, MASKS_DIR, IMAGESETS_DIR, LABELMAP_FILE]:
    assert os.path.exists(path), f"Not found: {path}"
    print(f"ok             : {path}")
print("=" * 60)


## 4. Parse labelmap
Reads `labelmap.txt` to map class names to RGB colours. For semantic (not instance) segmentation, only the foreground class needs a colour mapping. The colour for the label matching `FOREGROUND_LABEL` from the config cell is used for binary mask conversion.


In [ ]:
import numpy as np

def parse_labelmap(path: str) -> dict:
    """Return {label_name: (r, g, b)} from a Pascal VOC segmentation format (`name:r,g,b::`)."""
    label_colours = {}
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            name, rgb_part, *_ = line.split(":")
            r, g, b = [int(x) for x in rgb_part.split(",")]
            label_colours[name] = (r, g, b)
    return label_colours

label_colours = parse_labelmap(LABELMAP_FILE)
print("=" * 60)
for name, (r, g, b) in label_colours.items():
    print(f"\033[48;2;{r};{g};{b}m    \033[0m {name:12s} : ({r}, {g}, {b})")

assert FOREGROUND_LABEL in label_colours, (
    f"'{FOREGROUND_LABEL}' not in labelmap. Available: {list(label_colours)}"
)

FG_COLOUR = np.array(label_colours[FOREGROUND_LABEL], dtype=np.uint8)
print(f"foreground     : '{FOREGROUND_LABEL}' RGB{tuple(FG_COLOUR)}")
print("=" * 60)

# === Helper function ===
def rgb_mask_to_binary(mask_rgb: np.ndarray) -> np.ndarray:
    """Convert the rgb mask to a binary mask where the foreground class is 1 and the background is 0."""
    return np.all(mask_rgb == FG_COLOUR, axis=-1).astype(np.float32)


## 5. Download dataset to local disk
Copy images and masks from GCS to /tmp/lap_data for faster training. Skips the download if files are already cached locally.


In [ ]:
import os, glob
LOCAL_IMAGES_DIR = f"{DATA_ROOT}/lap_data/JPEGImages"
LOCAL_MASKS_DIR = f"{DATA_ROOT}/lap_data/SegmentationClass"
NOTOOLS_LOCAL = f"{DATA_ROOT}/lap_data/NoTools"

def _count(root, ext):
    """Count files in the local cache."""
    return len(glob.glob(f"{root}/**/*.{ext}", recursive=True))

if _count(LOCAL_IMAGES_DIR, "jpg") == 0:
    print("Downloading images...")
    !mkdir -p {LOCAL_IMAGES_DIR}
    !gsutil -m rsync -r gs://{GCS_BUCKET}/trainingFrames/JPEGImages {LOCAL_IMAGES_DIR}
else:
    print(f"images         : already cached ({_count(LOCAL_IMAGES_DIR, 'jpg')})")

if _count(LOCAL_MASKS_DIR, "png") == 0:
    print("Downloading masks...")
    !mkdir -p {LOCAL_MASKS_DIR}
    !gsutil -m rsync -r gs://{GCS_BUCKET}/trainingFrames/SegmentationClass {LOCAL_MASKS_DIR}
else:
    print(f"masks          : already cached ({_count(LOCAL_MASKS_DIR, 'png')})")

if _count(NOTOOLS_LOCAL, "jpg") == 0:
    print("Downloading no-tool frames...")
    !mkdir -p {NOTOOLS_LOCAL}
    !gsutil -m rsync -r gs://{GCS_BUCKET}/trainingFrames/NoTools {NOTOOLS_LOCAL}
else:
    print(f"no-tool        : already cached ({_count(NOTOOLS_LOCAL, 'jpg')})")

# Summarise what landed locally and fail early if images or masks are missing.
n_img, n_msk, n_nt = _count(LOCAL_IMAGES_DIR, "jpg"), _count(LOCAL_MASKS_DIR, "png"), _count(NOTOOLS_LOCAL, "jpg")
print("=" * 60)
print(f"images         : {n_img}")
print(f"masks          : {n_msk}")
print(f"no-tool        : {n_nt}")
print("=" * 60)
assert n_img > 0 and n_msk > 0, "image or mask cache is empty - check the bucket paths"

notool_imgs = sorted(glob.glob(f"{NOTOOLS_LOCAL}/**/*.jpg", recursive=True))

IMAGES_DIR = LOCAL_IMAGES_DIR
MASKS_DIR = LOCAL_MASKS_DIR


## 6. Frame list, metadata, and 5-fold splits
Build the training frame list, join spreadsheet labels, and create video-grouped 5-fold cross-validation splits (no video leakage).


### 6a. Build frame pairs

Reads `default.txt` (Pascal VOC ImageSets) to build image/mask pairs.

Implementation details:
- Missing masks are dropped and reported.
- Missing images fail after merging NoTools/ frames.
- NoTools/ frames are marked as `(image, EMPTY_MASK)` and any that also appear in `default.txt` are dropped as dupes (likely an annotator mistake).


In [ ]:
from pathlib import Path
from collections import Counter

def read_split(name: str) -> list:
    """Read an ImageSets split file and return a list of (image_path, mask_path) tuples."""
    split_file = Path(IMAGESETS_DIR) / f"{name}.txt"
    if not split_file.exists():
        return None
    stems = [l.strip() for l in split_file.read_text().splitlines() if l.strip()]
    return [
        (str(Path(IMAGES_DIR) / (stem + ".jpg")),
         str(Path(MASKS_DIR) / (stem + ".png")))
        for stem in stems
    ]

all_pairs = read_split("default")
if all_pairs is None:
    raise RuntimeError("default.txt not found in ImageSets/Segmentation/")

n_listed = len(all_pairs)
_dropped = [img for img, m in all_pairs if not os.path.exists(m)]
all_pairs = [(img, m) for img, m in all_pairs if os.path.exists(m)]
print(f"default.txt    : {n_listed} listed, {len(_dropped)} dropped (missing mask)")
if _dropped:
    print(f"  e.g.         : {_dropped[:5]}")
    print(f"  note         : confirm known-unannotated frames, not failed mask exports")
assert len(all_pairs) > 0, "no usable pairs - check that default.txt stems match the JPEGImages layout"

EMPTY_MASK = "__EMPTY__"	# Mark for frames with no tools

def frame_key(path):
    """Creates a unique identifier for a frame based on its video ID and stem."""
    parts = str(path).split("/")
    return f"{parts[-2]}/{Path(parts[-1]).stem}"

annotated_keys = {frame_key(img) for img, _ in all_pairs}
dupes = [i for i in notool_imgs if frame_key(i) in annotated_keys]
if dupes:
    print(f"notools dupes  : {len(dupes)} also in default.txt (drops NoTools copy)")
    print(f"  e.g.         : {dupes[:3]}")
    notool_imgs = [i for i in notool_imgs if frame_key(i) not in annotated_keys]

# 3. Warn if NoTools/ has no per-video folders (flat).
notools_parents = set(Path(i).parent.name for i in notool_imgs)
flat = len(notool_imgs) > 0 and notools_parents == {Path(NOTOOLS_LOCAL).name}

if flat:
    annotated_stems = set(Path(img).stem for img, _ in all_pairs)
    same_name = [i for i in notool_imgs if Path(i).stem in annotated_stems]
    print(f"notools layout : flat - videoID/stem dedup cannot match "
          f"({len(same_name)}/{len(notool_imgs)} share a bare filename with an annotated frame)")
    if same_name:
        print(f"  e.g.         : {same_name[:3]}")

notool_pairs = [(img, EMPTY_MASK) for img in notool_imgs]
all_pairs = all_pairs + notool_pairs

img_paths = [img for img, _ in all_pairs]
assert len(set(img_paths)) == len(img_paths), "duplicate image paths in all_pairs"
missing = [p for p in img_paths if not os.path.exists(p)]
assert not missing, f"{len(missing)} images missing on local disk, e.g. {missing[:3]}"

key_counts = Counter(frame_key(p) for p in img_paths)
dup_keys = [k for k, c in key_counts.items() if c > 1]
if dup_keys:
    print(f"dup keys       : {len(dup_keys)} videoID/stem keys appear more than once, e.g. {dup_keys[:3]}")

print("=" * 60)
print(f"pairs          : {len(all_pairs)}")
print(f"tool           : {len(all_pairs) - len(notool_pairs)}")
print(f"no-tool        : {len(notool_pairs)}")
print("=" * 60)


### 6b. Join spreadsheet metadata

Loads the Google Sheet (via configured `SHEET_ID`) and joins `all_pairs` (from section 6a) on the videoID/stem key.

Implementation details:
- Frames with Status deleted/invalid/duplicate/excluded are dropped. (Substrings match so "Deleted (Invalid)" is caught too.)
- Per-frame labels are built for stratified group CV (`video_ids`, tool, challenge, robotic).
- NoTools/ frames are forced tool-absent.
- Fails if any annotated frame misses the sheet join, or if one group holds too large a share of frames.


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

# 1. Load the spreadsheet and match each frame to a key
meta_df = pd.read_csv(f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=csv")
meta_df.columns = meta_df.columns.str.strip()	# sheet headers carry stray spaces
meta_df = meta_df.rename(columns={"Column 1": "Frame ID"})	# rename the first column to "Frame ID"

meta_df["key"] = meta_df["File Name"].map(frame_key)
lookup = meta_df.drop_duplicates("key").set_index("key")

keys = [frame_key(img) for img, _ in all_pairs]

matched = sum(k in lookup.index for k in keys)
print("=" * 60)
print(f"metadata match : {matched}/{len(all_pairs)}")
print(f"pair key       : {keys[0]}")
print(f"sheet key      : {meta_df['key'].iloc[0]}")
print("=" * 60)

# (Helper functions)
def col(k, name, default):
    """Get the value of a column for a given key, returning a default if the key is not found or the cell is empty."""
    if name not in lookup.columns or k not in lookup.index:
        return default
    v = lookup.at[k, name]
    try:
        if pd.isna(v):
            return default
    except (TypeError, ValueError):
        pass
    return v

def truthy(v):
    """Check if a cell is truthy."""
    return str(v).strip().lower() not in ("", "nan", "none", "false", "0", "-", "n/a", "no")

DELETED_STATUSES = ("deleted", "invalid", "duplicate", "excluded")
assert "Status" in lookup.columns, f"no Status column: {list(lookup.columns)}"

def is_retired(k):
    """Check if a frame is marked as deleted / invalid / duplicate / excluded."""
    s = str(col(k, "Status", "")).lower()
    return any(w in s for w in DELETED_STATUSES)

n_before = len(all_pairs)
kept_pairs, kept_keys = [], []
for p, k in zip(all_pairs, keys):
    if not is_retired(k):
        kept_pairs.append(p)
        kept_keys.append(k)
all_pairs, keys = kept_pairs, kept_keys
print("status values  :", lookup["Status"].astype(str).str.lower().value_counts().to_dict())
print(f"retired        : {n_before - len(all_pairs)} frames ({len(all_pairs)} remaining)")

is_notool = np.array([m == EMPTY_MASK for _, m in all_pairs])

def fallback_video(img_path):
    """Use the parent folder as the video ID if the frame has no video ID."""
    return Path(img_path).parent.name

# Use Video ID from the sheet, else the parent folder name
video_ids = np.array([
    str(col(k, "Video ID", fallback_video(img)))
    for k, (img, _) in zip(keys, all_pairs)
])

# True if: tools present (sheet "No Tools" is false).
tool = np.array([
    str(col(k, "No Tools", "false")).strip().lower() != "true"
    for k in keys
])	
tool = tool & ~is_notool	# NoTools/ frames always override.

# True if: Any non-empty Challenges cell (used for fold balancing)
challenge = np.array([truthy(col(k, "Challenges", "")) for k in keys])

chal_raw = np.array([str(col(k, "Challenges", "")).strip() for k in keys])

# True if: Robotic (used for fold balancing)
robotic = np.array([
    str(col(k, "Robotic", "false")).strip().lower() == "true"
    for k in keys
])

n_ann = int((~is_notool).sum()) # Number of annotated frames
ann_matched = sum(k in lookup.index for k, nt in zip(keys, is_notool) if not nt) # Number of annotated frames that joined the sheet
notool_matched = sum(k in lookup.index for k, nt in zip(keys, is_notool) if nt) # Number of NoTools/ frames that joined the sheet

grp = pd.Series(video_ids).value_counts() # Number of frames per video
nt_groups = pd.Series(video_ids[is_notool]).value_counts() # Number of NoTools/ frames per video
n_nt = int(is_notool.sum()) # Number of NoTools/ frames
ann_vids, nt_vids = set(video_ids[~is_notool]), set(video_ids[is_notool]) # Set of video IDs for annotated and NoTools/ frames

print("=" * 60)
print(f"tool           : {tool.sum()}")
print(f"no-tool        : {(~tool).sum()}")
print(f"challenge      : {challenge.sum()}")
print(f"robotic        : {robotic.sum()}")
print(f"ann matched    : {ann_matched}/{n_ann}")
print(f"notool matched : {notool_matched}/{n_nt}")
print(f"groups         : {len(grp)}")
print(f"notool groups  : {pd.Series(video_ids[is_notool]).value_counts().to_dict()}")
print(f"tool-absent    : {n_nt} frames in {len(nt_groups)} groups "
      f"({len(nt_vids & ann_vids)} also carry annotated frames)")
print(f"notool sample  : {notool_imgs[:3]}")
print("=" * 60)

# A check to ensure that all annotated frames have been joined to the sheet
assert ann_matched == n_ann, (
    f"{n_ann - ann_matched} annotated frames couldn't be joined to the sheet (check the key before splitting)"
)

# If there are less than 15 groups, print a warning. 
if len(grp) < 15:
    print(f"warning        : only {len(grp)} groups for 5-fold CV "
          f"(~{len(grp)//5} videos/test fold - fold std mostly video identity)")

assert grp.max() / len(all_pairs) < 0.30, (
    f"group '{grp.idxmax()}' contains {grp.max()/len(all_pairs):.0%} of all frames and cannot be split "
    f"across folds. If this is the flat NoTools/ folder, set NOTOOL_VIDEO_FN in the config cell."
)

assert n_nt == 0 or nt_groups.max() <= 0.30 * n_nt, (
    f"group '{nt_groups.idxmax()}' holds {nt_groups.max()}/{n_nt} tool-absent frames "
    f"and cannot spread across folds. Set NOTOOL_VIDEO_FN in the config cell."
)

if challenge.all() or not challenge.any():
    print("warning        : 'Challenges' is constant across every frame (check column name/contents)")


### 6c. Build instrument size classes

One mask pass over all pairs (cached). Builds per-frame size / thinness labels used for fold balancing and later Phase 5 loss weighting.

Implementation details:
- Mask stats are cached keyed on the exact pair list (and a stats version). Bump the version if the returned statistics change.
- Sheet "No Tools" is reconciled against mask pixels; the mask wins when they disagree (likely an annotator mistake).
- Size buckets (No Tool / Tiny / Small / Medium / Large) come from analysis in `03_tooling/03a_tool_size_distribution.ipynb`.
- Thinness is mean local width at model-input resolution, cut at the 25th percentile of tool-present frames to avoid outliers.


In [ ]:
import cv2, os, hashlib
from tqdm.auto import tqdm
import numpy as np
import pandas as pd

_STATS_VERSION = "v2-width-in-model-px"
_pairs_hash = hashlib.md5(
    ("\n".join(m for _, m in all_pairs) + _STATS_VERSION).encode()).hexdigest()[:12]
_FG_CACHE = f"/tmp/fg_fraction_{_STATS_VERSION}_{_pairs_hash}.npz"

if os.path.exists(_FG_CACHE):
    _cached = np.load(_FG_CACHE)
    fg, thin = _cached["fg"], _cached["thin"]
    print(f"mask stats     : loaded cache ({_FG_CACHE})")
else:
    def _mask_stats(mask_path):
        """Returns the fraction of foreground pixels and the mean inscribed radius compared to the model-input resolution for a mask."""
        if mask_path == EMPTY_MASK:
            return 0.0, 0.0
        raw_mask = cv2.imread(mask_path)
        if raw_mask is None:
            raise RuntimeError(f"unreadable mask (corrupt PNG?): {mask_path}")
        binary_mask = rgb_mask_to_binary(cv2.cvtColor(raw_mask, cv2.COLOR_BGR2RGB))
        frac = float(binary_mask.mean())
        if frac == 0.0:
            return 0.0, 0.0
        mask_u8 = binary_mask.astype(np.uint8)
        radius = float(cv2.distanceTransform(mask_u8, cv2.DIST_L2, 5)[mask_u8 > 0].mean())  # mean inscribed radius (~half local width)
        scale = np.sqrt((IMAGE_HEIGHT * IMAGE_WIDTH) / (binary_mask.shape[0] * binary_mask.shape[1]))  # rescale to model-input px
        return frac, radius * scale

    _stats = [_mask_stats(mask_path) for _, mask_path in
              tqdm(all_pairs, desc="Mask stats", unit="mask")]
    fg = np.array([stat[0] for stat in _stats])
    thin = np.array([stat[1] for stat in _stats])
    np.savez(_FG_CACHE, fg=fg, thin=thin)
    print(f"mask stats     : computed and cached ({_FG_CACHE})")

# 2. Prefer mask over sheet when they disagree on tool presence
has_tool = fg > 0
sheet_only = tool & ~has_tool   # sheet: tool, mask: empty
mask_only = ~tool & has_tool    # sheet: none, mask: non-empty
n_sheet_mask_contradictions = int(sheet_only.sum() + mask_only.sum())
print(f"sheet vs mask  : {n_sheet_mask_contradictions} disagreements "
      f"({int(sheet_only.sum())} sheet=tool/mask empty, "
      f"{int(mask_only.sum())} sheet=none/mask non-empty)")
if mask_only.any():
    bad = np.where(mask_only)[0]
    print("warning        : sheet says 'No Tools' but mask has instrument pixels "
          "(using mask). Check by hand:")
    for i in bad[:15]:
        print(f"  e.g.         : {keys[i]} ({fg[i] * 100:.2f}% tool pixels)")
    if len(bad) > 15:
        print(f"  e.g.         : ... and {len(bad) - 15} more")
tool = has_tool

# 3. Convert absolute fraction to percentage
fg_pct = fg * 100

size_cls = np.zeros_like(fg_pct, dtype=int)
size_cls[(fg_pct > 0) & (fg_pct < 4.5)] = 1
size_cls[(fg_pct >= 4.5) & (fg_pct < 8.0)] = 2
size_cls[(fg_pct >= 8.0) & (fg_pct < 19.0)] = 3
size_cls[fg_pct >= 19.0] = 4

# 5. Thinness from mean local width (model-input px); cut = p25 of tool frames
width_px = 2.0 * thin
THIN_CUT = float(np.percentile(width_px[tool], 25)) if tool.any() else 0.0
thin_cls = np.where(~tool, 0, np.where(width_px < THIN_CUT, 1, 2))	# 0 none / 1 thin / 2 wide

# 6. Print the size and thinness class counts
_size_counts = np.bincount(size_cls, minlength=5)
_thin_counts = np.bincount(thin_cls, minlength=3)
print("=" * 60)
print(f"size class     : {_size_counts.tolist()} [NoTool/Tiny/Small/Med/Large]")
print(f"thin cut       : {THIN_CUT:.1f}px at {IMAGE_HEIGHT}x{IMAGE_WIDTH} (p25 of tool frames)")
print(f"thin class     : {_thin_counts.tolist()} [NoTool/Thin/Wide]")
if tool.any():
    print(f"width_px       : p05={np.percentile(width_px[tool], 5):.1f}  "
          f"p50={np.percentile(width_px[tool], 50):.1f}  "
          f"p95={np.percentile(width_px[tool], 95):.1f}")
    _ov = ((size_cls == 1) & (thin_cls == 1)).sum() / max((size_cls == 1).sum(), 1)
    print(f"tiny&thin      : {_ov:.0%} of tiny frames are also thin")
print("=" * 60)

assert (size_cls > 0).sum() == tool.sum(), "size class and tool flag disagree after reconciliation"


### 6d. Leak guard to ensure no video straddles folds

Frames from the same video must stay in the same fold. This section builds a group key for `StratifiedGroupKFold` from the sheet's `Video ID` and each frame's parent directory.

For most frames the sheet `Video ID` is enough. Edge cases remain, such as one operation split into multiple clips with different `Video ID`s.

This section:
- Runs a union-find over sheet `Video ID` and parent directory so same-video frames share a group.
- Skips shared directories (`JPEGImages/`, `NoTools/`) that hold many unrelated videos.
- Optionally scans for near-duplicate frames across groups (on by default; used for the baseline).

When changing this section, avoid over-merging (coarser folds) and under-merging (silent score inflation).


In [ ]:
LEAK_CHECK_PHASH = True	# scan for near-duplicate frames across groups
PHASH_HAMMING = 4	# the number of differing bits (of 64) counts to consider a frame near-duplicate

import re, cv2, numpy as np
from tqdm.auto import tqdm
from pathlib import Path
from collections import Counter, defaultdict

def _normalise(s):
    """Strip a video/dir name down to lowercase alphanumerics so 'Vid_12 ' == 'vid-12'."""
    return re.sub(r"[^a-z0-9]+", "", str(s).strip().lower())

# Union-find: merge labels that evidence says belong to the same real video
_parent = {}

def _find(x):
    """Return the root label (a.k.a parent group) for x."""
    _parent.setdefault(x, x)
    r = x
    while _parent[r] != r:
        r = _parent[r]
    while _parent[x] != r:	# path compression
        _parent[x], x = r, _parent[x]
    return r

def _union(a, b):
    """Merge the groups of a and b so they share one root (a.k.a parent group)."""
    ra, rb = _find(a), _find(b)
    if ra != rb:
        _parent[ra] = rb

# 1. Normalise Video ID / directory keys; skip container dirs
dir_key = np.array([_normalise(Path(img).parent.name) for img, _ in all_pairs])
vid_key = np.array([_normalise(v) for v in video_ids])

dir_n = Counter(dir_key)
dir_ids = defaultdict(set)
for d, v in zip(dir_key, vid_key):
    dir_ids[d].add(v)
container = {d for d in dir_n
             if dir_n[d] > 0.05 * len(all_pairs)
             or len(dir_ids[d]) > 0.05 * len(set(vid_key))}
if container:
    print(f"containers     : {len(container)} skipped as video evidence "
          f"({[str(d) for d in sorted(container)][:5]})")

# 2. Link Video IDs that share a non-shared directory
for d, v in zip(dir_key, vid_key):
    _find(("vid", v))
    if d not in container:
        _union(("vid", v), ("dir", d))

print(f"video ids      : {len(set(video_ids))} raw → {len(set(vid_key))} normalised "
      f"→ {len({_find(('vid', v)) for v in vid_key})} after directory union")

# 3. Optional near-duplicate scan (difference hash); merge cross-group pairs
_dupe_pairs = np.zeros((0, 2), int)
if LEAK_CHECK_PHASH:

    # 3a. Helpers: fingerprint an image, and find pairs within a Hamming distance
    def _dhash(path):
        """Create a fingerprint for an image by shrinking it to a small greyscale thumbnail and comparing adjacent pixels."""
        im = cv2.imread(path, cv2.IMREAD_REDUCED_GRAYSCALE_8)
        if im is None:
            im = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        thumb = cv2.resize(im, (9, 8), interpolation=cv2.INTER_AREA).astype(np.int16)
        brighter_than_left = thumb[:, 1:] > thumb[:, :-1]
        return np.packbits(brighter_than_left), float(thumb.std())

    def _pairs_within_hamming(packed_hashes, frame_idxs, max_dist):
        """Find all pairs of frames whose hashes differ by at most max_dist bits. """
        # One row per frame: 64 columns of 0/1
        bit_vecs = np.unpackbits(np.stack(packed_hashes)[frame_idxs], axis=1).astype(np.float32)
        n_ones = bit_vecs.sum(1)
        found = []
        for start in range(0, len(bit_vecs), 512):
            block = bit_vecs[start:start + 512]
            dist = n_ones[start:start + 512, None] + n_ones[None, :] - 2.0 * (block @ bit_vecs.T)
            rows, cols = np.where(np.rint(dist) <= max_dist)
            # Keep each unordered pair once (upper triangle of the distance matrix)
            once = cols > (start + rows)
            found.append(np.stack([
                frame_idxs[start + rows[once]],
                frame_idxs[cols[once]],
            ], 1))
        return np.concatenate(found) if found else np.zeros((0, 2), int)

    hashes, stds = zip(*(_dhash(img) for img, _ in
                         tqdm(all_pairs, desc="Hashing frames", unit="img")))
    stds = np.array(stds)
    usable = np.where(stds >= 2.0)[0]
    if len(usable) < len(all_pairs):
        print(f"phash skip     : {len(all_pairs) - len(usable)} near-featureless frames excluded")
    # 3c. Find near-duplicate pairs (hashes within PHASH_HAMMING bits)
    _dupe_pairs = _pairs_within_hamming(hashes, usable, PHASH_HAMMING)
    # 3d. Merge pairs that currently sit in different leak-guard groups
    cross = np.array([
        p for p in _dupe_pairs
        if _find(("vid", vid_key[p[0]])) != _find(("vid", vid_key[p[1]]))
    ])
    print(f"near-dupes     : {len(_dupe_pairs)} pairs (Hamming <= {PHASH_HAMMING}), "
          f"{len(cross)} cross-group")
    if len(cross):
        involved = len(set(cross.flatten()))
        for i, j in cross[:5]:
            print(f"  e.g.         : {video_ids[i]}/{Path(all_pairs[i][0]).stem}  ==  "
                  f"{video_ids[j]}/{Path(all_pairs[j][0]).stem}")
        # Too many hits usually means the threshold is loose, not that 5%+ is duplicated
        if involved > 0.05 * len(all_pairs):
            print(f"warning        : {involved / len(all_pairs):.0%} of frames in cross-group "
                  f"pairs - if examples above are not the same shot, lower PHASH_HAMMING")
        for i, j in cross:
            _union(("vid", vid_key[i]), ("vid", vid_key[j]))
        print(f"phash merged   : {involved} frames into shared groups")

roots = [_find(("vid", v)) for v in vid_key]
members = defaultdict(list)
for r, v in zip(roots, video_ids):
    members[r].append(v)
label = {r: Counter(vs).most_common(1)[0][0] for r, vs in members.items()}
video_group = np.array([label[r] for r in roots])

group_counts = Counter(video_group)
print(f"leak groups    : {len(group_counts)} (from {len(set(video_ids))} Video IDs), "
      f"largest={max(group_counts.values())} frames ({max(group_counts.values()) / len(all_pairs):.1%})")
split_ids = [(str(label[r]), len(set(vs))) for r, vs in members.items() if len(set(vs)) > 1]
if split_ids:
    print(f"  multi-id     : {len(split_ids)} group(s) absorbed >1 Video ID, e.g. "
          + ", ".join(f"{n} (+{c - 1})" for n, c in sorted(split_ids, key=lambda t: -t[1])[:5]))
# Check that no group holds more than 30% of the frames
assert max(group_counts.values()) / len(all_pairs) < 0.30, (
    f"group '{max(group_counts, key=group_counts.get)}' holds {max(group_counts.values()) / len(all_pairs):.0%} of frames "
    f"- union collapsed too far for 5-fold CV")
# Ensure there are at least 15 groups for 5-fold CV (3 groups per fold)
assert len(group_counts) >= 15, f"only {len(group_counts)} groups left for 5-fold grouped CV"


### 6e. Stratified 5-fold splits

Combines the previous sections into a 5-fold stratified split. No video (or near-duplicate footage) may straddle folds, and each fold should have a representative mix of size, no-tool, challenge, and robotic frames.

Current implementation:
- Stratifies on tool size, no-tool presence, challenge, and robotic, all aggregated to the group so each group has one label.
- Optionally loads a saved `FOLDS_JSON` so later experiment arms reuse identical splits (use this when comparing arms with different model architectures).
- With so few no-tool frames, the sampler uses `NOTOOL_SAMPLE_SHARE` to oversample them. Changing this can badly skew the balance.
- Checks for leakage of groups and near-duplicate frames across folds.
- Warns when a fold cannot get a representative mix of challenge type and there are too few frames to support a per-fold claim.


In [ ]:
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

video_df = pd.DataFrame({"video": video_group, "fg_pct": fg_pct, "challenge": challenge, "robotic": robotic,
                    "small": (fg_pct > 0) & (fg_pct < 8.0)})

vid_small_frac = video_df.groupby("video")["small"].transform("mean").to_numpy()
vid_notool = video_df.groupby("video")["fg_pct"].transform(lambda s: (s == 0).mean()).to_numpy()
vid_chal = video_df.groupby("video")["challenge"].transform("mean").to_numpy() > 0.5
vid_rob = video_df.groupby("video")["robotic"].transform("mean").to_numpy() > 0.5

vid_size_cls = np.digitize(vid_small_frac, [0.20, 0.45, 0.70])  # 0..3: <20% / 20-45% / 45-70% / ≥70% small-tool frames
vid_nt_cls = (vid_notool > 0.05).astype(int)	# check for no-tool frames in video

strat = 8 * vid_size_cls + 4 * vid_nt_cls + 2 * vid_chal.astype(int) + vid_rob.astype(int)
groups = video_group	# from the leak-guard cell (6d)

vids_per = pd.DataFrame({"stratum": strat, "video": video_group}).drop_duplicates().groupby("stratum").size()
print(f"stratum vids   : {vids_per.to_dict()}")
if vids_per.min() < 5:
    print("warning        : a stratum has <5 videos, so some folds may get few "
          "(or no) examples from that combination - consider fewer factors in `strat`")

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
folds = [test_idx for _, test_idx in sgkf.split(all_pairs, strat, groups)]

if FOLDS_JSON:
    saved = json.load(open(FOLDS_JSON) if not FOLDS_JSON.startswith("gs://")
                      else os.popen(f"gsutil cat {FOLDS_JSON}"))
    key_to_idx = {frame_key(img): i for i, (img, _) in enumerate(all_pairs)}
    folds = [np.array([key_to_idx[k] for k in saved[str(i)]]) for i in range(5)]
    print(f"folds loaded   : {FOLDS_JSON}")

_notool_share = NOTOOL_SAMPLE_SHARE
sample_weight = np.where(tool, (1.0 - _notool_share) / tool.sum(), _notool_share / (~tool).sum())
print(f"presence       : tool={tool.sum()}  no-tool={(~tool).sum()} "
      f"({(~tool).sum() / len(tool):.1%} natural, sampled toward {_notool_share:.0%})")

def make_split(i):
    """Creates a train / val / test split for fold i, returning the train index array and the train / val / test pair lists."""
    test_idx = folds[i]
    val_idx = folds[(i + 1) % 5]
    train_idx = np.concatenate([folds[j] for j in range(5) if j not in (i, (i + 1) % 5)])
    pick = lambda idx: [all_pairs[k] for k in idx]
    return pick(train_idx), pick(val_idx), pick(test_idx), train_idx

# 4. Leak checks using group key and near-duplicate pixels
for i in range(5):
    rest = np.concatenate([folds[j] for j in range(5) if j != i])
    assert not (set(video_group[folds[i]]) & set(video_group[rest])), (
        f"group leak in fold {i}")
if LEAK_CHECK_PHASH and len(_dupe_pairs):
    fold_of = np.empty(len(all_pairs), int)
    for i in range(5):
        fold_of[folds[i]] = i
    straddle = int((fold_of[_dupe_pairs[:, 0]] != fold_of[_dupe_pairs[:, 1]]).sum())
    assert straddle == 0, f"{straddle} near-duplicate frame pairs straddle folds"
print("leak check     : no video leaks across folds "
      "(group key, near-duplicate pixels)")

# 5. Per-fold composition (tool-free, tiny, thin counts; sampler share)
nt_shares = []
for i in range(5):
    test_idx = folds[i]
    train_idx = make_split(i)[3]
    train_weights = sample_weight[train_idx]
    nt_share = train_weights[~tool[train_idx]].sum() / train_weights.sum()
    nt_shares.append(float(nt_share))
    rep_nt = nt_share * len(train_idx) / max(int((~tool[train_idx]).sum()), 1)
    rep_t = (1 - nt_share) * len(train_idx) / max(int(tool[train_idx].sum()), 1)
    print(f"fold {i}        : n={len(test_idx):5d}  tool={tool[test_idx].sum():5d}  "
          f"no-tool={(~tool[test_idx]).sum():4d}  tiny={(size_cls[test_idx] == 1).sum():4d}  "
          f"thin={(thin_cls[test_idx] == 1).sum():4d}  videos={len(set(video_group[test_idx])):3d}  "
          f"| as train: no-tool share {nt_share:.2f}, "
          f"draws/epoch {rep_nt:.1f}x no-tool vs {rep_t:.2f}x tool")

# Warn if the no-tool frames are uneven across folds (threshold is 0.15)
SAMPLER_NT_SHARE = nt_shares
print(f"sampler share  : no-tool min {min(nt_shares):.2f}  max {max(nt_shares):.2f}")
if max(nt_shares) - min(nt_shares) > 0.15:
    print("warning        : no-tool frames uneven across folds - FP metrics may "
          "vary fold-to-fold for reasons unrelated to the model")

cats = sorted({c.strip() for row in chal_raw for c in row.split(",")
               if c.strip() and c.strip().lower() not in ("nan", "none")})
print("challenge cov  : frames per test fold")
for c in cats:
    hit = np.array([c in row for row in chal_raw])
    per = [int(hit[folds[i]].sum()) for i in range(5)]
    flag = "  (too sparse to report per fold)" if min(per) < 20 else ""
    print(f"  {c:34s} n={int(hit.sum()):4d}  per fold: {per}{flag}")

fold_assignment = {str(i): [frame_key(all_pairs[k][0]) for k in folds[i]] for i in range(5)}
with open("/tmp/folds.json", "w") as f:
    json.dump(fold_assignment, f, indent=2)
!gsutil -q cp /tmp/folds.json {GCS_CKPT_DIR}/folds.json
print(f"folds saved    : {GCS_CKPT_DIR}/folds.json")


## 7. Dataset
Defines the PyTorch `Dataset` that loads image/mask pairs from disk and applies the Albumentations pipeline.

Training and validation use different transforms: training needs augmentations; validation stays clean.

No-tool frames use an all-zero mask instead of loading a mask file, since there is none. Vertical flip is skipped because a laparoscopic camera is never upside down. A couple of augmentation strengths are set explicitly rather than left on library defaults, because the defaults were strong enough to erase the thin tools this study cares about. When `RUN_SYNTHETIC_AUG` is on, `build_train_transform(AUG_PRESET)` swaps in the smoke/blood/exposure suite calibrated in `03d_empirical_tuning_tool.ipynb`.


In [ ]:
import cv2
import random
import numpy as np
import torch
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset

def tool_free_box(tool_mask, side, tries=60):
    """A square box of `side` containing no tool pixels, or None if the frame has no room.

    Section 12e negatives. Every crop built around a ground-truth box contains a tool by
    construction, so the model learns P(tool | zoomed crop) close to 1, which is what doubled the
    no-tool false positives on the first fine-tune. Sized at the zoom cap, i.e. the most-zoomed
    end, because that is where the measured precision loss concentrates. Integral image so each
    candidate is an O(1) test rather than O(side^2).
    """
    h, w = tool_mask.shape
    if w <= side or h <= side:
        return None
    integ = cv2.integral(tool_mask.astype(np.uint8))
    for _ in range(tries):
        x, y = random.randrange(0, w - side), random.randrange(0, h - side)
        if integ[y + side, x + side] - integ[y, x + side] - integ[y + side, x] + integ[y, x] == 0:
            return (x, y, side, side)
    return None

class BinarySegDataset(Dataset):
    """PyTorch Dataset for binary tool segmentation - yields one (image, mask) tensor pair per frame."""
    def __init__(self, pairs, transform=None, tile_mode=None, roi_crop_mode=False):
        self.pairs, self.transform, self.tile_mode = pairs, transform, tile_mode
        self.roi_crop_mode = roi_crop_mode
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, idx):
        img_path, mask_path = self.pairs[idx]
        image = cv2.imread(img_path)
        if image is None:
            raise FileNotFoundError(f"Could not read image: {img_path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        if mask_path == EMPTY_MASK:
            mask = np.zeros(image.shape[:2], dtype=np.float32)          # no-tool → empty
        else:
            mask_rgb = cv2.imread(mask_path)
            if mask_rgb is None:
                raise FileNotFoundError(f"Could not read mask: {mask_path}")
            mask = rgb_mask_to_binary(cv2.cvtColor(mask_rgb, cv2.COLOR_BGR2RGB))
        if self.tile_mode:
            if is_tileable(image.shape[0], image.shape[1]):
                x0, y0 = pick_tile_origin(image.shape[0], image.shape[1], mask, self.tile_mode)
                image = image[y0:y0 + IMAGE_HEIGHT, x0:x0 + IMAGE_WIDTH]
                mask  = mask[y0:y0 + IMAGE_HEIGHT, x0:x0 + IMAGE_WIDTH]
            else:
                image = cv2.resize(image, (IMAGE_WIDTH, IMAGE_HEIGHT), interpolation=cv2.INTER_LINEAR)
                mask  = cv2.resize(mask,  (IMAGE_WIDTH, IMAGE_HEIGHT), interpolation=cv2.INTER_NEAREST)
        elif self.roi_crop_mode and random.random() < ROI_FT_P:
            _tool = mask > 0.5
            _side = zoom_capped_min_px(image.shape[1], image.shape[0])
            _crop = None
            if ROI_FT_NEG_P and random.random() < ROI_FT_NEG_P:
                _crop = tool_free_box(_tool, _side)
            if _crop is None:
                _boxes = component_boxes(_tool, CASCADE_MAX_ROIS, CASCADE_MIN_COMPONENT_PX)
                if _boxes:
                    _crop = pad_box(random.choice(_boxes), image.shape[1], image.shape[0],
                                    random.uniform(*ROI_FT_PAD_RANGE), _side)
            if _crop is not None:
                _x, _y, _cw, _ch = _crop
                image = image[_y:_y + _ch, _x:_x + _cw]
                mask  = mask[_y:_y + _ch, _x:_x + _cw]
        if self.transform:
            aug = self.transform(image=image, mask=mask)
            image, mask = aug["image"], aug["mask"].unsqueeze(0)
        return image, mask

_mean = [0.485, 0.456, 0.406]
_std  = [0.229, 0.224, 0.225]

_resize_op = [] if RUN_STATIC_TILING else [A.Resize(IMAGE_HEIGHT, IMAGE_WIDTH)]

def build_train_transform(preset=None, ablation=None):
    """Baseline, full/partial synth preset, or an ablation variant.

    Presets: low / medium / high use the full synth suite from AUG_LEVELS (12c).
    fatal restricts to the ablation (12d) winning op subset at the same level/p -
    a preset spec's "augs" key controls which ops fire; omitted = full suite.
    Ablation variants: baseline is the generic recipe; high_vol / fatal keep safe
    ElasticTransform plus the named subset in AUG_ABLATION_VARIANTS.
    """
    if ablation is not None:
        if ablation not in AUG_ABLATION_VARIANTS:
            raise ValueError(f"Unknown ablation '{ablation}'; choose from {list(AUG_ABLATION_VARIANTS)}")
        if ablation == "baseline":
            return build_train_transform(None)
        lvl = AUG_LEVELS[AUG_ABLATION_LEVEL]
        p = float(AUG_ABLATION_P)
        wanted = set(AUG_ABLATION_VARIANTS[ablation]["augs"]) | {"ElasticTransform"}
        ops = [*_resize_op, A.HorizontalFlip(p=0.5)]
        if "MotionBlur" in wanted:
            ops.append(A.MotionBlur(blur_limit=(lvl["motion_blur"], lvl["motion_blur"]), p=p))
        if "ColorJitter" in wanted:
            ops.append(A.ColorJitter(
                brightness=(1.0, 1.0), contrast=(1.0, 1.0),
                saturation=(lvl["saturation"], lvl["saturation"]),
                hue=(-0.1, 0.1), p=p,
            ))
        if "RandomFog" in wanted:
            ops.append(A.RandomFog(
                fog_coef_range=(lvl["fog_coef"], lvl["fog_coef"]),
                alpha_coef=lvl["alpha_coef"], p=p,
            ))
        if "RandomBrightnessContrast" in wanted:
            ops.append(A.RandomBrightnessContrast(
                brightness_limit=lvl["brightness_limit"],
                contrast_limit=lvl["contrast_limit"], p=p,
            ))
        ops.append(A.GaussNoise(std_range=(0.02, 0.08), p=0.2))
        if "ElasticTransform" in wanted:
            ops.append(A.ElasticTransform(
                alpha=lvl["elastic_alpha"], sigma=lvl["elastic_sigma"], p=p))
        ops += [A.Normalize(mean=_mean, std=_std), ToTensorV2()]
        return A.Compose(ops)
    if preset is None:
        return A.Compose([
            *_resize_op,
            A.HorizontalFlip(p=0.5),
            A.RandomBrightnessContrast(p=0.3),
            A.GaussNoise(std_range=(0.02, 0.08), p=0.2),
            A.ElasticTransform(alpha=40.0, sigma=6.0, p=0.2),
            A.Normalize(mean=_mean, std=_std),
            ToTensorV2(),
        ])
    if preset not in AUG_PRESETS:
        raise ValueError(f"Unknown AUG_PRESET '{preset}'; choose from {list(AUG_PRESETS)}")
    spec = AUG_PRESETS[preset]
    lvl, p = AUG_LEVELS[spec["level"]], float(spec["p"])
    if spec.get("extend"):
        want = set(spec.get("augs", ()))
        extra = []
        if "MotionBlur" in want:
            extra.append(A.MotionBlur(blur_limit=(lvl["motion_blur"], lvl["motion_blur"]), p=p))
        if "ColorJitter" in want:
            extra.append(A.ColorJitter(
                brightness=(1.0, 1.0), contrast=(1.0, 1.0),
                saturation=(lvl["saturation"], lvl["saturation"]), hue=(-0.1, 0.1), p=p))
        if "RandomFog" in want:
            extra.append(A.RandomFog(
                fog_coef_range=(lvl["fog_coef"], lvl["fog_coef"]),
                alpha_coef=lvl["alpha_coef"], p=p))
        base = list(build_train_transform(None).transforms)
        cut = next(k for k, t in enumerate(base) if isinstance(t, A.Normalize))
        return A.Compose([*base[:cut], *extra, *base[cut:]])
    wanted = set(spec.get("augs", ("MotionBlur", "ColorJitter", "RandomFog", "RandomBrightnessContrast")))
    ops = [*_resize_op, A.HorizontalFlip(p=0.5)]
    if "MotionBlur" in wanted:
        ops.append(A.MotionBlur(blur_limit=(lvl["motion_blur"], lvl["motion_blur"]), p=p))
    if "ColorJitter" in wanted:
        ops.append(A.ColorJitter(
            brightness=(1.0, 1.0), contrast=(1.0, 1.0),
            saturation=(lvl["saturation"], lvl["saturation"]),
            hue=(-0.1, 0.1), p=p,
        ))
    if "RandomFog" in wanted:
        ops.append(A.RandomFog(
            fog_coef_range=(lvl["fog_coef"], lvl["fog_coef"]),
            alpha_coef=lvl["alpha_coef"], p=p,
        ))
    if "RandomBrightnessContrast" in wanted:
        ops.append(A.RandomBrightnessContrast(
            brightness_limit=lvl["brightness_limit"],
            contrast_limit=lvl["contrast_limit"], p=p,
        ))
    ops.append(A.GaussNoise(std_range=(0.02, 0.08), p=0.2))
    ops.append(A.ElasticTransform(alpha=lvl["elastic_alpha"], sigma=lvl["elastic_sigma"], p=p))
    ops += [A.Normalize(mean=_mean, std=_std), ToTensorV2()]
    return A.Compose(ops)

train_transform = build_train_transform(AUG_PRESET if RUN_SYNTHETIC_AUG else None)
print(f"train_transform: {'synth-' + AUG_PRESET if RUN_SYNTHETIC_AUG else 'baseline'}")

val_transform = A.Compose([
    *_resize_op,
    A.Normalize(mean=_mean, std=_std),
    ToTensorV2(),
])

tile_infer_transform = A.Compose([
    A.Normalize(mean=_mean, std=_std),
    ToTensorV2(),
])

# === Helper function ===
def seed_worker(worker_id):
    """Give each DataLoader worker its own seed so augmentations stay varied and reproducible.

    Also caps cv2 to one thread per worker process: cv2 defaults to multi-threading
    each call, which fights NUM_WORKERS processes for the same physical cores.
    Measured 1.18x img/s on a g2-standard-8 (8 vCPU / 4 physical cores).
    """
    cv2.setNumThreads(0)
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)
    if hasattr(train_transform, "set_random_seed"):
        train_transform.set_random_seed(worker_seed)

def eval_worker_init(worker_id):
    """Same cv2 thread cap as seed_worker, without touching RNG state since eval transforms have no randomness."""
    cv2.setNumThreads(0)


### 7a. Static tiling helpers (native-resolution tile grid)

Static tiling experiment (`RUN_STATIC_TILING`): instead of resizing a whole 1280x720 or
960x720 frame down to `IMAGE_HEIGHT` x `IMAGE_WIDTH`, which crushes a ~15x15px tool down to
~6x6px before it ever reaches the encoder, extract `IMAGE_HEIGHT` x `IMAGE_WIDTH` tiles
directly from the native-resolution frame.

`tile_origins` places tile origins evenly across a dimension so the grid covers the whole
frame (overlapping where the frame is not an exact multiple of the tile size, never padding
or cropping content away).

Tile choice is tool-aware, not geometric: laparoscopic tools are frequently clamped near a
trocar port at the frame edge, so a naive centre or uniformly-random tile would routinely
crop the tool out and silently convert a tool-present frame into an accidental negative,
for training *and* for the val_iou checkpoint-selection signal. `pick_tile_origin` restricts
candidates to grid tiles that actually contain foreground pixels whenever the frame has any
(a no-tool frame has no tile to be biased towards, so every tile is a valid candidate
there). Train samples one random tile from that candidate set per fetch, so dataset size and
epoch time stay identical to the baseline (patch-style augmentation, not a 5x compute
increase from enumerating every tile every epoch). Val deterministically picks the
highest-tool-coverage candidate (`"best"`), a stable per-frame crop across epochs, cheap,
and guaranteed to show the tool when the frame has one - val_iou is only used for relative
checkpoint selection within a run, not reported as the arm's headline number, so it does not
need to see every region of the frame the way test-time stitching does.

One deliberate trade-off: because tool-present tiles are always biased onto the tool, the
model trains on fewer pure-background crops from tool-present frames than it will see at
test time (stitching still runs every tile, including tool-free ones). No-tool frames still
supply plenty of background-only training signal; this is judged an acceptable trade against
the alternative of routinely training on accidental negatives, which is exactly the failure
mode static tiling exists to fix.

Final test metrics need the whole frame, not one tile, so `stitch_predict` runs every tile
through the model and average-stitches the probabilities back onto a native-resolution
canvas - see section 13a for how that feeds into the reported scores on the same footing as
every other arm.

In [ ]:
import math
import random
from itertools import product

def tile_origins(dim, tile):
    """Evenly spaced tile origins covering [0, dim) with `tile`-sized windows.

    Overlaps where dim is not an exact multiple of tile; never leaves a gap or pads.
    """
    n = math.ceil(dim / tile)
    if n <= 1:
        return [0]
    step = (dim - tile) / (n - 1)
    return [round(i * step) for i in range(n)]

def frame_tile_grid(h, w):
    """(xs, ys) tile origins covering a native h x w frame with IMAGE_HEIGHT x IMAGE_WIDTH tiles.

    Callers must check is_tileable(h, w) first - this asserts rather than handling a
    too-small frame itself, so a missed check fails loudly instead of returning a bogus grid.
    """
    assert h >= IMAGE_HEIGHT and w >= IMAGE_WIDTH, (
        f"frame {h}x{w} is smaller than the tile size {IMAGE_HEIGHT}x{IMAGE_WIDTH}")
    return tile_origins(w, IMAGE_WIDTH), tile_origins(h, IMAGE_HEIGHT)

def is_tileable(h, w):
    """False when a frame is smaller than one tile in either dimension. The dataset is
    mostly 1280x720 / 960x720, but not exclusively (seen: 702x498) - a frame with no
    spare resolution over one tile has nothing for tiling to preserve, so it falls back
    to the same whole-frame resize every other arm already uses."""
    return h >= IMAGE_HEIGHT and w >= IMAGE_WIDTH

def tiles_with_tool(mask, xs, ys):
    """Grid tile origins (x0, y0) whose crop overlaps a foreground pixel in mask.

    Shared by pick_tile_origin (train/val crop choice) and evaluate_tiled's per-frame tile
    coverage diagnostic (section 13a), so the two never define 'tool-containing tile'
    differently.
    """
    return [(x0, y0) for y0 in ys for x0 in xs
            if mask[y0:y0 + IMAGE_HEIGHT, x0:x0 + IMAGE_WIDTH].any()]

def pick_tile_origin(h, w, mask, mode):
    """One (x0, y0) tile origin for a frame, biased towards tiles that actually contain the
    tool: laparoscopic tools are often clamped at a trocar port near the frame edge, so a
    geometry-only pick (naive centre or uniform random over the whole grid) frequently crops
    the tool out and silently turns a tool-present frame into an accidental negative.

    'random' picks uniformly among tool-containing tiles, for train-time variety at no extra
    epoch cost. 'best' deterministically picks the tile with the most tool pixels, for a
    stable val / checkpoint-selection crop. A no-tool frame has no tile to be biased towards,
    so every grid tile is a valid candidate there (and 'best' still picks a fixed one, since
    max() ties break on list order, which is itself fixed by frame_tile_grid).
    """
    xs, ys = frame_tile_grid(h, w)
    all_origins = [(x0, y0) for y0 in ys for x0 in xs]
    candidates = tiles_with_tool(mask, xs, ys) if mask.any() else all_origins
    if mode == "random":
        return random.choice(candidates)
    if mode == "best":
        return max(candidates,
                    key=lambda o: mask[o[1]:o[1] + IMAGE_HEIGHT, o[0]:o[0] + IMAGE_WIDTH].sum())
    raise ValueError(f"Unknown tile_mode '{mode}'")

def stitch_predict(model, image_rgb):
    """Run every tile of a native-resolution frame through model and average-stitch the
    sigmoid probabilities onto a native-resolution canvas. Only used for test-set scoring
    and visualisation, where the full frame (not one tile) has to be judged.

    Falls back to a single whole-frame resize/predict/resize-back-up for a frame smaller
    than one tile (see is_tileable) - still returns a native-(h, w) probability map either
    way, so callers do not need to know which path ran.
    """
    h, w = image_rgb.shape[:2]
    if not is_tileable(h, w):
        resized = cv2.resize(image_rgb, (IMAGE_WIDTH, IMAGE_HEIGHT), interpolation=cv2.INTER_LINEAR)
        tile = tile_infer_transform(image=resized)["image"].unsqueeze(0).to(device)
        with torch.no_grad():
            probs_512 = model(tile).sigmoid().cpu().numpy()[0, 0]
        return cv2.resize(probs_512, (w, h), interpolation=cv2.INTER_AREA)
    xs, ys = frame_tile_grid(h, w)
    origins = list(product(ys, xs))
    tiles = [tile_infer_transform(image=image_rgb[y0:y0 + IMAGE_HEIGHT, x0:x0 + IMAGE_WIDTH])["image"]
             for y0, x0 in origins]
    batch = torch.stack(tiles).to(device)
    with torch.no_grad():
        probs = model(batch).sigmoid().cpu().numpy()[:, 0]
    canvas = np.zeros((h, w), dtype=np.float32)
    counts = np.zeros((h, w), dtype=np.float32)
    for (y0, x0), p in zip(origins, probs):
        canvas[y0:y0 + IMAGE_HEIGHT, x0:x0 + IMAGE_WIDTH] += p
        counts[y0:y0 + IMAGE_HEIGHT, x0:x0 + IMAGE_WIDTH] += 1
    return canvas / counts

print(f"static tiling  : {'on' if RUN_STATIC_TILING else 'off'}")
if RUN_STATIC_TILING:
    print(f"  tile size    : {IMAGE_WIDTH}x{IMAGE_HEIGHT}")
    print("  grid shape   : per-frame, see per_image_results.csv n_tiles / n_tiles_with_tool (13a)")


### 7b. Temporal clip corpus (tracking cascade)

The annotated set is not a video sequence. Each video contributes up to 4 frames, sampled
independently, a median of 117s apart (p25 67s, p75 219s), and `frame_00001` is a global
dataset index rather than a per-video frame number, so sorting by filename is not even
chronological. No frame-to-frame tracker survives a 117s gap in laparoscopic footage: the
camera pose, the tools, and the anatomy have all changed.

The tracking cascade therefore evaluates on a lead-in clip per annotated frame, built by
`03_tooling/03f_temporal_clip_builder.py` from the raw LEMON videos and published to
`temporalTrainingFrames/`. Each clip is the 5.0s of real video immediately before one
annotated frame, at native resolution and native frame rate, so the tracker gets genuine
25-30fps motion to lock onto.

Two properties make the join exact rather than approximate:

- The original sampler extracted every frame at `normalized_position * duration`, and
  `pipeline/sampling_plan.csv` still holds each `normalized_position`. The annotated
  frame's timestamp is recoverable to the millisecond instead of the sheet's 1s-truncated
  `HH:MM:SS`.
- Every clip was verified by pixel-matching its tail against the stored annotated JPEG.
  The matched index is in the manifest as `target_idx`, so nothing here has to assume
  which clip frame the ground-truth mask belongs to. A true match measured 1.4-2.9 mean
  absolute difference (x264 re-encode loss) against 13-34 for a neighbouring frame.

The scored frame is always the stored JPEG from `JPEGImages/`, never the re-encoded clip
frame. The clip drives the tracker up to that point and nothing else, so the model input
being scored is byte-identical to what the baseline, tiling, and augmentation arms saw and
re-encode noise cannot become a between-arm confound.

In [ ]:
import csv, glob, os

CLIPS_LOCAL = f"{DATA_ROOT}/lap_clips"
clip_manifest = {}

if RUN_TRACKING_CASCADE:
    # Same cache-then-skip shape as the dataset download in section 5.
    if len(glob.glob(f"{CLIPS_LOCAL}/**/*.mp4", recursive=True)) == 0:
        print("Downloading lead-in clips (~9 GB)...")
        !mkdir -p {CLIPS_LOCAL}
        !gsutil -m rsync -r {CLIPS_GCS}/clips {CLIPS_LOCAL}
    !gsutil -q cp {CLIPS_GCS}/manifest.csv /tmp/clip_manifest.csv

    with open("/tmp/clip_manifest.csv") as f:
        for row in csv.DictReader(f):
            if row["status"] == "ok":
                clip_manifest[row["key"]] = row

    _keys = {frame_key(img) for img, _ in all_pairs}
    _have = _keys & set(clip_manifest)
    print("=" * 60)
    print(f"clips local    : {len(glob.glob(f'{CLIPS_LOCAL}/**/*.mp4', recursive=True))}")
    print(f"manifest ok    : {len(clip_manifest)}")
    print(f"coverage       : {len(_have)}/{len(_keys)} frames ({len(_have)/len(_keys):.1%})")
    _lead = np.array([float(clip_manifest[k]["lead_actual_s"]) for k in _have])
    print(f"lead-in        : median {np.median(_lead):.2f}s  "
          f"min {_lead.min():.2f}s  under 1s: {int((_lead < 1.0).sum())} frames")
    if len(_have) < len(_keys):
        print(f"WARNING        : {len(_keys) - len(_have)} frames have no clip and will "
              f"be scored full-frame (cascade_has_clip=0 in per_image_results.csv)")
    print("=" * 60)
else:
    print("tracking cascade: off (section 7b skipped)")

def clip_for(img_path):
    """(clip_path, target_idx, fps, lead_s) for one frame, or Nones when it has no clip."""
    row = clip_manifest.get(frame_key(img_path))
    if row is None:
        return None, None, None, None
    path = os.path.join(CLIPS_LOCAL, row["annotator"], row["video_id"],
                        f"frame_{int(row['frame_id']):05d}.mp4")
    if not os.path.exists(path):
        return None, None, None, None
    return path, int(row["target_idx"]), float(row["fps"]), float(row["lead_actual_s"])


### 7c. Tracking cascade helpers (padded ROI crop, 3-tier recovery)

A spatial reading of FARTrack's temporal token sparsification: instead of sparsifying tokens
over time, spend the model's fixed 512x512 input budget on the region that actually contains
the tool. The baseline downsamples a 1280x720 frame to 512x512 and a tool covering 3% of the
frame loses most of its pixels on the way in. Cropping the tool's neighbourhood at native
resolution and resizing that up instead is a hardware-friendly way to raise the tool's
effective resolution without a bigger model or more labels.

This is a test-time procedure. Nothing about training changes, so the arm differs from the
baseline in how a frame becomes model input, not in how the model learned.

**Phases.** Bootstrap a box from a full-frame pass on the first lead-in frame, cache that crop
as the pristine template, track forward to the annotated frame, then predict inside the padded
ROI and paste the result back onto a native-resolution canvas.

The bootstrap box comes from the model's own prediction, not the ground-truth mask. No ground
truth exists at the clip start, and using the annotated frame's GT box would leak the answer
into the arm being measured.

**The 3-tier recovery**, on lock loss or an empty ROI prediction:

1. `cv2.matchTemplate` against the cached pristine template ($t_0$), the cheap recovery for a
   tool that briefly vanished into smoke or blood and came back.
2. On a confident match, snap to it and re-initialise the tracker. Recovery without paying for
   a full-frame pass.
3. Otherwise a full-frame redetection, which re-seeds the box and caches a fresh secondary
   template ($t_{-1}$).

Both triggers are needed, not just lock loss. In testing, a tracker kept reporting a confident
box after the target had left the frame entirely, and only the empty-ROI check caught it.

**Two design choices that are not in the original plan**, both forced by measurement:

- `CASCADE_SQUARE_ROI` makes the ROI isotropic before the square resize. A 5x300 needle box
  stretched straight to 512x512 is deformed 60x more horizontally than vertically, which is a
  far larger distribution shift than the scale change under test. A minimum-size box is grown
  rather than rejected for the same reason: rejecting a 5px-wide box would discard exactly the
  thin tools this arm exists to help.
  cascade stays a one-variable change against the baseline. Routing the fallback through
  `stitch_predict` instead makes the arm a blend of tiling and cascading whenever the fallback
  fires, which it does often, so `"stitch"` is opt-in.

**What the first measured run changed.** The `protected` arm regressed to 0.786 IoU against
the baseline's 0.848, and `02b` localised all of it to the ROI path: -0.146 IoU on the 43% of
frames where it fired, against +0.001 on the fallback. The cause is not the scale shift this
arm was written to test. On those frames precision held at 0.911 against the baseline's 0.926
and stayed flat out to 2x zoom, while recall fell from 0.907 to 0.755, and dropping the
decision threshold from 0.50 to 0.10 recovered only 1173 of the 4864 excess false-negative
pixels. A false negative no threshold can recover is a pixel whose probability is exactly
zero, which is what the paste-back onto a blank canvas paints everywhere outside the box. The
damage also runs the wrong way for a scale effect: -0.241 IoU on large tools against -0.083 on
tiny, because a bigger tool is likelier to extend past a box fitted two seconds earlier.

The `fused` variants answer that with three changes, all test-time:

- **Composite instead of paste-back.** `composite_predict` starts from the full-frame pass and
  lets each ROI act only inside its own box, so the baseline becomes the worst case rather
  than the best case. `CASCADE_COMBINE` picks how the two meet: `max` (an ROI can only add,
  which is the recall claim this hypothesis makes), `blend`, or `replace` for the old behaviour.
- **A zoom cap.** Precision does break eventually, -0.085 in the 2.5-3.5x band and -0.232
  beyond, so `CASCADE_MAX_ZOOM` floors the box side at `max(w, h) / cap`. That floor is handed
  to `pad_box`'s existing `min_px`, so it needs no second geometry path.
- **One ROI per component.** The bootstrap box spanned all predicted foreground, so two tools
  on opposite sides of the frame produced a near-full-frame ROI: `bootstrap_too_large` fired on
  50% of tool frames, which is half the dataset never reaching the mechanism at all.

`fused_redetect` adds one more step, unioning each tracked box with the scored frame's own
detection. It should cover the tool better, but the ROI is then no longer purely
tracking-derived, so it is kept as a separate variant rather than folded in: the pair is what
lets the write-up separate what tracking contributes from what coarse-to-fine refinement does.


**Headroom.** Measured against ground-truth ROIs over 1200 real frames, the linear zoom gain
over the baseline is 2.31x for tiny tools (median 3.2% frame area, so about 5.3x the pixels),
1.48x for small, 1.04x for medium and 1.00x for large, and roughly half of all frames yield an
ROI covering over 90% of the frame and so fall back to full-frame. This is an oracle ceiling
that a real tracker lands below, and it says the mechanism is concentrated on precisely the
weak region the hypothesis names. Expect `iou_tiny` and `iou_thin` to move while mean IoU
barely does; read this arm on the subgroup metrics, not the average.

In [ ]:
import cv2
import numpy as np

def make_tracker():
    try:
        return cv2.TrackerCSRT_create()
    except AttributeError:
        return cv2.legacy.TrackerCSRT_create()

def safe_init(tracker, frame, box):
    """Initialise a tracker, reporting failure instead of raising.

    OpenCV trackers assert rather than return False on a box they cannot work with: MIL
    throws '!posSamples.empty()' for a box with no room to sample around it, which real
    frames produce whenever the detection covers most of the image. One bad frame in 4979
    must not end the run.
    """
    try:
        tracker.init(frame, tuple(int(v) for v in box))
        return True
    except Exception:
        return False

def safe_update(tracker, frame):
    """Update a tracker, mapping an internal assertion to an ordinary lock loss."""
    try:
        ok, raw = tracker.update(frame)
        return bool(ok), raw
    except Exception:
        return False, (0, 0, 0, 0)

def mask_to_bbox(mask):
    """Tight (x, y, w, h) around every foreground pixel, or None when the mask is empty.

    Deliberately the extent of all foreground, not the largest connected component: a
    grasper split into jaw and shaft by an occluding fold is one tool, and cropping to
    one component would discard the other half.
    """
    ys, xs = np.nonzero(mask)
    if len(xs) == 0:
        return None
    x0, x1 = int(xs.min()), int(xs.max())
    y0, y1 = int(ys.min()), int(ys.max())
    return (x0, y0, x1 - x0 + 1, y1 - y0 + 1)

def _grow_to(lo, hi, want, limit):
    """Widen the interval [lo, hi) to `want` about its centre, clamped to [0, limit)."""
    if hi - lo >= want:
        return max(0, lo), min(limit, hi)
    if limit <= want:
        return 0, limit
    c = (lo + hi) / 2.0
    lo = int(round(c - want / 2.0))
    lo = max(0, min(lo, limit - want))
    return lo, lo + want

def pad_box(box, frame_w, frame_h, frac, min_px=1, square=None):
    """Expand a box outward by frac on each side and clamp it inside the frame.

    Clamping is what keeps negative or past-the-edge coordinates out of the crop. The
    padding exists so fine structure at the tool tip is not sliced off by a box fitted
    to the previous frame's extent, which is the target-deformation failure FARTrack
    describes.

    A dimension below min_px is grown to min_px rather than rejected: a needle driver can
    legitimately produce a 5px-wide box, and rejecting it would drop exactly the thin
    tools this arm exists to help. Only a frame genuinely too small yields None.

    square makes the ROI isotropic before it is resized to the model's square input.
    Without it a 5x300 box is stretched 60x more horizontally than vertically, which
    deforms the tool far beyond the scale shift the arm is meant to test.
    """
    if square is None:
        square = CASCADE_SQUARE_ROI
    x, y, w, h = box
    dx, dy = w * frac, h * frac
    x0, y0 = int(np.floor(x - dx)), int(np.floor(y - dy))
    x1, y1 = int(np.ceil(x + w + dx)), int(np.ceil(y + h + dy))

    if square:
        side = max(x1 - x0, y1 - y0)
        x0, x1 = _grow_to(x0, x1, side, frame_w)
        y0, y1 = _grow_to(y0, y1, side, frame_h)

    x0, x1 = _grow_to(x0, x1, max(min_px, x1 - x0), frame_w)
    y0, y1 = _grow_to(y0, y1, max(min_px, y1 - y0), frame_h)
    x0, y0 = max(0, x0), max(0, y0)
    x1, y1 = min(frame_w, x1), min(frame_h, y1)
    out_w, out_h = x1 - x0, y1 - y0
    if out_w < 1 or out_h < 1:
        return None
    return (x0, y0, out_w, out_h)

def _box_ok(box, frame_w, frame_h, min_px):
    if box is None:
        return False
    x, y, w, h = box
    return (w >= min_px and h >= min_px and x >= 0 and y >= 0
            and x + w <= frame_w and y + h <= frame_h)

def full_frame_predict(model, image_rgb):
    """Native-resolution probability map from one whole-frame pass.

    "resize" is the default so the cascade is a one-variable change against the baseline
    arm. "stitch" routes the fallback through the static-tiling path instead, which
    confounds two interventions whenever the fallback fires often, so it is opt-in.
    """
    h, w = image_rgb.shape[:2]
    resized = cv2.resize(image_rgb, (IMAGE_WIDTH, IMAGE_HEIGHT), interpolation=cv2.INTER_LINEAR)
    t = tile_infer_transform(image=resized)["image"].unsqueeze(0).to(device)
    with torch.no_grad():
        probs = model(t).float().sigmoid().cpu().numpy()[0, 0]
    return cv2.resize(probs, (w, h), interpolation=cv2.INTER_LINEAR)

def roi_predict(model, image_rgb, box):
    """Predict inside one ROI and paste the result onto a blank native canvas.

    Resizing the crop up to the model's native input is the whole point of the arm: a
    tool covering 2% of a 1280x720 frame survives a 512x512 crop-and-zoom at far more
    pixels than it does the baseline's whole-frame downsample.
    """
    h, w = image_rgb.shape[:2]
    x, y, bw, bh = box
    crop = image_rgb[y:y + bh, x:x + bw]
    crop_512 = cv2.resize(crop, (IMAGE_WIDTH, IMAGE_HEIGHT), interpolation=cv2.INTER_LINEAR)
    t = tile_infer_transform(image=crop_512)["image"].unsqueeze(0).to(device)
    with torch.no_grad():
        probs_512 = model(t).float().sigmoid().cpu().numpy()[0, 0]
    canvas = np.zeros((h, w), dtype=np.float32)
    canvas[y:y + bh, x:x + bw] = cv2.resize(probs_512, (bw, bh), interpolation=cv2.INTER_LINEAR)
    return canvas

def zoom_capped_min_px(frame_w, frame_h):
    """Smallest ROI side that keeps linear zoom at or under CASCADE_MAX_ZOOM.

    For a square box of side s the arm's zoom is max(w, h) / s, so a cap on zoom is a floor
    on s. Handed straight to pad_box's min_px, which already grows a box about its centre.
    """
    return max(CASCADE_MIN_BOX_PX, int(np.ceil(max(frame_w, frame_h) / float(CASCADE_MAX_ZOOM))))

def component_boxes(binary, max_boxes, min_area):
    """Boxes of the largest connected components, largest first.

    One box spanning all foreground puts two tools on opposite sides of the frame into a
    near-full-frame ROI, which is why bootstrap_too_large fired on half the tool frames.
    Falls back to the whole-mask extent when every component is under min_area, so a tiny
    tool is never dropped by the noise filter.
    """
    n, _, stats, _ = cv2.connectedComponentsWithStats(binary.astype(np.uint8), 8)
    keep = [(int(stats[i, cv2.CC_STAT_AREA]),
             (int(stats[i, cv2.CC_STAT_LEFT]), int(stats[i, cv2.CC_STAT_TOP]),
              int(stats[i, cv2.CC_STAT_WIDTH]), int(stats[i, cv2.CC_STAT_HEIGHT])))
            for i in range(1, n) if stats[i, cv2.CC_STAT_AREA] >= min_area]
    if not keep:
        bb = mask_to_bbox(binary)
        return [bb] if bb is not None else []
    keep.sort(key=lambda t: -t[0])
    return [b for _, b in keep[:max_boxes]]

def _union_box(a, b):
    x0, y0 = min(a[0], b[0]), min(a[1], b[1])
    x1, y1 = max(a[0] + a[2], b[0] + b[2]), max(a[1] + a[3], b[1] + b[3])
    return (x0, y0, x1 - x0, y1 - y0)

def _boxes_overlap(a, b):
    return (a[0] < b[0] + b[2] and b[0] < a[0] + a[2]
            and a[1] < b[1] + b[3] and b[1] < a[1] + a[3])

def merge_detections(tracked, detected, frame_w, frame_h, pad, min_px):
    """Widen each tracked box to cover the detection it overlaps, and keep detections the
    tracker never had.

    A tracked box is the tool as it looked at the start of the lead-in and CSRT barely adapts
    scale, so by the scored frame the box under-covers. This is the fused_redetect step that
    trades tracking purity for coverage.
    """
    out, used = [], set()
    for tb in tracked:
        merged = tb
        for i, db in enumerate(detected):
            if _boxes_overlap(merged, db):
                merged = _union_box(merged, db)
                used.add(i)
        out.append(pad_box(merged, frame_w, frame_h, pad, min_px) or merged)
    for i, db in enumerate(detected):
        if i not in used:
            pb = pad_box(db, frame_w, frame_h, pad, min_px)
            if pb is not None:
                out.append(pb)
    out.sort(key=lambda b: -(b[2] * b[3]))
    keep = [b for b in out if (b[2] * b[3]) / float(frame_w * frame_h) <= CASCADE_MAX_ROI_FRAC]
    if not keep:
        keep = [min(out, key=lambda b: b[2] * b[3])]
    return keep[:CASCADE_MAX_ROIS]

def composite_predict(model, image_rgb, boxes, base_probs):
    """ROI passes laid over a full-frame base, one pass per box.

    The protected arm pasted the ROI onto zeros, so every tool pixel outside the box was a
    false negative no threshold could recover: that alone is its whole -0.146 IoU on ROI
    frames. Here the full-frame pass is the floor and an ROI only acts inside its own box,
    which makes the baseline the worst case rather than the best case.
    """
    out = base_probs.copy()
    for box in boxes:
        x, y, bw, bh = box
        roi = roi_predict(model, image_rgb, box)[y:y + bh, x:x + bw]
        if CASCADE_COMBINE == "max":
            out[y:y + bh, x:x + bw] = np.maximum(out[y:y + bh, x:x + bw], roi)
        elif CASCADE_COMBINE == "blend":
            out[y:y + bh, x:x + bw] = (CASCADE_BLEND_W * roi
                                       + (1.0 - CASCADE_BLEND_W) * out[y:y + bh, x:x + bw])
        else:
            out[y:y + bh, x:x + bw] = roi
    return out

def _scale_box(box, s):
    return (box[0] * s, box[1] * s, box[2] * s, box[3] * s)

def template_match(frame_rgb, template):
    """Tier 1 recovery: best (score, box) for the cached pristine template.

    Cheap by design. It exists so a tool that briefly vanished into smoke and came back
    is recovered without paying for a full-frame pass.
    """
    if template is None:
        return -1.0, None
    th, tw = template.shape[:2]
    fh, fw = frame_rgb.shape[:2]
    if th > fh or tw > fw or th < 2 or tw < 2:
        return -1.0, None
    if float(template.std()) < 1.0:
        return -1.0, None
    res = cv2.matchTemplate(frame_rgb, template, cv2.TM_CCOEFF_NORMED)
    _, score, _, loc = cv2.minMaxLoc(res)
    return float(score), (int(loc[0]), int(loc[1]), tw, th)

def read_clip_frames(clip_path, target_idx, horizon_frames):
    """Lead-in frames strictly before the annotated frame, newest-last, as RGB.

    Decodes forward and keeps only the needed window: seeking by frame index is not
    reliable across every h264 clip, and these clips are short enough that a forward
    decode costs less than getting the seek wrong.
    """
    start = max(0, target_idx - horizon_frames)
    cap = cv2.VideoCapture(str(clip_path))
    frames, i = [], 0
    while i < target_idx:
        ok, frame = cap.read()
        if not ok:
            break
        if i >= start:
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        i += 1
    cap.release()
    return frames

def cascade_predict(model, image_rgb, clip_path, target_idx, fps, variant, diag=None):
    """Tracking-by-detection cascade for one annotated frame.

    Runs the tracker over the frame's lead-in clip, then predicts inside the padded ROI
    the tracker lands on. Returns a native-resolution probability map, so the caller
    scores it exactly like any other arm.

    variant is the progressive ablation:
      base            tracker plus ROI crop, no padding, no recovery
      padded          adds the geometric safety margin
      protected       adds the 3-tier recovery (template memory, re-init, full-frame redetect)
      fused           adds the full-frame composite, the zoom cap, and one ROI per component
      fused_redetect  also unions each tracked box with the scored frame's own detection
    """
    h, w = image_rgb.shape[:2]
    d = diag if diag is not None else {}
    d.setdefault("tier1_hits", 0)
    d.setdefault("tier2_reinits", 0)
    d.setdefault("tier3_redetects", 0)
    d.setdefault("track_losses", 0)
    d.setdefault("n_track_frames", 0)
    fused = variant.startswith("fused")
    protected = (variant == "protected") or fused
    pad = CASCADE_PAD_FRAC
    min_px = zoom_capped_min_px(w, h) if fused else CASCADE_MIN_BOX_PX

    def finish(tier, probs):
        d["final_tier"] = tier
        return probs

    horizon = int(round(TRACK_HORIZON_S * fps)) if fps and fps > 0 else 0
    frames = read_clip_frames(clip_path, target_idx, horizon) if (clip_path and horizon) else []
    d["n_track_frames"] = len(frames)
    if not frames:
        return finish("no_clip", full_frame_predict(model, image_rgb))

    s = TRACK_DOWNSCALE
    def small(fr):
        return cv2.resize(fr, (max(2, int(fr.shape[1] * s)), max(2, int(fr.shape[0] * s))),
                          interpolation=cv2.INTER_AREA) if s != 1.0 else fr

    boot = frames[0]
    boot_probs = full_frame_predict(model, boot)
    boot_bin = boot_probs > PRIMARY_THRESHOLD
    if fused:
        raw_boxes = component_boxes(boot_bin, CASCADE_MAX_ROIS, CASCADE_MIN_COMPONENT_PX)
    else:
        _bb = mask_to_bbox(boot_bin)
        raw_boxes = [] if _bb is None else [_bb]
    if not raw_boxes:
        d["bootstrap"] = "empty"
        return finish("bootstrap_empty", full_frame_predict(model, image_rgb))

    boot_boxes = []
    for _bb in raw_boxes:
        _pb = pad_box(_bb, w, h, pad, min_px) or _bb
        if (_pb[2] * _pb[3]) / float(w * h) > CASCADE_MAX_ROI_FRAC:
            continue
        boot_boxes.append(_pb)
    if not boot_boxes:
        d["bootstrap"] = "too_large"
        return finish("bootstrap_too_large", full_frame_predict(model, image_rgb))
    d["bootstrap"] = "ok"

    if min(min(b[2], b[3]) for b in boot_boxes) * s < CASCADE_MIN_TRACK_PX:
        s = 1.0
    d["track_scale"] = s

    sf = small(boot)
    tracks = []
    for _bb in boot_boxes:
        _t = make_tracker()
        if not safe_init(_t, sf, _scale_box(_bb, s)):
            continue
        bx, by, bw, bh = _bb
        tracks.append({"tracker": _t, "box": _bb,
                       "pristine": boot[by:by + bh, bx:bx + bw].copy(), "secondary": None})
    if not tracks:
        d["bootstrap"] = "init_failed"
        return finish("tracker_init_failed", full_frame_predict(model, image_rgb))

    def redetect(frame):
        """Tier 3: full-frame pass, re-init, and cache a fresh secondary template."""
        probs = full_frame_predict(model, frame)
        bb = mask_to_bbox(probs > PRIMARY_THRESHOLD)
        if bb is None:
            return None, None
        bb = pad_box(bb, w, h, pad, min_px)
        if bb is None:
            return None, None
        x2, y2, w2, h2 = bb
        return bb, frame[y2:y2 + h2, x2:x2 + w2].copy()

    def recover(frame, trk):
        """Tiers 1-3 in order. Returns a usable box, or None if every tier failed."""
        for tmpl in (trk["pristine"], trk["secondary"]):
            score, loc = template_match(frame, tmpl)
            if loc is not None and score >= TEMPLATE_MATCH_MIN:
                d["tier1_hits"] += 1
                pb = pad_box(loc, w, h, pad, min_px)
                if pb is not None:
                    d["tier2_reinits"] += 1
                    return pb
        d["tier3_redetects"] += 1
        bb, tmpl = redetect(frame)
        return bb

    for fr in frames[1:] + [image_rgb]:
        for trk in tracks:
            ok, raw = safe_update(trk["tracker"], small(fr))
            cand = pad_box(_scale_box(raw, 1.0 / s), w, h, pad, min_px) if ok else None
            if not ok or not _box_ok(cand, w, h, min_px):
                d["track_losses"] += 1
                if not protected:
                    continue
                new_box = recover(fr, trk)
                if new_box is None:
                    continue
                trk["box"] = new_box
                if fr is not image_rgb:
                    trk["tracker"] = make_tracker()
                    if safe_init(trk["tracker"], small(fr), _scale_box(new_box, s)):
                        nx, ny, nw, nh = new_box
                        trk["secondary"] = fr[ny:ny + nh, nx:nx + nw].copy()
                    else:
                        d["reinit_failures"] = d.get("reinit_failures", 0) + 1
            else:
                trk["box"] = cand

    boxes = [trk["box"] for trk in tracks]
    base_probs = full_frame_predict(model, image_rgb) if fused else None
    if variant == "fused_redetect":
        boxes = merge_detections(boxes, component_boxes(base_probs > PRIMARY_THRESHOLD,
                                                        CASCADE_MAX_ROIS,
                                                        CASCADE_MIN_COMPONENT_PX),
                                 w, h, pad, min_px)
    boxes.sort(key=lambda b: -(b[2] * b[3]))

    d["roi_box"] = tuple(int(v) for v in boxes[0])
    d["roi_boxes"] = [tuple(int(v) for v in b) for b in boxes]
    d["n_rois"] = len(boxes)
    d["roi_frac"] = float(sum(b[2] * b[3] for b in boxes)) / float(w * h)
    d["zoom_capped"] = int(fused and any(min(b[2], b[3]) <= min_px for b in boxes))
    probs = (composite_predict(model, image_rgb, boxes, base_probs) if fused
             else roi_predict(model, image_rgb, boxes[0]))

    if protected and not (probs > PRIMARY_THRESHOLD).any():
        d["roi_empty"] = 1
        new_box = recover(image_rgb, tracks[0])
        if new_box is not None and new_box != boxes[0]:
            probs2 = (composite_predict(model, image_rgb, [new_box], base_probs) if fused
                      else roi_predict(model, image_rgb, new_box))
            if (probs2 > PRIMARY_THRESHOLD).any():
                d["roi_box"] = tuple(int(v) for v in new_box)
                d["roi_boxes"] = [d["roi_box"]]
                d["n_rois"] = 1
                return finish("roi_recovered", probs2)
        return finish("fullframe_after_empty_roi", full_frame_predict(model, image_rgb))

    d.setdefault("roi_empty", 0)
    return finish("roi", probs)

if RUN_TRACKING_CASCADE:
    print("=" * 60)
    print(f"cascade variant: {CASCADE_VARIANT}")
    print(f"pad / horizon  : {CASCADE_PAD_FRAC:.0%} margin, {TRACK_HORIZON_S:.1f}s lead-in "
          f"at {TRACK_DOWNSCALE:g}x tracker scale")
    if CASCADE_VARIANT.startswith("fused"):
        print(f"composite      : {CASCADE_COMBINE}, zoom cap {CASCADE_MAX_ZOOM:g}x linear "
              f"(min ROI side scales with the frame), up to {CASCADE_MAX_ROIS} ROIs per frame")
    print("=" * 60)
else:
    print("cascade helpers: defined, arm off")


## 8. DataLoaders
Build train, val, and test DataLoaders from split pairs, with weighted sampling to balance tool vs no-tool frames in each batch.


In [ ]:
from torch.utils.data import DataLoader, WeightedRandomSampler

def build_loaders(train_pairs, val_pairs, test_pairs, train_weights=None, roi_crop=False):
    """Build train/val/test Dataset and DataLoader objects for one fold, with optional weighted sampling for train."""
    _tm_train = "random" if RUN_STATIC_TILING else None
    _tm_eval  = "best" if RUN_STATIC_TILING else None
    train_ds = BinarySegDataset(train_pairs, train_transform, tile_mode=_tm_train,
                                roi_crop_mode=roi_crop)
    val_ds   = BinarySegDataset(val_pairs,   val_transform,   tile_mode=_tm_eval)
    test_ds  = BinarySegDataset(test_pairs,  val_transform,   tile_mode=_tm_eval) if test_pairs else None

    generator = torch.Generator()
    generator.manual_seed(SEED)

    common = dict(num_workers=NUM_WORKERS, pin_memory=True,
                  persistent_workers=NUM_WORKERS > 0,
                  prefetch_factor=2 if NUM_WORKERS > 0 else None)
    eval_common = dict(num_workers=NUM_WORKERS, pin_memory=True,
                       persistent_workers=False,
                       prefetch_factor=2 if NUM_WORKERS > 0 else None)

    if train_weights is not None:
        sampler = WeightedRandomSampler(
            train_weights.tolist(),
            num_samples=len(train_ds),
            replacement=True,
            generator=generator,
        )
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                                  worker_init_fn=seed_worker, **common)
    else:
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                                  worker_init_fn=seed_worker, generator=generator, **common)

    val_loader  = DataLoader(val_ds,  batch_size=BATCH_SIZE, shuffle=False,
                             worker_init_fn=eval_worker_init, **eval_common)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                             worker_init_fn=eval_worker_init, **eval_common) if test_ds else None
    return train_ds, val_ds, test_ds, train_loader, val_loader, test_loader


## 9. Model
Build a binary smp model from `ARCHITECTURE` and `ENCODER`.

Uses `classes=1` with raw logits; sigmoid is applied in the loss and at inference.


In [ ]:
import segmentation_models_pytorch as smp

_ARCH_REGISTRY = {
    "Unet":          smp.Unet,
    "UnetPlusPlus":  smp.UnetPlusPlus,
    "FPN":           smp.FPN,
    "DeepLabV3":     smp.DeepLabV3,
    "DeepLabV3Plus": smp.DeepLabV3Plus,
    "MAnet":         smp.MAnet,
    "Linknet":       smp.Linknet,
    "PAN":           smp.PAN,
}

def build_model(arch, encoder, encoder_weights):
    """Build an untrained segmentation_models_pytorch model for the given architecture/encoder combination."""
    if arch not in _ARCH_REGISTRY:
        raise ValueError(f"Unknown architecture '{arch}'. Options: {list(_ARCH_REGISTRY)}")
    return _ARCH_REGISTRY[arch](
        encoder_name=encoder, encoder_weights=encoder_weights,
        in_channels=3, classes=1, activation=None,
    )

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert device.type == "cuda", (
    "CUDA is not visible, so this run would train on CPU. Re-run the LD_LIBRARY_PATH cells "
    "at the top, restart the runtime, and confirm the pip cell did not replace the "
    "preinstalled torch build."
)
print(f"Device: {device}  |  GPU: {torch.cuda.get_device_name(0)} "
      f"({torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB)")

def build_fresh_model():
    """Build a fresh model with a fixed, seeded initialisation so every fold starts from the same weights."""
    seed(SEED)                              # identical init every fold
    return build_model(ARCHITECTURE, ENCODER, ENCODER_WEIGHTS).to(device)


## 10. Loss, optimiser & metrics
Defines what the model optimises during training (`LOSS`), what is measured on validation/test data, and the AdamW + cosine learning-rate setup.


### 10a. Loss function

The baseline uses BCE.

The loss has to be a smooth, differentiable function so gradient descent can optimise it directly. Metrics like IoU are not usable for this, which is why training optimises BCE while evaluation still reports IoU/Dice/precision/recall. If BCE + Dice is used, the values of each need to be tuned as their magnitudes are different.

Don't change `LOSS` without a documented reason (an experiment result or a clear argument for it). The active loss is saved into every checkpoint, so keep track of which loss produced which result.


### 10a-i. BCE-Tversky compound loss

**Purpose**: Address the baseline's poor recall on small/thin tools. Tversky loss generalises Dice with separate weights for false positives (`alpha`) and false negatives (`beta`), enabling explicit recall bias.


In [ ]:
def bce_tversky_loss(logits, targets):
    """Weighted BCE-Tversky compound loss, used when LOSS == "bce_tversky".

    Rebuilt on every call so TVERSKY_ALPHA and TVERSKY_BETA can change between
    loss balance sweep iterations (section 12b) without redefining this function.
    The construction cost is negligible next to the forward pass.
    """
    tversky = smp.losses.TverskyLoss(mode="binary", from_logits=True,
                                     alpha=TVERSKY_ALPHA, beta=TVERSKY_BETA)
    return (BCE_TVERSKY_LAMBDA * bce_loss(logits, targets)
            + (1.0 - BCE_TVERSKY_LAMBDA) * tversky(logits, targets))

print(f"BCE-Tversky ready: alpha={TVERSKY_ALPHA:.2f} beta={TVERSKY_BETA:.2f} lambda={BCE_TVERSKY_LAMBDA:.2f}")


### 10a-ii. BCE, Dice and the loss dispatcher

Builds `bce_loss` and `dice_loss`, and defines `criterion()`, which dispatches to the active `LOSS` branch.


In [ ]:
from torch import nn

bce_loss  = nn.BCEWithLogitsLoss()
dice_loss = smp.losses.DiceLoss(mode="binary", from_logits=True)

def criterion(logits, targets):
    if LOSS == "bce":
        return bce_loss(logits, targets)
    if LOSS == "bce_dice":
        return (BCE_DICE_LAMBDA * bce_loss(logits, targets)
                + (1 - BCE_DICE_LAMBDA) * dice_loss(logits, targets))
    if LOSS == "bce_tversky":
        return bce_tversky_loss(logits, targets)
    raise ValueError(f"Unknown LOSS '{LOSS}'")

print(f"Loss function ready: {LOSS}")


### 10b. Overlap metrics

These measure how much of the predicted mask overlaps the true mask. One approach is TP/FP/FN pixel counts: TP = predicted tool pixels that really are tool, FP = predicted tool pixels that are actually background, FN = tool pixels the model missed.

IoU is the stricter of the two overlap scores and the primary metric. Dice is kept alongside it since most other papers report Dice, not IoU. Precision and recall are reported separately, not just folded into IoU/Dice, because the hypothesis is about errors in both directions: missing a tool and over-segmenting onto tissue are different failure modes that one overlap score would average away. All four return a value between 0 and 1, where higher is better.


In [ ]:
def _counts(logits, targets, threshold=0.5):
    """Per-frame TP/FP/FN pixel counts for a batch (one triple per image in the batch)."""
    preds = (logits.sigmoid() > threshold).float()
    tp = (preds * targets).sum(dim=(1, 2, 3))
    fp = (preds * (1 - targets)).sum(dim=(1, 2, 3))
    fn = ((1 - preds) * targets).sum(dim=(1, 2, 3))
    return tp, fp, fn

def iou_score(logits: torch.Tensor, targets: torch.Tensor,
              threshold: float = 0.5) -> float:
    """Mean IoU over a batch. Note an empty GT with an empty prediction scores 1.0."""
    tp, fp, fn = _counts(logits, targets, threshold)
    return ((tp + 1e-6) / (tp + fp + fn + 1e-6)).mean().item()

def dice_score(logits: torch.Tensor, targets: torch.Tensor,
               threshold: float = 0.5) -> float:
    """Mean Dice coefficient over a batch."""
    tp, fp, fn = _counts(logits, targets, threshold)
    return ((2 * tp + 1e-6) / (2 * tp + fp + fn + 1e-6)).mean().item()

def precision_score(logits: torch.Tensor, targets: torch.Tensor,
                    threshold: float = 0.5) -> float:
    """TP / predicted-positives. Undefined (skipped) when nothing is predicted."""
    tp, fp, _ = _counts(logits, targets, threshold)
    ok = (tp + fp) > 0
    return (tp[ok] / (tp[ok] + fp[ok])).mean().item() if ok.any() else float("nan")

def recall_score(logits: torch.Tensor, targets: torch.Tensor,
                 threshold: float = 0.5) -> float:
    """TP / ground-truth-positives. Undefined (skipped) on tool-absent frames."""
    tp, _, fn = _counts(logits, targets, threshold)
    ok = (tp + fn) > 0
    return (tp[ok] / (tp[ok] + fn[ok])).mean().item() if ok.any() else float("nan")

print("Overlap metrics ready.")


### 10c. Boundary metrics

Measures boundary prediction accuracy (Hausdorff Distance, Mean Surface Distance). Evaluates how tightly the mask traces the tool outline.


In [ ]:
import numpy as np
from scipy.ndimage import binary_erosion, distance_transform_edt

def _get_boundary(mask: np.ndarray) -> np.ndarray:
    """Return only the boundary pixels of a binary mask via erosion."""
    eroded = binary_erosion(mask)
    return mask & ~eroded

def _surface_distances(pred: np.ndarray, gt: np.ndarray):
    """
    Distance from every predicted boundary pixel to the nearest GT boundary
    pixel, and vice versa. Returns (None, None) if either mask has no boundary.
    """
    pred_boundary = _get_boundary(pred)
    gt_boundary   = _get_boundary(gt)

    if not pred_boundary.any() or not gt_boundary.any():
        return None, None

    dist_pred_to_gt = distance_transform_edt(~gt_boundary)[pred_boundary]
    dist_gt_to_pred = distance_transform_edt(~pred_boundary)[gt_boundary]

    return dist_pred_to_gt, dist_gt_to_pred

def hausdorff_95(logits: torch.Tensor, targets: torch.Tensor,
                 threshold: float = 0.5) -> float:
    """
    95th-percentile Hausdorff distance averaged over a batch.
    Measures the largest gap between boundaries. We use the 95th percentile
    rather than the maximum to avoid a single stray pixel inflating the score.
    """
    preds  = (logits.sigmoid() > threshold).cpu().numpy()[:, 0]
    masks  = targets.cpu().numpy()[:, 0]
    scores = []

    for pred, gt in zip(preds, masks):
        d_p2g, d_g2p = _surface_distances(pred.astype(bool), gt.astype(bool))
        if d_p2g is None:
            continue
        scores.append(max(np.percentile(d_p2g, 95), np.percentile(d_g2p, 95)))

    return float(np.mean(scores)) if scores else float("nan")

def msd_score(logits: torch.Tensor, targets: torch.Tensor,
              threshold: float = 0.5) -> float:
    """
    Mean Surface Distance averaged over a batch.
    Average distance from each predicted boundary pixel to the nearest
    GT boundary pixel, one direction only (pred to GT).
    """
    preds  = (logits.sigmoid() > threshold).cpu().numpy()[:, 0]
    masks  = targets.cpu().numpy()[:, 0]
    scores = []

    for pred, gt in zip(preds, masks):
        d_p2g, _ = _surface_distances(pred.astype(bool), gt.astype(bool))
        if d_p2g is None:
            continue
        scores.append(d_p2g.mean())

    return float(np.mean(scores)) if scores else float("nan")

def asd_score(logits: torch.Tensor, targets: torch.Tensor,
              threshold: float = 0.5) -> float:
    """
    Average Symmetric Surface Distance averaged over a batch.
    Like MSD but measured in both directions and averaged.
    The most commonly reported boundary metric in segmentation papers.
    """
    preds  = (logits.sigmoid() > threshold).cpu().numpy()[:, 0]
    masks  = targets.cpu().numpy()[:, 0]
    scores = []

    for pred, gt in zip(preds, masks):
        d_p2g, d_g2p = _surface_distances(pred.astype(bool), gt.astype(bool))
        if d_p2g is None:
            continue
        scores.append((d_p2g.mean() + d_g2p.mean()) / 2)

    return float(np.mean(scores)) if scores else float("nan")

print("Boundary metrics ready.")


### 10d. Optimiser & scheduler

Sets up AdamW and the cosine learning-rate schedule.


In [ ]:
from torch import optim

def build_optimizer(model, lr=LR, t_max=None):
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=t_max or NUM_EPOCHS, eta_min=lr * 0.01)
    return optimizer, scheduler


## 11. Training loop
Define `train_one_fold()` to run one train/val epoch loop with early stopping on val loss and checkpoint saving.


In [ ]:
import time
from tqdm.auto import tqdm    # .auto falls back to a text bar without ipywidgets

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

def train_one_fold(model, optimizer, scheduler, train_loader, val_loader, ckpt_path,
                   max_epochs=NUM_EPOCHS, early_stop=True, fold=None):
    """Train one fold: run epochs until early stopping, saving a checkpoint whenever val_iou improves."""
    history = {"train_loss": [], "val_loss": [], "val_iou": [], "val_dice": [],
               "val_iou_all": [], "val_notool_fp": [], "lr": []}
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP and device.type == "cuda")
    best_score, best_epoch, epochs_no_improve = -float("inf"), 0, 0
    best_snapshot = {}

    for epoch in range(1, max_epochs + 1):
        # === Train ===
        model.train()
        train_loss = 0.0
        _train_t0 = time.perf_counter()
        for images, masks in tqdm(train_loader, desc=f"epoch {epoch}", unit="it",
                                  leave=False):
            images = images.to(device, non_blocking=True)
            masks  = masks.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=scaler.is_enabled()):
                logits = model(images)
                loss   = criterion(logits, masks)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item()
        train_loss /= len(train_loader)
        _t_train = time.perf_counter() - _train_t0

        # === Validate ===
        model.eval()
        _val_t0 = time.perf_counter()
        loss_sum = iou_sum = dice_sum = iou_all_sum = fp_nt_sum = 0.0
        n_tool = n_all = n_notool = 0
        with torch.no_grad():
            for images, masks in val_loader:
                images = images.to(device, non_blocking=True)
                masks  = masks.to(device, non_blocking=True)
                with torch.amp.autocast("cuda", enabled=scaler.is_enabled()):
                    logits = model(images)
                logits    = logits.float()
                loss_sum += criterion(logits, masks).item() * images.size(0)

                tp, fp, fn = _counts(logits, masks, PRIMARY_THRESHOLD)
                iou  = (tp + 1e-6) / (tp + fp + fn + 1e-6)
                dice = (2 * tp + 1e-6) / (2 * tp + fp + fn + 1e-6)
                has  = (tp + fn) > 0

                iou_all_sum += iou.sum().item();      n_all  += images.size(0)
                iou_sum     += iou[has].sum().item()
                dice_sum    += dice[has].sum().item(); n_tool += int(has.sum().item())
                fp_nt_sum   += fp[~has].sum().item(); n_notool += int((~has).sum().item())

        _t_val   = time.perf_counter() - _val_t0
        _t_epoch = _t_train + _t_val
        _it_s  = len(train_loader)         / max(_t_train, 1e-9)
        _img_s = len(train_loader.dataset) / max(_t_train, 1e-9)

        val_loss    = loss_sum / len(val_loader.dataset)
        val_iou     = iou_sum  / max(n_tool, 1)
        val_dice    = dice_sum / max(n_tool, 1)
        val_iou_all = iou_all_sum / max(n_all, 1)
        val_nt_fp   = fp_nt_sum / max(n_notool, 1)

        history["lr"].append(optimizer.param_groups[0]["lr"])
        scheduler.step()
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_iou"].append(val_iou)
        history["val_dice"].append(val_dice)
        history["val_iou_all"].append(val_iou_all)
        history["val_notool_fp"].append(val_nt_fp)

        # === Checkpoint ===
        score = val_iou
        flag  = ""
        if score > best_score + MIN_DELTA:
            best_score, best_epoch, epochs_no_improve = score, epoch, 0
            best_snapshot = {"epoch": epoch, "val_loss": val_loss, "val_iou": val_iou,
                             "val_dice": val_dice, "val_iou_all": val_iou_all,
                             "val_notool_fp": val_nt_fp}
            torch.save({
                "epoch": epoch, "architecture": ARCHITECTURE, "encoder": ENCODER,
                "loss": LOSS, "select_on": "val_iou", "threshold": PRIMARY_THRESHOLD,
                "lr": optimizer.param_groups[0]["lr"],
                "foreground_label": FOREGROUND_LABEL, "fg_colour": FG_COLOUR.tolist(),
                "model_state": model.state_dict(),
                "val_loss": val_loss, "val_iou": val_iou, "val_dice": val_dice,
            }, ckpt_path)
            flag = f"  best val_iou - saved"
        else:
            epochs_no_improve += 1

        print(f"Epoch {epoch:3d} | loss {train_loss:.4f} -> {val_loss:.4f} | "
              f"IoU {val_iou:.4f} (all {val_iou_all:.4f}) | Dice {val_dice:.4f} | "
              f"ntFP {val_nt_fp:6.0f}px | "
              f"lr {history['lr'][-1]:.2e} | {_it_s:.2f} it/s ({_img_s:.0f} img/s) | "
              f"{_t_epoch:.0f}s | no-improve {epochs_no_improve}/{PATIENCE}{flag}")

        _fold_lbl = f"fold {fold} · " if fold is not None else ""
        _best_iou = best_snapshot.get("val_iou", val_iou)
        notify("epoch", f"{_fold_lbl}epoch {epoch}/{max_epochs}", [
            f"loss {train_loss:.4f} → {val_loss:.4f} | IoU {val_iou:.4f} | Dice {val_dice:.4f}",
            f"best IoU {_best_iou:.4f} @ ep{best_epoch} | patience {epochs_no_improve}/{PATIENCE} | {_t_epoch:.0f}s"
            + (" | NEW BEST" if flag else ""),
        ])

        if epoch == 1:
            # Printed once, while there is still time to change the machine or kill the run.
            print(f"          ^ {_t_train:.0f}s train + {_t_val:.0f}s val. At this rate one "
                  f"fold is ~{_t_epoch * max_epochs / 3600:.1f}h for all {max_epochs} "
                  f"epochs, before early stopping saves anything.")

        if early_stop and epochs_no_improve >= PATIENCE:
            print(f"\n  Early stop at epoch {epoch}: no val_iou improvement for "
                  f"{PATIENCE} epochs (best epoch {best_epoch})")
            break

    best_snapshot["epochs_trained"] = epoch
    best_snapshot["select_on"]      = "val_iou"
    return history, best_snapshot


## 12. Sweeps (optional)

Optional grid searches for learning rate, BCE-Tversky loss balance, synthetic-augmentation intensity, and aug-group ablation. Each runs a reduced-epoch grid search on one fold.


### 12a. LR sweep (optional)
Grid search over `LR_GRID` on fold 0 to pick the best learning rate by val loss.

Set `RUN_SWEEP = True` in the config cell to run; skipped by default.


In [ ]:
if RUN_SWEEP:
    import json
    import gc
    import matplotlib.pyplot as plt

    SWEEP_FOLD   = 0
    SWEEP_EPOCHS = 8

    _sweep_cached = (RESUME_INCOMPLETE and
                     os.system(f"gsutil -q stat {GCS_CKPT_DIR}/sweep_results.json") == 0)

    if _sweep_cached:
        !gsutil -q cp {GCS_CKPT_DIR}/sweep_results.json /tmp/sweep_results.json
        sweep_out  = json.load(open("/tmp/sweep_results.json"))
        sweep_rows = sweep_out["grid"]
        best_lr    = float(sweep_out["best_lr"])
        print("Sweep already in this run folder - reusing it rather than re-sweeping.")
        for row in sweep_rows:
            print(f"  LR {row['lr']:.0e} | min val loss {row['min_val_loss']:.4f} | "
                  f"best val IoU {row['best_val_iou']:.4f}")
    else:
        train_pairs, val_pairs, test_pairs, train_idx = make_split(SWEEP_FOLD)
        _, _, _, sweep_train, sweep_val, _ = build_loaders(train_pairs, val_pairs, test_pairs, sample_weight[train_idx])

        sweep_rows = []
        for lr in LR_GRID:
            model = build_fresh_model()
            optimizer, scheduler = build_optimizer(model, lr=lr)
            hist, _ = train_one_fold(
                model, optimizer, scheduler, sweep_train, sweep_val,
                os.path.join(CHECKPOINT_DIR, f"sweep_lr{lr:.0e}.pt"),
                max_epochs=SWEEP_EPOCHS, early_stop=False, fold=SWEEP_FOLD)
            row = {
                "lr":            float(lr),
                "min_val_loss":  float(min(hist["val_loss"])),
                "best_val_iou":  float(max(hist["val_iou"])),
                "val_loss_hist": [float(x) for x in hist["val_loss"]],  # for per-epoch curves
                "val_iou_hist":  [float(x) for x in hist["val_iou"]],
            }
            sweep_rows.append(row)
            print(f"LR {lr:.0e} | min val loss {row['min_val_loss']:.4f} | best val IoU {row['best_val_iou']:.4f}")
            notify("fold", f"LR {lr:.0e} done", [
            	f"min val loss {row['min_val_loss']:.4f} | best val IoU {row['best_val_iou']:.4f}",
            	f"{len(sweep_rows)}/{len(LR_GRID)} grid points",
            ])

        best_lr = max(sweep_rows, key=lambda row: row["best_val_iou"])["lr"]

        # === Save graphable results to this run's GCS folder ===
        sweep_out = {
            "run":     RUN_NAME,
            "run_id":  RUN_ID,
            "fold":    SWEEP_FOLD,
            "epochs":  SWEEP_EPOCHS,
            "seed":    SEED,
            "grid":    sweep_rows,     # list of {lr, min_val_loss, best_val_iou, val_loss_hist, val_iou_hist}
            "best_lr": float(best_lr),
        }
        with open("/tmp/sweep_results.json", "w") as f:
            json.dump(sweep_out, f, indent=2)

        # quick plot: best val IoU vs LR (log-x)
        lrs  = [row["lr"] for row in sweep_rows]
        ious = [row["best_val_iou"] for row in sweep_rows]
        plt.figure()
        plt.semilogx(lrs, ious, "o-", color=FIG_PRIMARY)
        plt.xlabel("learning rate"); plt.ylabel("best val IoU")
        plt.title(f"LR sweep (fold {SWEEP_FOLD}, {SWEEP_EPOCHS} epochs)"); plt.grid(True, which="both")
        plt.savefig("/tmp/sweep_curve.png", dpi=150); plt.show()

        !gsutil -q cp /tmp/sweep_results.json {GCS_CKPT_DIR}/sweep_results.json
        !gsutil -q cp /tmp/sweep_curve.png    {GCS_CKPT_DIR}/sweep_curve.png
        !gsutil -q cp /tmp/sweep_results.json gs://{CKPT_BUCKET}/_latest_sweep.json
        print("Saved sweep to", GCS_CKPT_DIR)

        for _n in ("model", "optimizer", "scheduler", "sweep_train", "sweep_val"):
            globals().pop(_n, None)
        gc.collect()
        torch.cuda.empty_cache()

    print(f"\nBest LR by val IoU: {best_lr:.0e}")
    notify("fold", f"LR sweep done - {best_lr:.0e}", [
    	f"fold {SWEEP_FOLD} | {SWEEP_EPOCHS} epochs | {len(sweep_rows)} points",
    	("reused from cache" if _sweep_cached else "fresh sweep"),
    ])
    if best_lr in (LR_GRID[0], LR_GRID[-1]):
        print("⚠ the winner is at the edge of LR_GRID - extend the grid that way and "
              "re-sweep before trusting it")

    # === Adopt it for the cross-validation in this same execution ===
    if RUN_CV:
        print(f"Adopting swept LR for cross-validation: {LR:.0e} -> {best_lr:.0e}")
        LR        = float(best_lr)
        LR_SOURCE = f"sweep on fold {SWEEP_FOLD}, {SWEEP_EPOCHS} epochs, selected by val_iou"
else:
    print("LR sweep skipped (RUN_SWEEP=False)")


### 12b. Loss balance sweep (optional)

Only runs when `LOSS == "bce_tversky"` and `RUN_LOSS_BALANCE_SWEEP` is True. Evaluates a grid of `beta` values, adopting the winner for the main cross-validation run.


In [ ]:
if LOSS == "bce_tversky" and RUN_LOSS_BALANCE_SWEEP:
    import json, gc
    import matplotlib.pyplot as plt

    BALANCE_FOLD   = 0
    BALANCE_EPOCHS = 8
    BALANCE_SEEDS  = [42, 43]  # repeats per beta; the spread between them is the noise floor
    BALANCE_ANCHOR = 0.70      # fallback when nothing clears noise (Salehi et al.)
    BALANCE_REF    = 0.50      # reference only, never adopted: alpha = beta = 0.5 makes
                               # Tversky exactly Dice, collapsing this arm into bce_dice
    TINY_PCT_MAX   = 4.5       # matches the "tiny" SIZE_BUCKET in section 14a

    _n = len(TVERSKY_BETA_GRID) * len(BALANCE_SEEDS)
    print(f"Balance sweep: {len(TVERSKY_BETA_GRID)} betas x {len(BALANCE_SEEDS)} seeds "
          f"= {_n} runs x {BALANCE_EPOCHS} epochs (~{_n * 0.3:.1f}h at logged rates)")

    def _eval_ckpt(ckpt_path, loader):
        """Score a checkpoint on the val fold across every EVAL_THRESHOLDS operating point.

        Raising beta shifts the predicted-probability distribution, so a comparison at a
        single fixed threshold cannot separate "better model" from "different operating
        point". Counts are kept per threshold so the two can be told apart below.
        """
        m = build_model(ARCHITECTURE, ENCODER, ENCODER_WEIGHTS).to(device)
        m.load_state_dict(torch.load(ckpt_path, map_location=device,
                                     weights_only=True)["model_state"])
        m.eval()
        tps = {t: [] for t in EVAL_THRESHOLDS}
        fps = {t: [] for t in EVAL_THRESHOLDS}
        fns = {t: [] for t in EVAL_THRESHOLDS}
        gt_px = []
        with torch.no_grad():
            for images, masks in loader:
                images = images.to(device, non_blocking=True)
                masks  = masks.to(device, non_blocking=True)
                probs  = m(images).float().sigmoid()
                gt_px += masks.sum(dim=(1, 2, 3)).cpu().tolist()
                for t in EVAL_THRESHOLDS:
                    p  = (probs > t).float()
                    tps[t] += (p * masks).sum(dim=(1, 2, 3)).cpu().tolist()
                    fps[t] += (p * (1 - masks)).sum(dim=(1, 2, 3)).cpu().tolist()
                    fns[t] += ((1 - p) * masks).sum(dim=(1, 2, 3)).cpu().tolist()
        del m
        gc.collect(); torch.cuda.empty_cache()

        gt   = np.array(gt_px, float)
        has  = gt > 0                                   # tool-present frames
        pct  = gt / (IMAGE_HEIGHT * IMAGE_WIDTH) * 100
        tiny = has & (pct < TINY_PCT_MAX)

        def _at(t):
            tp = np.array(tps[t], float); fp = np.array(fps[t], float)
            fn = np.array(fns[t], float); pp = tp + fp
            iou  = (tp + 1e-6) / (tp + fp + fn + 1e-6)
            rec  = np.divide(tp, gt, out=np.zeros_like(tp), where=gt > 0)
            prec = np.divide(tp, pp, out=np.zeros_like(tp), where=pp > 0)
            return {
                "iou":              float(iou[has].mean()),
                "recall":           float(rec[has].mean()),
                "precision_strict": float(prec[has].mean()),
                "iou_tiny":         float(iou[tiny].mean()) if tiny.any() else float("nan"),
                "recall_tiny":      float(rec[tiny].mean()) if tiny.any() else float("nan"),
                "miss_rate":        float((pp[has] == 0).mean()),
                # the axis val_iou is blind to: FP area on frames with nothing to find
                "notool_fp":        float(fp[~has].mean()) if (~has).any() else float("nan"),
            }

        per_t = {f"{t:.2f}": _at(t) for t in EVAL_THRESHOLDS}
        out = dict(per_t[f"{PRIMARY_THRESHOLD:.2f}"])   # headline = primary threshold
        _bt = max(EVAL_THRESHOLDS, key=lambda t: per_t[f"{t:.2f}"]["iou_tiny"])
        out["iou_tiny_bestthr"] = per_t[f"{_bt:.2f}"]["iou_tiny"]
        out["best_thr"]         = float(_bt)
        out["n_tiny"]  = int(tiny.sum())
        out["n_tool"]  = int(has.sum())
        out["per_threshold"] = per_t
        return out

    best_beta, balance_out = BALANCE_ANCHOR, {}
    try:
        _cached = (RESUME_INCOMPLETE and os.system(
            f"gsutil -q stat {GCS_CKPT_DIR}/loss_balance_results.json") == 0)

        if _cached:
            !gsutil -q cp {GCS_CKPT_DIR}/loss_balance_results.json /tmp/lbr.json
            balance_out = json.load(open("/tmp/lbr.json"))
            rows, agg = balance_out["runs"], balance_out["by_beta"]
            best_beta = float(balance_out["best_beta"])
            print("Sweep already in this run folder, reusing it instead of re-sweeping.")
        else:
            train_pairs, val_pairs, test_pairs, train_idx = make_split(BALANCE_FOLD)
            _, _, _, balance_train, balance_val, _ = build_loaders(
                train_pairs, val_pairs, test_pairs, sample_weight[train_idx])

            rows = []
            for beta in TVERSKY_BETA_GRID:
                for s in BALANCE_SEEDS:
                    TVERSKY_BETA, TVERSKY_ALPHA = float(beta), float(1.0 - beta)
                    seed(s)
                    model = build_model(ARCHITECTURE, ENCODER, ENCODER_WEIGHTS).to(device)
                    optimizer, scheduler = build_optimizer(model, t_max=BALANCE_EPOCHS)
                    ckpt = os.path.join(CHECKPOINT_DIR, f"bal_b{beta:.2f}_s{s}.pt")
                    hist, _best = train_one_fold(
                        model, optimizer, scheduler, balance_train, balance_val, ckpt,
                        max_epochs=BALANCE_EPOCHS, early_stop=False, fold=BALANCE_FOLD)

                    row = {"beta": float(beta), "alpha": float(1.0 - beta), "seed": int(s),
                           "best_val_iou": float(max(hist["val_iou"])),
                           "min_val_loss": float(min(hist["val_loss"])),
                           "val_iou_hist": [float(x) for x in hist["val_iou"]]}
                    row.update(_eval_ckpt(ckpt, balance_val))
                    rows.append(row)
                    print(f"beta {beta:.2f} seed {s} | tinyIoU {row['iou_tiny']:.4f} "
                          f"(bestthr {row['iou_tiny_bestthr']:.4f} @ {row['best_thr']:.2f}) | "
                          f"IoU {row['iou']:.4f} | R {row['recall']:.4f} | "
                          f"P {row['precision_strict']:.4f} | ntFP {row['notool_fp']:.0f}px")

                    for _n_ in ("model", "optimizer", "scheduler"):
                        globals().pop(_n_, None)
                    gc.collect(); torch.cuda.empty_cache()

            AGG_KEYS = ("iou", "iou_tiny", "iou_tiny_bestthr", "recall", "recall_tiny",
                        "precision_strict", "miss_rate", "notool_fp")
            agg = []
            for b in sorted({r["beta"] for r in rows}):
                rs = [r for r in rows if r["beta"] == b]
                g = {"beta": b, "n_seeds": len(rs)}
                for k in AGG_KEYS:
                    v = [r[k] for r in rs]
                    g[k] = float(np.mean(v))
                    g[f"{k}_spread"] = float(max(v) - min(v))
                agg.append(g)

            noise = {k: max(g[f"{k}_spread"] for g in agg) for k in AGG_KEYS}

            anchor = next((g for g in agg if abs(g["beta"] - BALANCE_ANCHOR) < 1e-9), None)
            # beta = 0.5 is a reference point, not a candidate: it removes the intervention.
            cands  = [g for g in agg if g["beta"] > BALANCE_REF]

            if anchor is not None:
                ok = [g for g in cands
                      if g["iou"]       >= anchor["iou"]       - noise["iou"]
                      and g["notool_fp"] <= anchor["notool_fp"] + noise["notool_fp"]]
                rejected = [g["beta"] for g in cands if g not in ok]
            else:
                ok, rejected = cands, []

            top = max(ok, key=lambda g: g["iou_tiny"]) if ok else None
            if top is None:
                best_beta, reason = BALANCE_ANCHOR, "no candidate satisfied the constraints"
            elif anchor is None:
                best_beta, reason = top["beta"], "anchor not in grid, took constrained best"
            elif top["iou_tiny"] - anchor["iou_tiny"] > noise["iou_tiny"]:
                best_beta = top["beta"]
                reason = (f"tiny IoU beats anchor by {top['iou_tiny'] - anchor['iou_tiny']:.4f} "
                          f"> noise {noise['iou_tiny']:.4f}, with no constraint breached")
            else:
                best_beta = BALANCE_ANCHOR
                reason = (f"best margin {top['iou_tiny'] - anchor['iou_tiny']:.4f} within "
                          f"noise {noise['iou_tiny']:.4f}, held at literature anchor")

            balance_out = {
                "run": RUN_NAME, "run_id": RUN_ID, "fold": BALANCE_FOLD,
                "epochs": BALANCE_EPOCHS, "seeds": BALANCE_SEEDS,
                "lambda": BCE_TVERSKY_LAMBDA, "primary_threshold": PRIMARY_THRESHOLD,
                "select_on": "iou_tiny, constrained on iou and notool_fp",
                "tiny_pct_max": TINY_PCT_MAX, "noise_floor": noise,
                "rejected_by_constraint": rejected,
                "runs": rows, "by_beta": agg,
                "best_beta": float(best_beta), "reason": reason,
            }
            with open("/tmp/lbr.json", "w") as f:
                json.dump(balance_out, f, indent=2)

            _betas = [g["beta"] for g in agg]
            fig, ax = plt.subplots(1, 3, figsize=(15, 4))
            for j, (key, ttl) in enumerate([
                    ("iou_tiny", f"tiny tools (<{TINY_PCT_MAX}% frame) @ thr {PRIMARY_THRESHOLD:.2f}"),
                    ("iou",      "all tool-present frames"),
                    ("notool_fp", "FP area on tool-absent frames")]):
                ax[j].plot(_betas, [g[key] for g in agg], "o-", color=FIG_PRIMARY, label="seed mean")
                ax[j].scatter([r["beta"] for r in rows], [r[key] for r in rows],
                              c=FIG_SECONDARY, s=18, zorder=3, label="per seed")
                ax[j].axvline(BALANCE_REF, ls=":", c=FIG_RULE)
                ax[j].set_xlabel("Tversky beta (alpha = 1 - beta)")
                ax[j].set_ylabel(key); ax[j].set_title(ttl)
                ax[j].grid(True); ax[j].legend(fontsize=8)
            ax[0].plot(_betas, [g["iou_tiny_bestthr"] for g in agg], "s--", color=FIG_SERIES[3],
                       label="best threshold"); ax[0].legend(fontsize=8)
            fig.suptitle(f"Loss balance sweep (fold {BALANCE_FOLD}, {BALANCE_EPOCHS} epochs, "
                         f"{len(BALANCE_SEEDS)} seeds)")
            fig.tight_layout()
            fig.savefig("/tmp/loss_balance_curve.png", dpi=150); plt.show()

            !gsutil -q cp /tmp/lbr.json {GCS_CKPT_DIR}/loss_balance_results.json
            !gsutil -q cp /tmp/loss_balance_curve.png {GCS_CKPT_DIR}/loss_balance_curve.png

            for _n_ in ("balance_train", "balance_val"):
                globals().pop(_n_, None)
            gc.collect(); torch.cuda.empty_cache()

        print("\nbeta | tinyIoU (spread) | bestthr |    IoU | recall | precis | miss | ntFP")
        for g in agg:
            print(f"{g['beta']:.2f} | {g['iou_tiny']:.4f} ({g['iou_tiny_spread']:.4f}) | "
                  f"{g['iou_tiny_bestthr']:.4f}  | {g['iou']:.4f} | {g['recall']:.4f} | "
                  f"{g['precision_strict']:.4f} | {g['miss_rate']:.3f} | {g['notool_fp']:.0f}")
        if balance_out.get("rejected_by_constraint"):
            print(f"rejected on the overall-IoU / no-tool-FP constraints: "
                  f"{balance_out['rejected_by_constraint']}")
        print(f"\nAdopted beta {best_beta:.2f}: {balance_out.get('reason', 'from cache')}")
        if best_beta == max(TVERSKY_BETA_GRID):
            print("winner sits at the top edge of TVERSKY_BETA_GRID; extend it and re-sweep "
                  "before trusting it")

    except Exception as _e:
        import traceback; traceback.print_exc()
        print(f"Balance sweep failed ({type(_e).__name__}: {_e}); "
              f"falling back to beta={BALANCE_ANCHOR}")
        best_beta = BALANCE_ANCHOR
        notify("error", "balance sweep failed", [f"{type(_e).__name__}: {_e}",
                                                 f"falling back to beta {BALANCE_ANCHOR:.2f}"])

    TVERSKY_BETA  = float(best_beta)
    TVERSKY_ALPHA = 1.0 - TVERSKY_BETA
    seed(SEED)   # restore the config seed so CV is unaffected by the sweep repeats
    print(f"Adopting for cross-validation: alpha={TVERSKY_ALPHA:.2f} beta={TVERSKY_BETA:.2f}")
    notify("fold", f"balance sweep done - beta {TVERSKY_BETA:.2f}",
           [balance_out.get("reason", "adopted from cache"), "starting cross-validation"])
else:
    print("Loss balance sweep skipped (LOSS is not bce_tversky, or RUN_LOSS_BALANCE_SWEEP is False)")


### 12c. Synthetic augmentation mini-sweep (optional)

Only runs when `RUN_AUG_SWEEP` is True. Hard evaluation of the safe/extreme bounds identified during visual tuning, scored on `iou_tiny` and `notool_clean_rate`.


In [ ]:
if RUN_AUG_SWEEP:
    import json, gc
    import matplotlib.pyplot as plt

    AUG_SWEEP_FOLD   = 0
    AUG_SWEEP_EPOCHS = 20 # As the models take about 75 to run, this is a good middle ground for simply testing performance
    TINY_PCT_MAX     = 4.5 # matches the "tiny" SIZE_BUCKET in section 14a / loss balance sweep
    AUG_SWEEP_ORDER  = ["low", "medium", "high"]

    print(f"Aug mini-sweep: presets={AUG_SWEEP_ORDER} | fold {AUG_SWEEP_FOLD} | "
          f"{AUG_SWEEP_EPOCHS} epochs | p_low={AUG_P_LOW} p_std={AUG_P_STD}")

    def _aug_eval_ckpt(ckpt_path, loader):
        """Score a checkpoint on the val fold; headline metrics + per-threshold counts.

        Borrowed from the loss balance sweep so iou_tiny and notool_fp are available here
        even though train_one_fold only logs overall val_iou / val_notool_fp.
        """
        m = build_model(ARCHITECTURE, ENCODER, ENCODER_WEIGHTS).to(device)
        m.load_state_dict(torch.load(ckpt_path, map_location=device,
                                     weights_only=True)["model_state"])
        m.eval()
        tps = {t: [] for t in EVAL_THRESHOLDS}
        fps = {t: [] for t in EVAL_THRESHOLDS}
        fns = {t: [] for t in EVAL_THRESHOLDS}
        gt_px = []
        with torch.no_grad():
            for images, masks in loader:
                images = images.to(device, non_blocking=True)
                masks  = masks.to(device, non_blocking=True)
                probs  = m(images).float().sigmoid()
                gt_px += masks.sum(dim=(1, 2, 3)).cpu().tolist()
                for t in EVAL_THRESHOLDS:
                    p  = (probs > t).float()
                    tps[t] += (p * masks).sum(dim=(1, 2, 3)).cpu().tolist()
                    fps[t] += (p * (1 - masks)).sum(dim=(1, 2, 3)).cpu().tolist()
                    fns[t] += ((1 - p) * masks).sum(dim=(1, 2, 3)).cpu().tolist()
        del m
        gc.collect(); torch.cuda.empty_cache()

        gt   = np.array(gt_px, float)
        has  = gt > 0
        pct  = gt / (IMAGE_HEIGHT * IMAGE_WIDTH) * 100
        tiny = has & (pct < TINY_PCT_MAX)

        def _at(t):
            tp = np.array(tps[t], float); fp = np.array(fps[t], float)
            fn = np.array(fns[t], float); pp = tp + fp
            iou  = (tp + 1e-6) / (tp + fp + fn + 1e-6)
            rec  = np.divide(tp, gt, out=np.zeros_like(tp), where=gt > 0)
            prec = np.divide(tp, pp, out=np.zeros_like(tp), where=pp > 0)
            return {
                "iou":              float(iou[has].mean()),
                "recall":           float(rec[has].mean()),
                "precision_strict": float(prec[has].mean()),
                "iou_tiny":         float(iou[tiny].mean()) if tiny.any() else float("nan"),
                "recall_tiny":      float(rec[tiny].mean()) if tiny.any() else float("nan"),
                "miss_rate":        float((pp[has] == 0).mean()),
                "notool_fp":        float(fp[~has].mean()) if (~has).any() else float("nan"),
            }

        per_t = {f"{t:.2f}": _at(t) for t in EVAL_THRESHOLDS}
        out = dict(per_t[f"{PRIMARY_THRESHOLD:.2f}"])
        _bt = max(EVAL_THRESHOLDS, key=lambda t: per_t[f"{t:.2f}"]["iou_tiny"])
        out["iou_tiny_bestthr"] = per_t[f"{_bt:.2f}"]["iou_tiny"]
        out["best_thr"]         = float(_bt)
        out["n_tiny"]  = int(tiny.sum())
        out["n_tool"]  = int(has.sum())
        out["per_threshold"] = per_t
        return out

    best_preset, aug_sweep_out = AUG_PRESET, {}
    try:
        _cached = (RESUME_INCOMPLETE and os.system(
            f"gsutil -q stat {GCS_CKPT_DIR}/aug_sweep_results.json") == 0)

        if _cached:
            !gsutil -q cp {GCS_CKPT_DIR}/aug_sweep_results.json /tmp/aug_sweep_results.json
            aug_sweep_out = json.load(open("/tmp/aug_sweep_results.json"))
            rows = aug_sweep_out["runs"]
            best_preset = aug_sweep_out["best_preset"]
            print("Aug sweep already in this run folder, reusing it instead of re-sweeping.")
        else:
            rows = []
            for preset in AUG_SWEEP_ORDER:
                # Swap the global transform so build_loaders / seed_worker see this preset.
                train_transform = build_train_transform(preset)
                train_pairs, val_pairs, test_pairs, train_idx = make_split(AUG_SWEEP_FOLD)
                _, _, _, aug_train, aug_val, _ = build_loaders(
                    train_pairs, val_pairs, test_pairs, sample_weight[train_idx])

                model = build_fresh_model()
                optimizer, scheduler = build_optimizer(model, t_max=AUG_SWEEP_EPOCHS)
                ckpt = os.path.join(CHECKPOINT_DIR, f"aug_{preset}.pt")
                hist, _best = train_one_fold(
                    model, optimizer, scheduler, aug_train, aug_val, ckpt,
                    max_epochs=AUG_SWEEP_EPOCHS, early_stop=False, fold=AUG_SWEEP_FOLD)

                row = {
                    "preset":       preset,
                    "level":        AUG_PRESETS[preset]["level"],
                    "p":            float(AUG_PRESETS[preset]["p"]),
                    "best_val_iou": float(max(hist["val_iou"])),
                    "min_val_loss": float(min(hist["val_loss"])),
                    "val_iou_hist": [float(x) for x in hist["val_iou"]],
                    "val_notool_fp_hist": [float(x) for x in hist["val_notool_fp"]],
                }
                row.update(_aug_eval_ckpt(ckpt, aug_val))
                rows.append(row)
                print(f"{preset:6s} | tinyIoU {row['iou_tiny']:.4f} "
                      f"(bestthr {row['iou_tiny_bestthr']:.4f} @ {row['best_thr']:.2f}) | "
                      f"IoU {row['iou']:.4f} | ntFP {row['notool_fp']:.0f}px | "
                      f"train-best IoU {row['best_val_iou']:.4f}")
                notify("fold", f"aug sweep {preset} done", [
                    f"tinyIoU {row['iou_tiny']:.4f} | IoU {row['iou']:.4f} | "
                    f"ntFP {row['notool_fp']:.0f}px",
                    f"{len(rows)}/{len(AUG_SWEEP_ORDER)} presets",
                ])

                for _n_ in ("model", "optimizer", "scheduler", "aug_train", "aug_val"):
                    globals().pop(_n_, None)
                gc.collect(); torch.cuda.empty_cache()

            best_row = max(rows, key=lambda r: r["iou_tiny"])
            best_preset = best_row["preset"]
            med = next((r for r in rows if r["preset"] == "medium"), None)
            warnings = []
            if med is not None:
                if best_row["iou"] + 1e-9 < med["iou"]:
                    warnings.append(
                        f"winner overall IoU {best_row['iou']:.4f} < medium {med['iou']:.4f}")
                if best_row["notool_fp"] - 1e-9 > med["notool_fp"]:
                    warnings.append(
                        f"winner ntFP {best_row['notool_fp']:.0f} > medium {med['notool_fp']:.0f}")
            reason = (f"highest iou_tiny={best_row['iou_tiny']:.4f} at preset={best_preset}"
                      + ("; " + "; ".join(warnings) if warnings else
                         "; overall IoU and ntFP not worse than medium"))

            aug_sweep_out = {
                "run": RUN_NAME, "run_id": RUN_ID,
                "fold": AUG_SWEEP_FOLD, "epochs": AUG_SWEEP_EPOCHS, "seed": SEED,
                "primary_threshold": PRIMARY_THRESHOLD,
                "select_on": "iou_tiny (overall iou + notool_fp reported)",
                "tiny_pct_max": TINY_PCT_MAX,
                "aug_levels": AUG_LEVELS,
                "aug_presets": {k: {"level": v["level"], "p": float(v["p"])}
                                for k, v in AUG_PRESETS.items()},
                "p_low": AUG_P_LOW, "p_std": AUG_P_STD,
                "runs": rows,
                "best_preset": best_preset,
                "best_iou_tiny": float(best_row["iou_tiny"]),
                "best_iou": float(best_row["iou"]),
                "best_notool_fp": float(best_row["notool_fp"]),
                "reason": reason,
                "warnings": warnings,
            }
            with open("/tmp/aug_sweep_results.json", "w") as f:
                json.dump(aug_sweep_out, f, indent=2)

            # Three-panel figure: hypothesis target, overall performance, no-tool safety.
            labels = [r["preset"] for r in rows]
            x = list(range(len(rows)))
            fig, ax = plt.subplots(1, 3, figsize=(15, 4))
            for j, (key, ttl, ylab) in enumerate([
                    ("iou_tiny",  f"tiny tools (<{TINY_PCT_MAX}% frame) @ thr {PRIMARY_THRESHOLD:.2f}",
                     "iou_tiny"),
                    ("iou",       "all tool-present frames", "iou"),
                    ("notool_fp", "FP area on tool-absent frames", "notool_fp (px)"),
            ]):
                vals = [r[key] for r in rows]
                ax[j].plot(x, vals, "o-", color=FIG_PRIMARY)
                ax[j].set_xticks(x); ax[j].set_xticklabels(labels)
                ax[j].set_xlabel("aug preset"); ax[j].set_ylabel(ylab); ax[j].set_title(ttl)
                ax[j].grid(True)
                # mark the winner
                wi = labels.index(best_preset)
                ax[j].scatter([wi], [vals[wi]], c=FIG_SERIES[6], s=80, zorder=3, label="winner")
                ax[j].legend(fontsize=8)
            ax[0].plot(x, [r["iou_tiny_bestthr"] for r in rows], "s--", color=FIG_SERIES[3],
                       label="best thr"); ax[0].legend(fontsize=8)
            fig.suptitle(f"Aug mini-sweep (fold {AUG_SWEEP_FOLD}, {AUG_SWEEP_EPOCHS} epochs) "
                         f"- winner={best_preset}")
            fig.tight_layout()
            fig.savefig("/tmp/aug_sweep_curve.png", dpi=150); plt.show()

            # Correlation-style companion: tiny IoU vs overall IoU, sized by ntFP.
            fig2, ax2 = plt.subplots(figsize=(6, 5))
            nt = [r["notool_fp"] for r in rows]
            nt_size = [40 + 80 * (v / max(nt)) for v in nt] if max(nt) > 0 else [60] * len(rows)
            ax2.scatter([r["iou"] for r in rows], [r["iou_tiny"] for r in rows],
                        s=nt_size, c=[FIG_SERIES[i % len(FIG_SERIES)] for i in x], zorder=3)
            for r in rows:
                ax2.annotate(r["preset"], (r["iou"], r["iou_tiny"]),
                             textcoords="offset points", xytext=(6, 4), fontsize=9)
            ax2.set_xlabel("overall IoU (tool-present)")
            ax2.set_ylabel("iou_tiny")
            ax2.set_title("Hypothesis check: tiny IoU vs overall IoU\n(marker size ~ notool_fp)")
            ax2.grid(True)
            fig2.tight_layout()
            fig2.savefig("/tmp/aug_sweep_tradeoff.png", dpi=150); plt.show()

            !gsutil -q cp /tmp/aug_sweep_results.json {GCS_CKPT_DIR}/aug_sweep_results.json
            !gsutil -q cp /tmp/aug_sweep_curve.png    {GCS_CKPT_DIR}/aug_sweep_curve.png
            !gsutil -q cp /tmp/aug_sweep_tradeoff.png {GCS_CKPT_DIR}/aug_sweep_tradeoff.png
            !gsutil -q cp /tmp/aug_sweep_results.json gs://{CKPT_BUCKET}/_latest_aug_sweep.json
            print("Saved aug sweep to", GCS_CKPT_DIR)

        print("\npreset | tinyIoU | bestthr |    IoU | recall | precis | miss | ntFP")
        for r in rows:
            print(f"{r['preset']:6s} | {r['iou_tiny']:.4f} | "
                  f"{r['iou_tiny_bestthr']:.4f}  | {r['iou']:.4f} | {r['recall']:.4f} | "
                  f"{r['precision_strict']:.4f} | {r['miss_rate']:.3f} | {r['notool_fp']:.0f}")
        print(f"\nWinner: {best_preset}")
        print(f"Reason: {aug_sweep_out.get('reason', 'from cache')}")
        if aug_sweep_out.get("warnings"):
            print("Warnings:", "; ".join(aug_sweep_out["warnings"]))

    except Exception as _e:
        import traceback; traceback.print_exc()
        print(f"Aug sweep failed ({type(_e).__name__}: {_e}); "
              f"holding AUG_PRESET={AUG_PRESET}")
        best_preset = AUG_PRESET
        notify("error", "aug sweep failed", [f"{type(_e).__name__}: {_e}",
                                             f"holding preset {AUG_PRESET}"])

    print(f"\nBest aug preset by iou_tiny: {best_preset}")
    notify("fold", f"aug sweep done - {best_preset}", [
        aug_sweep_out.get("reason", "adopted from cache") if aug_sweep_out else "fallback",
        f"fold {AUG_SWEEP_FOLD} | {AUG_SWEEP_EPOCHS} epochs",
    ])

    print(f"Adopting swept aug preset for the rest of this run: "
          f"{AUG_PRESET} -> {best_preset}")
    AUG_PRESET = best_preset
    RUN_SYNTHETIC_AUG = True
    train_transform = build_train_transform(AUG_PRESET)
else:
    print("Aug mini-sweep skipped (RUN_AUG_SWEEP=False)")


### 12d. Synthetic augmentation ablation (optional)

Only runs when `RUN_AUG_ABLATION` is True. Fold-0 comparison of high-volume appearance risk versus fatal smoke/exposure risk against the baseline recipe.


In [ ]:
if RUN_AUG_ABLATION:
    import json, gc
    import matplotlib.pyplot as plt

    ABLATION_FOLD   = 0
    ABLATION_EPOCHS = 20
    TINY_PCT_MAX    = 4.5  # matches section 14a / aug mini-sweep
    ABLATION_ORDER  = list(AUG_ABLATION_VARIANTS.keys())

    print(f"Aug ablation: variants={ABLATION_ORDER} | fold {ABLATION_FOLD} | "
          f"{ABLATION_EPOCHS} epochs | level={AUG_ABLATION_LEVEL} p={AUG_ABLATION_P}")
    for _k in ABLATION_ORDER:
        _v = AUG_ABLATION_VARIANTS[_k]
        _augs = (("baseline recipe",) if _k == "baseline"
                 else tuple(_v["augs"]) + ("ElasticTransform",))
        print(f"  {_k}: {_v['label']} -> {_augs}")

    def _ablation_eval_ckpt(ckpt_path, loader):
        """Score a checkpoint on the val fold; same schema as the aug mini-sweep JSON."""
        m = build_model(ARCHITECTURE, ENCODER, ENCODER_WEIGHTS).to(device)
        m.load_state_dict(torch.load(ckpt_path, map_location=device,
                                     weights_only=True)["model_state"])
        m.eval()
        tps = {t: [] for t in EVAL_THRESHOLDS}
        fps = {t: [] for t in EVAL_THRESHOLDS}
        fns = {t: [] for t in EVAL_THRESHOLDS}
        gt_px = []
        with torch.no_grad():
            for images, masks in loader:
                images = images.to(device, non_blocking=True)
                masks  = masks.to(device, non_blocking=True)
                probs  = m(images).float().sigmoid()
                gt_px += masks.sum(dim=(1, 2, 3)).cpu().tolist()
                for t in EVAL_THRESHOLDS:
                    p  = (probs > t).float()
                    tps[t] += (p * masks).sum(dim=(1, 2, 3)).cpu().tolist()
                    fps[t] += (p * (1 - masks)).sum(dim=(1, 2, 3)).cpu().tolist()
                    fns[t] += ((1 - p) * masks).sum(dim=(1, 2, 3)).cpu().tolist()
        del m
        gc.collect(); torch.cuda.empty_cache()

        gt   = np.array(gt_px, float)
        has  = gt > 0
        pct  = gt / (IMAGE_HEIGHT * IMAGE_WIDTH) * 100
        tiny = has & (pct < TINY_PCT_MAX)

        def _at(t):
            tp = np.array(tps[t], float); fp = np.array(fps[t], float)
            fn = np.array(fns[t], float); pp = tp + fp
            iou  = (tp + 1e-6) / (tp + fp + fn + 1e-6)
            rec  = np.divide(tp, gt, out=np.zeros_like(tp), where=gt > 0)
            prec = np.divide(tp, pp, out=np.zeros_like(tp), where=pp > 0)
            return {
                "iou":              float(iou[has].mean()),
                "recall":           float(rec[has].mean()),
                "precision_strict": float(prec[has].mean()),
                "iou_tiny":         float(iou[tiny].mean()) if tiny.any() else float("nan"),
                "recall_tiny":      float(rec[tiny].mean()) if tiny.any() else float("nan"),
                "miss_rate":        float((pp[has] == 0).mean()),
                "notool_fp":        float(fp[~has].mean()) if (~has).any() else float("nan"),
            }

        per_t = {f"{t:.2f}": _at(t) for t in EVAL_THRESHOLDS}
        out = dict(per_t[f"{PRIMARY_THRESHOLD:.2f}"])
        _bt = max(EVAL_THRESHOLDS, key=lambda t: per_t[f"{t:.2f}"]["iou_tiny"])
        out["iou_tiny_bestthr"] = per_t[f"{_bt:.2f}"]["iou_tiny"]
        out["best_thr"]         = float(_bt)
        out["n_tiny"]  = int(tiny.sum())
        out["n_tool"]  = int(has.sum())
        out["per_threshold"] = per_t
        return out

    aug_ablation_out = {}
    try:
        _cached = (RESUME_INCOMPLETE and os.system(
            f"gsutil -q stat {GCS_CKPT_DIR}/aug_ablation_results.json") == 0)

        if _cached:
            !gsutil -q cp {GCS_CKPT_DIR}/aug_ablation_results.json /tmp/aug_ablation_results.json
            aug_ablation_out = json.load(open("/tmp/aug_ablation_results.json"))
            rows = aug_ablation_out["runs"]
            print("Aug ablation already in this run folder, reusing it instead of re-running.")
        else:
            rows = []
            for variant in ABLATION_ORDER:
                train_transform = build_train_transform(ablation=variant)
                train_pairs, val_pairs, test_pairs, train_idx = make_split(ABLATION_FOLD)
                _, _, _, abl_train, abl_val, _ = build_loaders(
                    train_pairs, val_pairs, test_pairs, sample_weight[train_idx])

                model = build_fresh_model()
                optimizer, scheduler = build_optimizer(model, t_max=ABLATION_EPOCHS)
                ckpt = os.path.join(CHECKPOINT_DIR, f"ablation_{variant}.pt")
                hist, _best = train_one_fold(
                    model, optimizer, scheduler, abl_train, abl_val, ckpt,
                    max_epochs=ABLATION_EPOCHS, early_stop=False, fold=ABLATION_FOLD)

                spec = AUG_ABLATION_VARIANTS[variant]
                row = {
                    "variant":      variant,
                    "label":        spec["label"],
                    "augs":         (["baseline_recipe"] if variant == "baseline"
                                     else list(spec["augs"]) + ["ElasticTransform"]),
                    "level":        None if variant == "baseline" else AUG_ABLATION_LEVEL,
                    "p":            None if variant == "baseline" else float(AUG_ABLATION_P),
                    "best_val_iou": float(max(hist["val_iou"])),
                    "min_val_loss": float(min(hist["val_loss"])),
                    "val_iou_hist": [float(x) for x in hist["val_iou"]],
                    "val_notool_fp_hist": [float(x) for x in hist["val_notool_fp"]],
                    "val_loss_hist": [float(x) for x in hist["val_loss"]],
                }
                row.update(_ablation_eval_ckpt(ckpt, abl_val))
                rows.append(row)
                print(f"{variant:8s} | tinyIoU {row['iou_tiny']:.4f} "
                      f"(bestthr {row['iou_tiny_bestthr']:.4f} @ {row['best_thr']:.2f}) | "
                      f"IoU {row['iou']:.4f} | ntFP {row['notool_fp']:.0f}px | "
                      f"train-best IoU {row['best_val_iou']:.4f}")
                notify("fold", f"aug ablation {variant} done", [
                    f"{spec['label']}: tinyIoU {row['iou_tiny']:.4f} | "
                    f"IoU {row['iou']:.4f} | ntFP {row['notool_fp']:.0f}px",
                    f"{len(rows)}/{len(ABLATION_ORDER)} variants",
                ])

                for _n_ in ("model", "optimizer", "scheduler", "abl_train", "abl_val"):
                    globals().pop(_n_, None)
                gc.collect(); torch.cuda.empty_cache()

            # Rank for the write-up; do not adopt into train_transform.
            best_row = max(rows, key=lambda r: r["iou_tiny"])
            base_row = next((r for r in rows if r["variant"] == "baseline"), None)
            delta_vs_baseline = {}
            if base_row is not None:
                for r in rows:
                    if r["variant"] == "baseline":
                        continue
                    delta_vs_baseline[r["variant"]] = {
                        "iou_tiny": float(r["iou_tiny"] - base_row["iou_tiny"]),
                        "iou": float(r["iou"] - base_row["iou"]),
                        "notool_fp": float(r["notool_fp"] - base_row["notool_fp"]),
                        "recall": float(r["recall"] - base_row["recall"]),
                    }
            hv = next((r for r in rows if r["variant"] == "high_vol"), None)
            ft = next((r for r in rows if r["variant"] == "fatal"), None)
            delta_high_vol_vs_fatal = None
            if hv is not None and ft is not None:
                delta_high_vol_vs_fatal = {
                    "iou_tiny": float(hv["iou_tiny"] - ft["iou_tiny"]),
                    "iou": float(hv["iou"] - ft["iou"]),
                    "notool_fp": float(hv["notool_fp"] - ft["notool_fp"]),
                }
            reason = (f"{best_row['variant']} ({best_row['label']}) leads on iou_tiny="
                      f"{best_row['iou_tiny']:.4f}")
            if base_row is not None and best_row["variant"] != "baseline":
                d = delta_vs_baseline[best_row["variant"]]
                reason += (f"; vs baseline tiny {d['iou_tiny']:+.4f}, "
                           f"IoU {d['iou']:+.4f}, ntFP {d['notool_fp']:+.0f}px")
            elif base_row is not None:
                reason += "; baseline is best on iou_tiny"

            aug_ablation_out = {
                "run": RUN_NAME, "run_id": RUN_ID,
                "fold": ABLATION_FOLD, "epochs": ABLATION_EPOCHS, "seed": SEED,
                "primary_threshold": PRIMARY_THRESHOLD,
                "select_on": "iou_tiny (overall iou + notool_fp reported; not adopted)",
                "tiny_pct_max": TINY_PCT_MAX,
                "aug_levels": AUG_LEVELS,
                "ablation_level": AUG_ABLATION_LEVEL,
                "ablation_p": float(AUG_ABLATION_P),
                "variants": {
                    k: {"label": v["label"],
                        "augs": (["baseline_recipe"] if k == "baseline"
                                 else list(v["augs"]) + ["ElasticTransform"])}
                    for k, v in AUG_ABLATION_VARIANTS.items()
                },
                "runs": rows,
                "best_variant": best_row["variant"],
                "best_iou_tiny": float(best_row["iou_tiny"]),
                "best_iou": float(best_row["iou"]),
                "best_notool_fp": float(best_row["notool_fp"]),
                "delta_vs_baseline": delta_vs_baseline,
                "delta_high_vol_vs_fatal": delta_high_vol_vs_fatal,
                "reason": reason,
                # Offline comparison hook: point this at the mini-sweep JSON after the run.
                "compare_with_sweep_hint": (
                    f"gs://{CKPT_BUCKET}/_latest_aug_sweep.json "
                    "(or the augsweep run's aug_sweep_results.json); "
                    "match on fold/seed/primary_threshold and compare iou_tiny/iou/notool_fp"
                ),
            }
            with open("/tmp/aug_ablation_results.json", "w") as f:
                json.dump(aug_ablation_out, f, indent=2)

            labels = [r["variant"] for r in rows]
            x = list(range(len(rows)))
            fig, ax = plt.subplots(1, 3, figsize=(15, 4))
            for j, (key, ttl, ylab) in enumerate([
                    ("iou_tiny",  f"tiny tools (<{TINY_PCT_MAX}% frame) @ thr {PRIMARY_THRESHOLD:.2f}",
                     "iou_tiny"),
                    ("iou",       "all tool-present frames", "iou"),
                    ("notool_fp", "FP area on tool-absent frames", "notool_fp (px)"),
            ]):
                vals = [r[key] for r in rows]
                ax[j].bar(x, vals, color=[FIG_SECONDARY, FIG_SERIES[1], FIG_SERIES[5]][:len(rows)])  # baseline, high_vol, fatal
                ax[j].set_xticks(x); ax[j].set_xticklabels(
                    [f"{r['variant']}\n{r['label']}" for r in rows], fontsize=8)
                ax[j].set_ylabel(ylab); ax[j].set_title(ttl); ax[j].grid(True, axis="y")
                wi = labels.index(best_row["variant"])
                ax[j].scatter([wi], [vals[wi]], c=FIG_PRIMARY, s=80, zorder=3, label="best iou_tiny")
                ax[j].legend(fontsize=8)
            fig.suptitle(f"Aug ablation (fold {ABLATION_FOLD}, {ABLATION_EPOCHS} epochs, "
                         f"level={AUG_ABLATION_LEVEL}, p={AUG_ABLATION_P})")
            fig.tight_layout()
            fig.savefig("/tmp/aug_ablation_curve.png", dpi=150); plt.show()

            fig2, ax2 = plt.subplots(figsize=(6, 5))
            nt = [r["notool_fp"] for r in rows]
            nt_size = [60 + 100 * (v / max(nt)) for v in nt] if max(nt) > 0 else [80] * len(rows)
            ax2.scatter([r["iou"] for r in rows], [r["iou_tiny"] for r in rows],
                        s=nt_size, c=[FIG_SERIES[i % len(FIG_SERIES)] for i in x], zorder=3)
            for r in rows:
                ax2.annotate(f"{r['variant']}\n{r['label']}", (r["iou"], r["iou_tiny"]),
                             textcoords="offset points", xytext=(6, 4), fontsize=8)
            ax2.set_xlabel("overall IoU (tool-present)")
            ax2.set_ylabel("iou_tiny")
            ax2.set_title("Ablation: tiny IoU vs overall IoU\n(marker size ~ notool_fp)")
            ax2.grid(True)
            fig2.tight_layout()
            fig2.savefig("/tmp/aug_ablation_tradeoff.png", dpi=150); plt.show()

            !gsutil -q cp /tmp/aug_ablation_results.json {GCS_CKPT_DIR}/aug_ablation_results.json
            !gsutil -q cp /tmp/aug_ablation_curve.png    {GCS_CKPT_DIR}/aug_ablation_curve.png
            !gsutil -q cp /tmp/aug_ablation_tradeoff.png {GCS_CKPT_DIR}/aug_ablation_tradeoff.png
            !gsutil -q cp /tmp/aug_ablation_results.json gs://{CKPT_BUCKET}/_latest_aug_ablation.json
            print("Saved aug ablation to", GCS_CKPT_DIR)

        print("\nvariant  | tinyIoU | bestthr |    IoU | recall | precis | miss | ntFP | augs")
        for r in rows:
            print(f"{r['variant']:8s} | {r['iou_tiny']:.4f} | "
                  f"{r['iou_tiny_bestthr']:.4f}  | {r['iou']:.4f} | {r['recall']:.4f} | "
                  f"{r['precision_strict']:.4f} | {r['miss_rate']:.3f} | {r['notool_fp']:.0f} | "
                  f"{'+'.join(r['augs'])}")
        print(f"\nBest on iou_tiny: {aug_ablation_out.get('best_variant')} "
              f"({aug_ablation_out.get('reason', 'from cache')})")

        # restore whatever the dataset / prior sweep left as the active transform
        train_transform = build_train_transform(AUG_PRESET if RUN_SYNTHETIC_AUG else None)

    except Exception as _e:
        import traceback; traceback.print_exc()
        print(f"Aug ablation failed ({type(_e).__name__}: {_e})")
        notify("error", "aug ablation failed", [f"{type(_e).__name__}: {_e}"])
        train_transform = build_train_transform(AUG_PRESET if RUN_SYNTHETIC_AUG else None)
    else:
        notify("fold", f"aug ablation done - {aug_ablation_out.get('best_variant', '?')}", [
            aug_ablation_out.get("reason", "from cache"),
            f"fold {ABLATION_FOLD} | {ABLATION_EPOCHS} epochs | level={AUG_ABLATION_LEVEL}",
        ])
else:
    print("Aug ablation skipped (RUN_AUG_ABLATION=False)")


### 12e. ROI-crop fine-tune (optional)

Only runs when `RUN_ROI_FINETUNE` is True. Fine-tunes checkpoints from `CASCADE_WEIGHTS_FROM` on a mix of whole frames and GT-centred ROI crops to acclimate the encoder to zoomed geometry.


In [ ]:
if RUN_ROI_FINETUNE:
    import gc

    assert not RESUME_FROM_GCS, (
        "set RESUME_FROM_GCS = False while fine-tuning; the CV loop still reuses these "
        "checkpoints through RESUME_INCOMPLETE")
    assert not RUN_STATIC_TILING, "tiling crops native tiles, so its geometry is not what the cascade feeds"

    print(f"ROI-crop fine-tune: {ROI_FT_EPOCHS} epochs x 5 folds, p={ROI_FT_P:.2f}, "
          f"pad {ROI_FT_PAD_RANGE[0]:.0%}-{ROI_FT_PAD_RANGE[1]:.0%}")
    print(f"  seeded from  : {CASCADE_WEIGHTS_FROM}")
    print(f"  small bias   : x{ROI_FT_SMALL_W:.2f} on tiny-or-thin frames, neg crops p={ROI_FT_NEG_P:.2f}")
    print(f"  writing to   : {GCS_CKPT_DIR}")

    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    _seed_dir = f"{DATA_ROOT}/roi_ft_seed/{CASCADE_WEIGHTS_FROM.rsplit('/', 2)[-2]}/{CASCADE_WEIGHTS_FROM.rsplit('/', 1)[-1]}"	# keyed by source run: the cache below skips the download when the file exists, so a seed left over from a different CASCADE_WEIGHTS_FROM would otherwise be fine-tuned from silently
    os.makedirs(_seed_dir, exist_ok=True)
    roi_ft_log = []

    for i in range(5):
        ckpt = os.path.join(CHECKPOINT_DIR, f"best_fold{i}.pt")
        if RESUME_INCOMPLETE and os.system(f"gsutil -q stat {GCS_CKPT_DIR}/best_fold{i}.pt") == 0:
            print(f"  fold {i}: fine-tuned checkpoint already in the run dir - skipping")
            continue
        _seed_ckpt = f"{_seed_dir}/best_fold{i}.pt"
        if not os.path.exists(_seed_ckpt):
            !gsutil -q cp {CASCADE_WEIGHTS_FROM}/best_fold{i}.pt {_seed_ckpt}

        train_pairs, val_pairs, test_pairs, train_idx = make_split(i)
        _w = sample_weight[train_idx].copy()
        if ROI_FT_SMALL_W != 1.0:
            _tool_m = size_cls[train_idx] > 0
            _sel = _tool_m & ((size_cls[train_idx] == 1) | (thin_cls[train_idx] == 1))
            _mass = _w[_tool_m].sum()
            _w[_sel] *= ROI_FT_SMALL_W
            _w[_tool_m] *= _mass / _w[_tool_m].sum()
        _, _, _, train_loader, val_loader, _ = build_loaders(
            train_pairs, val_pairs, test_pairs, _w, roi_crop=True)

        model = build_model(ARCHITECTURE, ENCODER, ENCODER_WEIGHTS).to(device)
        model.load_state_dict(torch.load(_seed_ckpt, map_location=device,
                                         weights_only=True)["model_state"])
        optimizer, scheduler = build_optimizer(model, lr=LR * 0.1, t_max=ROI_FT_EPOCHS)
        history, best = train_one_fold(model, optimizer, scheduler, train_loader, val_loader,
                                       ckpt, max_epochs=ROI_FT_EPOCHS, fold=i)
        !gsutil -q cp {ckpt} {GCS_CKPT_DIR}/best_fold{i}.pt

        roi_ft_log.append(f"  fold {i}: val_iou {best.get('val_iou', float('nan')):.4f} "
                          f"at epoch {best.get('epoch', -1)}")
        print("\n".join(roi_ft_log))
        del model, optimizer, scheduler
        gc.collect(); torch.cuda.empty_cache()

    print("\nROI-crop fine-tune complete. The CV loop below reuses these checkpoints, so its "
          "cascade scores are the fine-tuned arm, not the baseline weights.")
else:
    print("ROI-crop fine-tune: off")


## 13. Test-set evaluation
Define `evaluate()` to load a fold checkpoint and score the held-out test set with overlap, boundary, and false-positive metrics.

Called from section 11 for each fold.


In [ ]:
def _accumulate_scores(probs, masks, scores):
    """Threshold probs against masks and append every score-dict key for this batch.

    Shared by evaluate() (batches straight off test_loader) and evaluate_tiled() (one
    stitched, native-resolution-then-downsampled frame at a time, see section 13a) so the
    metric definitions can never drift apart between the baseline and the tiling arm.
    """
    for t in EVAL_THRESHOLDS:                       # lets any operating point be recomputed later
        p  = (probs > t).float()
        tp = (p * masks).sum(dim=(1, 2, 3))
        fp = (p * (1 - masks)).sum(dim=(1, 2, 3))
        fn = ((1 - p) * masks).sum(dim=(1, 2, 3))
        scores[f"tp@{t:.2f}"].extend(tp.cpu().numpy().tolist())
        scores[f"fp@{t:.2f}"].extend(fp.cpu().numpy().tolist())
        scores[f"fn@{t:.2f}"].extend(fn.cpu().numpy().tolist())

    preds = (probs > PRIMARY_THRESHOLD).float()
    tp = (preds * masks).sum(dim=(1, 2, 3))
    fp = (preds * (1 - masks)).sum(dim=(1, 2, 3))
    fn = ((1 - preds) * masks).sum(dim=(1, 2, 3))
    gt = tp + fn
    pp = tp + fp

    iou  = ((tp + 1e-6) / (tp + fp + fn + 1e-6)).cpu().numpy()
    dice = ((2 * tp + 1e-6) / (2 * tp + fp + fn + 1e-6)).cpu().numpy()

    prec = (tp / (pp + 1e-6)).cpu().numpy();  prec[pp.cpu().numpy() == 0] = np.nan
    rec  = (tp / (gt + 1e-6)).cpu().numpy();  rec[gt.cpu().numpy()  == 0] = np.nan

    scores["iou"].extend(iou.tolist())
    scores["dice"].extend(dice.tolist())
    scores["precision"].extend(prec.tolist())
    scores["recall"].extend(rec.tolist())
    scores["tp_pixels"].extend(tp.cpu().numpy().tolist())
    scores["fp_pixels"].extend(fp.cpu().numpy().tolist())
    scores["fn_pixels"].extend(fn.cpu().numpy().tolist())
    scores["pred_pixels"].extend(pp.cpu().numpy().tolist())
    scores["tool_pixels"].extend(gt.cpu().numpy().astype(int).tolist())
    scores["pred_empty"].extend((pp == 0).cpu().numpy().astype(int).tolist())
    scores["gt_empty"].extend((gt == 0).cpu().numpy().astype(int).tolist())

    preds_np = preds.cpu().numpy()[:, 0]
    masks_np = masks.cpu().numpy()[:, 0]
    for pred, gt_m in zip(preds_np, masks_np):
        d_p2g, d_g2p = _surface_distances(pred.astype(bool), gt_m.astype(bool))
        if d_p2g is None:
            scores["hd95"].append(float("nan")); scores["hd"].append(float("nan"))
            scores["msd"].append(float("nan"));  scores["asd"].append(float("nan"))
        else:
            scores["hd95"].append(max(np.percentile(d_p2g, 95), np.percentile(d_g2p, 95)))
            scores["hd"].append(max(d_p2g.max(), d_g2p.max()))     # full HD (noisiest)
            scores["msd"].append(d_p2g.mean())
            scores["asd"].append((d_p2g.mean() + d_g2p.mean()) / 2)

def _new_score_dict():
    """Empty score dict with every key evaluate() / evaluate_tiled() append to."""
    keys = ["iou", "dice", "precision", "recall", "hd95", "hd", "msd", "asd",
            "tool_pixels", "pred_pixels", "tp_pixels", "fp_pixels", "fn_pixels",
            "pred_empty", "gt_empty"]
    for t in EVAL_THRESHOLDS:
        keys += [f"tp@{t:.2f}", f"fp@{t:.2f}", f"fn@{t:.2f}"]
    return {k: [] for k in keys}

def evaluate(model, ckpt_path, test_loader, test_ds):
    if not test_ds:
        print("No test set - skipping.")
        return None

    ckpt = torch.load(ckpt_path, map_location=device, weights_only=True)
    model.load_state_dict(ckpt["model_state"])
    model.eval()

    scores = _new_score_dict()

    with torch.no_grad():
        for images, masks in tqdm(test_loader, desc="Test"):
            images, masks = images.to(device), masks.to(device)
            probs = model(images).float().sigmoid()
            _accumulate_scores(probs, masks, scores)

    n = len(scores["iou"])
    assert all(len(v) == n for v in scores.values()), "score arrays fell out of alignment"
    return scores

# === Predicted-mask export ===
import os
from pathlib import Path

PRED_MASK_DIR = Path("/tmp/PredictedMasks")

def pred_mask_path(img_path):
    """Destination for one frame's mask, mirroring the source layout under JPEGImages/NoTools."""
    for root, prefix in ((LOCAL_IMAGES_DIR, "JPEGImages"), (NOTOOLS_LOCAL, "NoTools")):
        rel = os.path.relpath(str(img_path), root)
        if not rel.startswith(".."):
            return PRED_MASK_DIR / prefix / Path(rel).with_suffix(".png")
    return PRED_MASK_DIR / "other" / Path(str(img_path)).with_suffix(".png").name

def save_pred_mask(img_path, pred_binary):
    """Write one predicted mask at source resolution in the labelmap's instrument colour."""
    h, w = pred_binary.shape
    colour = np.zeros((h, w, 3), dtype=np.uint8)
    colour[pred_binary == 1] = FG_COLOUR[::-1].tolist()      # cv2 writes BGR
    out = pred_mask_path(img_path)
    out.parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(out), colour)


### 13a. Static tiling test evaluation (stitched)

Scores test frames by stitching native-resolution tiles. Ensures fair full-frame comparisons for the static tiling arm.


In [ ]:
def evaluate_tiled(model, ckpt_path, test_pairs):
    """Static-tiling variant of evaluate(): stitches native-resolution tile predictions per
    frame instead of scoring test_loader batches directly. See markdown above for why.

    Also records n_tiles / n_tiles_with_tool per frame (only evaluate_tiled does this - the
    baseline evaluate() has no tile concept), so per_image_results.csv can show how much of
    each frame's tile grid actually covered the tool.
    """
    if not test_pairs:
        print("No test set - skipping.")
        return None

    ckpt = torch.load(ckpt_path, map_location=device, weights_only=True)
    model.load_state_dict(ckpt["model_state"])
    model.eval()

    scores = _new_score_dict()

    for img_path, mask_path in tqdm(test_pairs, desc="Test (tiled)"):
        image = cv2.imread(img_path)
        if image is None:
            raise FileNotFoundError(f"Could not read image: {img_path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        if mask_path == EMPTY_MASK:
            mask = np.zeros(image.shape[:2], dtype=np.float32)
        else:
            mask_rgb = cv2.imread(mask_path)
            if mask_rgb is None:
                raise FileNotFoundError(f"Could not read mask: {mask_path}")
            mask = rgb_mask_to_binary(cv2.cvtColor(mask_rgb, cv2.COLOR_BGR2RGB))

        if is_tileable(*image.shape[:2]):
            xs, ys = frame_tile_grid(*image.shape[:2])
            n_tool_tiles = len(tiles_with_tool(mask, xs, ys)) if mask.any() else 0
            scores.setdefault("n_tiles", []).append(len(xs) * len(ys))
            scores.setdefault("n_tiles_with_tool", []).append(n_tool_tiles)
        else:
            scores.setdefault("n_tiles", []).append(1)
            scores.setdefault("n_tiles_with_tool", []).append(1 if mask.any() else 0)

        probs_native = stitch_predict(model, image)
        if EXPORT_PRED_MASKS:
            save_pred_mask(img_path, (probs_native > PRIMARY_THRESHOLD).astype(np.uint8))
        probs_512 = cv2.resize(probs_native, (IMAGE_WIDTH, IMAGE_HEIGHT), interpolation=cv2.INTER_AREA)
        mask_512  = cv2.resize(mask,         (IMAGE_WIDTH, IMAGE_HEIGHT), interpolation=cv2.INTER_NEAREST)

        probs_t = torch.from_numpy(probs_512)[None, None].to(device)
        mask_t  = torch.from_numpy(mask_512)[None, None].to(device)
        _accumulate_scores(probs_t, mask_t, scores)

    n = len(scores["iou"])
    assert all(len(v) == n for v in scores.values()), "score arrays fell out of alignment"
    return scores


### 13b. Tracking cascade test evaluation

Scores test frames using the tracking cascade. Predictions are processed at native resolution, downsampled to 512x512, and accumulated identically to other arms.


In [ ]:
from collections import Counter

CASCADE_DIAG_DEFAULTS = {
    "cascade_tier": "none", "cascade_has_clip": 0, "cascade_lead_s": float("nan"),
    "cascade_n_track_frames": 0, "cascade_track_losses": 0, "cascade_tier1_hits": 0,
    "cascade_tier2_reinits": 0, "cascade_tier3_redetects": 0, "cascade_roi_empty": 0,
    "cascade_roi_frac": float("nan"), "cascade_roi_zoom": float("nan"),
    "cascade_track_scale": float("nan"), "cascade_n_rois": 0,
    "cascade_zoom_capped": 0, "cascade_roi_cover_frac": float("nan"),
}

def evaluate_cascade(model, ckpt_path, test_pairs):
    """Tracking-cascade variant of evaluate(): one padded ROI pass per frame, driven by the
    frame's own lead-in clip. See the markdown above for why the ordering is left alone.
    """
    if not test_pairs:
        print("No test set - skipping.")
        return None

    ckpt = torch.load(ckpt_path, map_location=device, weights_only=True)
    model.load_state_dict(ckpt["model_state"])
    model.eval()

    scores = _new_score_dict()

    for img_path, mask_path in tqdm(test_pairs, desc=f"Test (cascade/{CASCADE_VARIANT})"):
        image = cv2.imread(img_path)
        if image is None:
            raise FileNotFoundError(f"Could not read image: {img_path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        if mask_path == EMPTY_MASK:
            mask = np.zeros(image.shape[:2], dtype=np.float32)
        else:
            mask_rgb = cv2.imread(mask_path)
            if mask_rgb is None:
                raise FileNotFoundError(f"Could not read mask: {mask_path}")
            mask = rgb_mask_to_binary(cv2.cvtColor(mask_rgb, cv2.COLOR_BGR2RGB))

        clip_path, target_idx, fps, lead_s = clip_for(img_path)
        diag = {}
        probs_native = cascade_predict(model, image, clip_path, target_idx, fps,
                                       CASCADE_VARIANT, diag)
        if EXPORT_PRED_MASKS:
            save_pred_mask(img_path, (probs_native > PRIMARY_THRESHOLD).astype(np.uint8))

        row = dict(CASCADE_DIAG_DEFAULTS)
        row["cascade_tier"] = diag.get("final_tier", "none")
        row["cascade_has_clip"] = int(clip_path is not None)
        row["cascade_lead_s"] = lead_s if lead_s is not None else float("nan")
        for k in ("n_track_frames", "track_losses", "tier1_hits", "tier2_reinits",
                  "tier3_redetects", "roi_empty", "n_rois", "zoom_capped"):
            row[f"cascade_{k}"] = int(diag.get(k, 0))
        row["cascade_roi_frac"] = float(diag.get("roi_frac", float("nan")))
        row["cascade_track_scale"] = float(diag.get("track_scale", float("nan")))
        _box = diag.get("roi_box")
        if _box is not None:
            _base = min(IMAGE_WIDTH / image.shape[1], IMAGE_HEIGHT / image.shape[0])
            row["cascade_roi_zoom"] = float(min(IMAGE_WIDTH / _box[2],
                                                IMAGE_HEIGHT / _box[3]) / _base)
        _boxes = diag.get("roi_boxes")
        _gt = mask > 0.5
        if _boxes and _gt.any():
            _cov = np.zeros(_gt.shape, dtype=bool)
            for _bx, _by, _bw, _bh in _boxes:
                _cov[_by:_by + _bh, _bx:_bx + _bw] = True
            row["cascade_roi_cover_frac"] = float(_gt[_cov].sum()) / float(_gt.sum())
        for k, v in row.items():
            scores.setdefault(k, []).append(v)

        probs_512 = cv2.resize(probs_native, (IMAGE_WIDTH, IMAGE_HEIGHT), interpolation=cv2.INTER_AREA)
        mask_512  = cv2.resize(mask,         (IMAGE_WIDTH, IMAGE_HEIGHT), interpolation=cv2.INTER_NEAREST)

        probs_t = torch.from_numpy(probs_512)[None, None].to(device)
        mask_t  = torch.from_numpy(mask_512)[None, None].to(device)
        _accumulate_scores(probs_t, mask_t, scores)

    n = len(scores["iou"])
    assert all(len(v) == n for v in scores.values()), "score arrays fell out of alignment"

    _tiers = Counter(scores["cascade_tier"])
    print(f"  cascade tiers: " + "  ".join(
        f"{k}={v} ({v/n:.0%})" for k, v in _tiers.most_common()))
    _z = np.array(scores["cascade_roi_zoom"], float)
    _z = _z[~np.isnan(_z)]
    if len(_z):
        print(f"  roi zoom     : median {np.median(_z):.2f}x linear over baseline "
              f"(n={len(_z)} frames took the ROI path)")
    _cv = np.array(scores["cascade_roi_cover_frac"], float)
    _cv = _cv[~np.isnan(_cv)]
    if len(_cv):
        print(f"  roi coverage : median {np.median(_cv):.3f} of tool pixels inside the ROI boxes, "
              f"{(_cv < 0.99).mean():.0%} of ROI frames truncate the tool")
    return scores


## 14. Cross-validation

The main 5-fold cross-validation (14a), and optionally a training-set-size learning curve (14b). Both reuse the folds, loaders, and training loop defined above.


### 14a. 5-fold cross-validation
Train or resume each fold, evaluate on the held-out test set, and report mean ± std across folds. Checkpoints are synced to GCS.


In [ ]:
if RUN_CV:
    import json

    METRICS      = ["iou", "dice", "precision", "precision_strict", "recall",
                    "hd95", "hd", "msd", "asd"]
    SIZE_BUCKETS = [("tiny", 0.0, 4.5), ("small", 4.5, 8.0), ("med", 8.0, 19.0), ("large", 19.0, 1e9)]

    def _json_safe(o):
        """NaN/Inf are not valid JSON. Python reads them back, pandas and JS do not."""
        if isinstance(o, float) and not np.isfinite(o):
            return None
        if isinstance(o, dict):
            return {k: _json_safe(v) for k, v in o.items()}
        if isinstance(o, (list, tuple)):
            return [_json_safe(v) for v in o]
        return o

    def summarise(scores, idx):
        """Fold metrics, kept strictly separate for tool-present and tool-absent frames.

        idx is this fold's index array into all_pairs. test_loader uses shuffle=False and
        test_ds.pairs is built from folds[i] in order, so scores align with thin_cls[idx].
        """
        gt_empty   = np.array(scores["gt_empty"],   dtype=bool)
        pred_empty = np.array(scores["pred_empty"], dtype=bool)
        has_tool   = ~gt_empty
        frame_pct  = np.array(scores["tool_pixels"], float) / (IMAGE_HEIGHT * IMAGE_WIDTH) * 100
        out = {}

        for m in ["iou", "dice", "precision", "recall"]:
            v = np.array(scores[m], float)[has_tool]
            out[m] = float(np.nanmean(v)) if len(v) and not np.all(np.isnan(v)) else float("nan")

        _prec = np.array(scores["precision"], float)
        _prec_strict = _prec.copy(); _prec_strict[has_tool & pred_empty] = 0.0
        out["precision_n"]      = int((has_tool & ~np.isnan(_prec)).sum())
        out["precision_strict"] = float(np.nanmean(_prec_strict[has_tool])) if has_tool.any() else float("nan")

        for m in ["hd95", "hd", "msd", "asd"]:
            v  = np.array(scores[m], float)
            ok = ~np.isnan(v)
            out[m]          = float(v[ok].mean()) if ok.any() else float("nan")
            out[f"{m}_n"]   = int(ok.sum())

        out["n_tool"]   = int(has_tool.sum())
        out["n_notool"] = int(gt_empty.sum())
        out["miss_rate"] = float(pred_empty[has_tool].mean()) if has_tool.any() else float("nan")
        _fp_nt = np.array(scores["fp_pixels"], float)[gt_empty]
        out["notool_clean_rate"] = float(pred_empty[gt_empty].mean())  if gt_empty.any() else float("nan")
        out["notool_fp_mean"]    = float(_fp_nt.mean())                if gt_empty.any() else float("nan")
        out["notool_fp_median"]  = float(np.median(_fp_nt))            if gt_empty.any() else float("nan")
        out["notool_fp_p95"]     = float(np.percentile(_fp_nt, 95))    if gt_empty.any() else float("nan")

        for name, lo, hi in SIZE_BUCKETS:
            sel = has_tool & (frame_pct >= lo) & (frame_pct < hi)
            out[f"iou_{name}"]    = float(np.nanmean(np.array(scores["iou"],    float)[sel])) if sel.any() else float("nan")
            out[f"recall_{name}"] = float(np.nanmean(np.array(scores["recall"], float)[sel])) if sel.any() else float("nan")
            out[f"n_{name}"]      = int(sel.sum())

        fold_thin = thin_cls[idx]
        assert len(fold_thin) == len(has_tool), "thin_cls and the score arrays are misaligned"
        for name, cls in [("thin", 1), ("wide", 2)]:
            sel = has_tool & (fold_thin == cls)
            out[f"iou_{name}"]    = float(np.nanmean(np.array(scores["iou"],    float)[sel])) if sel.any() else float("nan")
            out[f"recall_{name}"] = float(np.nanmean(np.array(scores["recall"], float)[sel])) if sel.any() else float("nan")
            out[f"n_{name}"]      = int(sel.sum())
        return out

    cv_summary, cv_best = [], []
    cv_histories = {}
    viz_scores, viz_pairs, viz_ckpt, viz_fold = None, [], [], []
    fold_log = []
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)

    for i in range(5):
        if fold_log:
            print("\nCompleted so far:\n" + "\n".join(fold_log))
        print(f"\n{'='*20} Fold {i} {'='*20}")

        train_pairs, val_pairs, test_pairs, train_idx = make_split(i)
        _, _, test_ds, train_loader, val_loader, test_loader = build_loaders(
            train_pairs, val_pairs, test_pairs, sample_weight[train_idx])
        ckpt = os.path.join(CHECKPOINT_DIR, f"best_fold{i}.pt")

        # Fold-level resume
        history   = None
        _has_ckpt = os.system(f"gsutil -q stat {GCS_CKPT_DIR}/best_fold{i}.pt") == 0
        if RESUME_FROM_GCS or (RESUME_INCOMPLETE and _has_ckpt):
            !gsutil -q cp {GCS_CKPT_DIR}/best_fold{i}.pt {ckpt}
            model = build_model(ARCHITECTURE, ENCODER, ENCODER_WEIGHTS).to(device)
            best  = {}
            print(f"  fold {i} checkpoint already in the run dir - reusing, not retraining")
        else:
            model = build_fresh_model()
            optimizer, scheduler = build_optimizer(model)
            history, best = train_one_fold(
                model, optimizer, scheduler, train_loader, val_loader, ckpt, fold=i)
            cv_histories[i] = history
            !gsutil -q cp {ckpt} {GCS_CKPT_DIR}/best_fold{i}.pt

        scores = (evaluate_cascade(model, ckpt, test_pairs) if RUN_TRACKING_CASCADE
                  else evaluate_tiled(model, ckpt, test_pairs) if RUN_STATIC_TILING
                  else evaluate(model, ckpt, test_loader, test_ds))
        if viz_scores is None:
            viz_scores = {k: [] for k in scores}
        for k in viz_scores: viz_scores[k].extend(scores[k])
        viz_pairs.extend(test_ds.pairs)
        viz_ckpt.extend([ckpt] * len(test_ds.pairs))
        viz_fold.extend([i] * len(test_ds.pairs))

        row = summarise(scores, folds[i])
        cv_summary.append(row)
        cv_best.append(best)

        # Persist per fold to GCS
        with open(f"/tmp/fold{i}_scores.json", "w") as f:
            json.dump(_json_safe({"fold": i, "summary": row, "best": best, "history": history,
                                  "images": [p[0] for p in test_ds.pairs], "scores": scores}), f)
        !gsutil -q cp /tmp/fold{i}_scores.json {GCS_CKPT_DIR}/fold{i}_scores.json

        fold_log.append(
            f"Fold {i}: IoU={row['iou']:.4f} Dice={row['dice']:.4f} "
            f"P={row['precision']:.4f} R={row['recall']:.4f} | "
            f"miss={row['miss_rate']:.3f} notool-clean={row['notool_clean_rate']:.3f} | "
            f"tiny IoU={row['iou_tiny']:.4f} (n={row['n_tiny']})")
        print(fold_log[-1])

        _best_ep = best.get("epoch", "?")
        _best_iou = best.get("val_iou", float("nan"))
        _epochs_tr = best.get("epochs_trained", "?")
        notify("fold", f"fold {i} done", [
            f"test IoU {row['iou']:.4f} | Dice {row['dice']:.4f} | P {row['precision']:.4f} | R {row['recall']:.4f}",
            f"best val IoU {_best_iou:.4f} @ ep{_best_ep} | epochs {_epochs_tr} | tiny IoU {row['iou_tiny']:.4f}",
            f"miss {row['miss_rate']:.3f} | notool-clean {row['notool_clean_rate']:.3f}",
        ])

    print("\n" + "="*70)
    print("Completed folds:\n" + "\n".join(fold_log))

    # Source vs resized no-tool counts
    _n_notool_eval = int(np.sum([row["n_notool"] for row in cv_summary]))
    _n_notool_src  = int((~tool).sum())
    _n_notool_vanished = _n_notool_eval - _n_notool_src
    print(f"\nTool-absent frames - source masks: {_n_notool_src}  |  after "
          f"{IMAGE_HEIGHT}x{IMAGE_WIDTH} resize: {_n_notool_eval}")
    if _n_notool_vanished:
        print(f"WARNING: {_n_notool_vanished} frames lost their instrument entirely in "
              f"the resize - the two size definitions will not reconcile in the write-up")
    notify("fold", "CV summary · no-tool resize check", [
        f"source {_n_notool_src} | eval {_n_notool_eval} | vanished {_n_notool_vanished}",
        (f"WARNING: {_n_notool_vanished} frames empty after resize"
         if _n_notool_vanished else "source and eval no-tool counts match"),
    ])

    def mstd(key):
        v = np.array([row[key] for row in cv_summary], float)
        return np.nanmean(v), np.nanstd(v, ddof=1)      # ddof=1: 5 folds is a sample, not a population

    print(f"\n{'='*20} 5-fold results (mean ± std across folds, ddof=1) {'='*20}")
    print("Tool-present frames only:")
    for m in METRICS:
        mu, sd = mstd(m)
        extra = f"   (n={int(np.sum([row[m+'_n'] for row in cv_summary]))} frames)" if f"{m}_n" in cv_summary[0] else ""
        print(f"  {m.upper():10s} {mu:.4f} ± {sd:.4f}{extra}")
    print("\nDirectional failure rates:")
    print("  (notool_clean_rate = share of blank frames with EXACTLY zero predicted pixels,")
    print("   not pixel-level specificity - read it next to the FP-area percentiles)")
    for m in ["miss_rate", "notool_clean_rate", "notool_fp_mean",
              "notool_fp_median", "notool_fp_p95"]:
        mu, sd = mstd(m)
        print(f"  {m:20s} {mu:.4f} ± {sd:.4f}")
    print("\nIoU by tool size (the weak-region breakdown the hypothesis targets):")
    for name, _, _ in SIZE_BUCKETS:
        mu, sd = mstd(f"iou_{name}")
        n = int(np.sum([row[f'n_{name}'] for row in cv_summary]))
        print(f"  {name:6s} IoU {mu:.4f} ± {sd:.4f}   recall {mstd(f'recall_{name}')[0]:.4f}   (n={n})")
    print(f"\nIoU by tool thinness (mean width cut at {THIN_CUT:.1f}px = p25 of tool frames):")
    for name in ["thin", "wide"]:
        mu, sd = mstd(f"iou_{name}")
        n = int(np.sum([row[f'n_{name}'] for row in cv_summary]))
        print(f"  {name:6s} IoU {mu:.4f} ± {sd:.4f}   recall {mstd(f'recall_{name}')[0]:.4f}   (n={n})")
    print(f"\nTest frames pooled: {len(viz_pairs)}  "
          f"(tool {int(np.sum([row['n_tool'] for row in cv_summary]))}, "
          f"no-tool {int(np.sum([row['n_notool'] for row in cv_summary]))})")
    print("\nNote on the ± above: 5 folds is 4 degrees of freedom, and with ~260 videos per\n"
          "fold the spread is dominated by video identity rather than model variance. Do NOT\n"
          "use it for arm-vs-arm significance - use a paired per-video bootstrap over\n"
          "per_image_results.csv, which the next cells write for exactly that purpose.")


### 14b. Power analysis (optional)

Trains on nested, size-stratified subsets to evaluate dataset size effects on accuracy, particularly for small tools.


In [ ]:
if RUN_POWER:
    import json, matplotlib.pyplot as plt

    def _json_safe(o):
        """NaN/Inf are not valid JSON. Python reads them back, pandas and JS do not."""
        if isinstance(o, float) and not np.isfinite(o):
            return None
        if isinstance(o, dict):
            return {k: _json_safe(v) for k, v in o.items()}
        if isinstance(o, (list, tuple)):
            return [_json_safe(v) for v in o]
        return o

    def _power_summarise(scores, idx):
        """Same metrics as CV summarise; local so RUN_POWER works with RUN_CV off."""
        gt_empty   = np.array(scores["gt_empty"],   dtype=bool)
        pred_empty = np.array(scores["pred_empty"], dtype=bool)
        has_tool   = ~gt_empty
        frame_pct  = np.array(scores["tool_pixels"], float) / (IMAGE_HEIGHT * IMAGE_WIDTH) * 100
        out = {}
        for m in ["iou", "dice", "precision", "recall"]:
            v = np.array(scores[m], float)[has_tool]
            out[m] = float(np.nanmean(v)) if len(v) and not np.all(np.isnan(v)) else float("nan")
        _prec = np.array(scores["precision"], float)
        _prec_strict = _prec.copy(); _prec_strict[has_tool & pred_empty] = 0.0
        out["precision_n"]      = int((has_tool & ~np.isnan(_prec)).sum())
        out["precision_strict"] = float(np.nanmean(_prec_strict[has_tool])) if has_tool.any() else float("nan")
        for m in ["hd95", "hd", "msd", "asd"]:
            v  = np.array(scores[m], float)
            ok = ~np.isnan(v)
            out[m]        = float(v[ok].mean()) if ok.any() else float("nan")
            out[f"{m}_n"] = int(ok.sum())
        out["n_tool"]   = int(has_tool.sum())
        out["n_notool"] = int(gt_empty.sum())
        out["miss_rate"] = float(pred_empty[has_tool].mean()) if has_tool.any() else float("nan")
        _fp_nt = np.array(scores["fp_pixels"], float)[gt_empty]
        out["notool_clean_rate"] = float(pred_empty[gt_empty].mean()) if gt_empty.any() else float("nan")
        out["notool_fp_mean"]    = float(_fp_nt.mean())               if gt_empty.any() else float("nan")
        out["notool_fp_median"]  = float(np.median(_fp_nt))           if gt_empty.any() else float("nan")
        out["notool_fp_p95"]     = float(np.percentile(_fp_nt, 95))   if gt_empty.any() else float("nan")
        for name, lo, hi in [("tiny", 0.0, 4.5), ("small", 4.5, 8.0), ("med", 8.0, 19.0), ("large", 19.0, 1e9)]:
            sel = has_tool & (frame_pct >= lo) & (frame_pct < hi)
            out[f"iou_{name}"]    = float(np.nanmean(np.array(scores["iou"],    float)[sel])) if sel.any() else float("nan")
            out[f"recall_{name}"] = float(np.nanmean(np.array(scores["recall"], float)[sel])) if sel.any() else float("nan")
            out[f"n_{name}"]      = int(sel.sum())
        fold_thin = thin_cls[idx]
        assert len(fold_thin) == len(has_tool), "thin_cls and the score arrays are misaligned"
        for name, cls in [("thin", 1), ("wide", 2)]:
            sel = has_tool & (fold_thin == cls)
            out[f"iou_{name}"]    = float(np.nanmean(np.array(scores["iou"],    float)[sel])) if sel.any() else float("nan")
            out[f"recall_{name}"] = float(np.nanmean(np.array(scores["recall"], float)[sel])) if sel.any() else float("nan")
            out[f"n_{name}"]      = int(sel.sum())
        return out

    SIZE_BINS = ["tiny", "small", "med", "large"]	# same cuts as _power_summarise / size_cls

    train_pairs_full, val_pairs, test_pairs, train_idx_full = make_split(POWER_FOLD)	# fixed val + test
    test_idx = folds[POWER_FOLD]

    _rng = np.random.default_rng(SEED)
    _cls = size_cls[train_idx_full]	# 0 no-tool / 1 tiny / 2 small / 3 med / 4 large
    _rank = np.empty(len(_cls), float)
    for c in np.unique(_cls):
        pos = np.where(_cls == c)[0]
        _rank[_rng.permutation(pos)] = (np.arange(len(pos)) + 0.5) / len(pos)
    order = np.argsort(_rank, kind="stable")

    curve = []
    for n in POWER_SIZES:
        n = min(n, len(train_pairs_full))
        sub = order[:n]
        mix = np.bincount(size_cls[train_idx_full[sub]], minlength=5) / n
        print(f"n={n:5d}  train mix: no-tool {mix[0]:.2f}  tiny {mix[1]:.2f}  "
              f"small {mix[2]:.2f}  med {mix[3]:.2f}  large {mix[4]:.2f}")

        train_pairs = [train_pairs_full[j] for j in sub]
        train_weights = sample_weight[train_idx_full[sub]]	# same weighted sampling as CV
        _, _, test_ds, train_loader, val_loader, test_loader = build_loaders(
            train_pairs, val_pairs, test_pairs, train_weights)

        model = build_fresh_model()
        optimizer, scheduler = build_optimizer(model)
        ckpt = os.path.join(CHECKPOINT_DIR, f"power_{n}.pt")
        history, best = train_one_fold(
            model, optimizer, scheduler, train_loader, val_loader, ckpt, fold=POWER_FOLD)
        !gsutil -q cp {ckpt} {GCS_CKPT_DIR}/power_{n}.pt

        scores = evaluate(model, ckpt, test_loader, test_ds)
        summary = _power_summarise(scores, test_idx)
        row = {
            "n": n,
            "iou": summary["iou"], "dice": summary["dice"],
            "precision": summary["precision"], "recall": summary["recall"],
            "miss_rate": summary["miss_rate"],
            "best_val_iou": best.get("val_iou"),
            "best_epoch": best.get("epoch"),
            "epochs_trained": best.get("epochs_trained"),
        }
        for i, k in enumerate(SIZE_BINS):	# per-size test IoU for the learning-curve-by-bin plot
            row[f"iou_{k}"] = summary[f"iou_{k}"]
            row[f"n_{k}"] = summary[f"n_{k}"]
        curve.append(row)

        with open(f"/tmp/power_{n}_scores.json", "w") as f:
            json.dump(_json_safe({"n": n, "fold": POWER_FOLD, "summary": summary,
                                  "best": best, "history": history}), f, indent=2)
        !gsutil -q cp /tmp/power_{n}_scores.json {GCS_CKPT_DIR}/power_{n}_scores.json

        print(f"n={n:5d}  IoU={summary['iou']:.4f}  Dice={summary['dice']:.4f}  "
              f"P={summary['precision']:.4f}  R={summary['recall']:.4f}  "
              f"miss={summary['miss_rate']:.3f}")
        size_bits = [f"{k} {summary['iou_' + k]:.4f} (n={summary['n_' + k]})" for k in SIZE_BINS]
        print("         IoU by tool size: " + "  ".join(size_bits))
        notify("fold", f"power n={n} done", [
            f"test IoU {summary['iou']:.4f} | Dice {summary['dice']:.4f} | "
            f"P {summary['precision']:.4f} | R {summary['recall']:.4f}",
            f"miss {summary['miss_rate']:.3f} | best val IoU {best.get('val_iou', float('nan')):.4f} "
            f"@ ep{best.get('epoch', '?')} | {len(curve)}/{len(POWER_SIZES)} sizes",
            "IoU by size: " + "  ".join(f"{k} {summary['iou_' + k]:.4f}" for k in SIZE_BINS),
        ])

    ns = [c["n"] for c in curve]
    plt.figure()
    plt.plot(ns, [c["iou"] for c in curve], "o-", color=FIG_PRIMARY, label="IoU")
    plt.plot(ns, [c["dice"] for c in curve], "s-", color=FIG_SERIES[1], label="Dice")
    plt.xlabel("# training images"); plt.ylabel("Test score")
    plt.title(f"Learning curve (fixed fold {POWER_FOLD})"); plt.grid(True); plt.legend()
    plt.savefig("/tmp/power_curve.png", dpi=150); plt.show()

    # Same curve, split by tool-size bin on the fixed test fold
    plt.figure()
    for k in SIZE_BINS:
        n_test = curve[-1][f"n_{k}"]
        plt.plot(ns, [c[f"iou_{k}"] for c in curve], "o-", color=FIG_SERIES[i % len(FIG_SERIES)],
                 label=f"{k} (n_test={n_test})")
    plt.xlabel("# training images"); plt.ylabel("Test IoU")
    plt.title(f"Learning curve by tool size (fixed fold {POWER_FOLD})")
    plt.grid(True); plt.legend()
    plt.savefig("/tmp/power_curve_size.png", dpi=150); plt.show()

    power_out = {
        "run": RUN_NAME, "run_id": RUN_ID,
        "fold": POWER_FOLD, "seed": SEED, "sizes": list(POWER_SIZES),
        "curve": curve,
    }
    with open("/tmp/power_analysis.json", "w") as f:
        json.dump(_json_safe(power_out), f, indent=2)
    !gsutil -q cp /tmp/power_analysis.json {GCS_CKPT_DIR}/power_analysis.json
    !gsutil -q cp /tmp/power_curve.png     {GCS_CKPT_DIR}/power_curve.png
    !gsutil -q cp /tmp/power_curve_size.png {GCS_CKPT_DIR}/power_curve_size.png
    notify("fold", "power analysis done", [
        f"fold {POWER_FOLD} | sizes {[c['n'] for c in curve]}",
        f"IoU {[round(c['iou'], 4) for c in curve]}",
        f"tiny IoU {[round(c['iou_tiny'], 4) for c in curve]}",
    ])
else:
    print("Power analysis skipped (RUN_POWER=False)")


### 15. Threshold re-evaluation across arms (inference only) (optional)

Re-evaluates finished checkpoints across a grid of thresholds (`EVAL_THRESHOLDS`). Allows for matched-precision comparisons between arms with different recall biases.


In [ ]:
if RUN_THRESHOLD_REEVAL:
	import json, gc
	import numpy as np
	from torch.utils.data import DataLoader

	ARMS = THRESHOLD_REEVAL_ARMS
	REFERENCE_ARM = THRESHOLD_REEVAL_REFERENCE
	_TKEYS = [f"{t:.2f}" for t in REEVAL_THRESHOLDS]

	_reeval_transform = A.Compose([
		A.Resize(IMAGE_HEIGHT, IMAGE_WIDTH),
		A.Normalize(mean=_mean, std=_std),
		ToTensorV2(),
	])

	def _eval_loader(pairs):
		return DataLoader(BinarySegDataset(pairs, _reeval_transform), batch_size=BATCH_SIZE,
		                  shuffle=False, num_workers=NUM_WORKERS, pin_memory=True,
		                  persistent_workers=False, worker_init_fn=eval_worker_init,
		                  prefetch_factor=2 if NUM_WORKERS > 0 else None)

	def _scan(model, loader, components=False):
		"""Per-frame TP/FP/FN at every threshold, optionally with mean loss components.

		The loader is unshuffled, so row order matches the fold's index order and the
		stratum labels can be joined positionally.
		"""
		acc = {t: ([], [], []) for t in REEVAL_THRESHOLDS}
		bs = ds = n = 0.0
		with torch.no_grad():
			for images, masks in loader:
				images = images.to(device, non_blocking=True)
				masks  = masks.to(device, non_blocking=True)
				logits = model(images).float()
				if components:
					bs += bce_loss(logits, masks).item() * images.size(0)
					ds += dice_loss(logits, masks).item() * images.size(0)
					n  += images.size(0)
				for t in REEVAL_THRESHOLDS:
					tp, fp, fn = _counts(logits, masks, t)
					acc[t][0].extend(tp.cpu().tolist())
					acc[t][1].extend(fp.cpu().tolist())
					acc[t][2].extend(fn.cpu().tolist())
		counts = {t: tuple(np.array(a, float) for a in v) for t, v in acc.items()}
		comp = {"bce": bs / n, "overlap": ds / n} if components and n else None
		return counts, comp

	def _metrics(tp, fp, fn, scls):
		gt, pp = tp + fn, tp + fp
		has  = gt > 0
		iou  = (tp + 1e-6) / (tp + fp + fn + 1e-6)
		rec  = np.divide(tp, gt, out=np.zeros_like(tp), where=has)
		prec = np.divide(tp, pp, out=np.zeros_like(tp), where=pp > 0)
		tiny = has & (scls == 1)            # same tiny definition the CV eval uses
		return {"iou": float(iou[has].mean()), "recall": float(rec[has].mean()),
		        "precision_strict": float(prec[has].mean()),
		        "iou_tiny": float(iou[tiny].mean()) if tiny.any() else float("nan"),
		        "recall_tiny": float(rec[tiny].mean()) if tiny.any() else float("nan"),
		        "miss_rate": float((pp[has] == 0).mean()),
		        "notool_clean_rate": float((fp[~has] == 0).mean()) if (~has).any() else float("nan"),
		        "notool_fp_mean": float(fp[~has].mean()) if (~has).any() else float("nan")}

	out = {}
	for arm, gcs in ARMS.items():
		print(f"\n=== {arm} ===")
		per_fold, comps = {"val": [], "test": []}, []
		for i in range(5):
			ck = f"{CHECKPOINT_DIR}/reeval_fold{i}.pt"
			!gsutil -q cp {gcs}/best_fold{i}.pt {ck}
			model = build_model(ARCHITECTURE, ENCODER, ENCODER_WEIGHTS).to(device)
			model.load_state_dict(torch.load(ck, map_location=device,
			                                 weights_only=True)["model_state"])
			model.eval()

			_, val_pairs, test_pairs, _ = make_split(i)
			val_idx, test_idx = folds[(i + 1) % 5], folds[i]
			for split, pairs, idx in [("val", val_pairs, val_idx), ("test", test_pairs, test_idx)]:
				counts, comp = _scan(model, _eval_loader(pairs), components=(split == "val"))
				per_fold[split].append({f"{t:.2f}": _metrics(*counts[t], size_cls[idx])
				                        for t in REEVAL_THRESHOLDS})
				if comp:
					comps.append(comp)
			del model; os.remove(ck); gc.collect(); torch.cuda.empty_cache()
			print(f"  fold {i} done")

		# one threshold per arm, chosen on pooled validation IoU, then applied to test
		val_iou = {t: float(np.mean([f[t]["iou"] for f in per_fold["val"]])) for t in _TKEYS}
		sel = max(val_iou, key=val_iou.get)
		bce_m = float(np.mean([c["bce"] for c in comps]))
		ovl_m = float(np.mean([c["overlap"] for c in comps]))
		out[arm] = {"per_fold": per_fold, "val_iou_by_threshold": val_iou,
		            "selected_threshold": sel, "thresholds": _TKEYS,
		            "loss_components": {"bce": bce_m, "overlap": ovl_m,
		                                "effective_ratio": ovl_m / (bce_m + 1e-9)}}
		print(f"  val-selected threshold {sel} (val IoU {val_iou[sel]:.4f}) | "
		      f"components BCE {bce_m:.4f} vs overlap {ovl_m:.4f} "
		      f"(nominal 1:1, effective 1:{ovl_m / (bce_m + 1e-9):.1f})")
		if sel in (_TKEYS[0], _TKEYS[-1]):
			print("  selected threshold is on a grid edge, widen REEVAL_THRESHOLDS")

	KEYS = ["iou", "precision_strict", "recall", "iou_tiny", "recall_tiny",
	        "notool_clean_rate", "notool_fp_mean", "miss_rate"]
	_tmean = lambda r, t, k: float(np.mean([f[t][k] for f in r["per_fold"]["test"]]))

	for arm, r in out.items():
		sel = r["selected_threshold"]
		print(f"\n{arm}: test @ 0.50 -> test @ val-selected {sel}")
		for k in KEYS:
			a, b = _tmean(r, "0.50", k), _tmean(r, sel, k)
			print(f"  {k:20s} {a:10.4f} -> {b:10.4f}  ({b - a:+.4f})")

	ref_p = _tmean(out[REFERENCE_ARM], "0.50", "precision_strict")
	print(f"\nmatched-precision comparison (reference {REFERENCE_ARM} precision {ref_p:.4f})")
	for arm, r in out.items():
		m = min(_TKEYS, key=lambda t: abs(_tmean(r, t, "precision_strict") - ref_p))
		print(f"  {arm:16s} thr {m} | P {_tmean(r, m, 'precision_strict'):.4f} | "
		      f"R {_tmean(r, m, 'recall'):.4f} | IoU {_tmean(r, m, 'iou'):.4f} | "
		      f"tiny {_tmean(r, m, 'iou_tiny'):.4f} | "
		      f"clean {_tmean(r, m, 'notool_clean_rate'):.4f}")

	threshold_reeval_out = out
	with open("/tmp/threshold_reeval.json", "w") as f:
		json.dump(out, f, indent=2)
	!gsutil -q cp /tmp/threshold_reeval.json {GCS_CKPT_DIR}/threshold_reeval.json
	print(f"\nsaved: {GCS_CKPT_DIR}/threshold_reeval.json")
else:
	print("Threshold re-eval skipped (RUN_THRESHOLD_REEVAL=False)")


## 16. Training curves
Plot per-fold train/val loss, val IoU, val Dice, and val no-tool false-positive area from `cv_histories`. Results and the figure are saved to GCS in the next cell.

Only folds that actually trained in this session contribute a curve, so this is empty when every fold was resumed from a checkpoint (`RESUME_FROM_GCS=True`, or `RESUME_INCOMPLETE` finding all five already done). The same curves are written per fold to `fold{i}_scores.json` in GCS regardless.


In [ ]:
if RUN_CV:
    import matplotlib.pyplot as plt

    if not globals().get("cv_histories"):
        print("No training histories to plot - a fold that was resumed from a checkpoint did no "
              "training, so it contributes no curve. Per-fold curves are also written to "
              "fold{i}_scores.json in GCS.")
    else:
        fig, axes = plt.subplots(1, 4, figsize=(20, 4))

        for i, h in sorted(cv_histories.items()):
            axes[0].plot(h["train_loss"], color=FIG_SECONDARY, alpha=0.4)
            axes[0].plot(h["val_loss"],   color=FIG_PRIMARY,   alpha=0.4)
            axes[1].plot(h["val_iou"],  color=FIG_SERIES[i % len(FIG_SERIES)], label=f"Fold {i}")
            axes[2].plot(h["val_dice"], color=FIG_SERIES[i % len(FIG_SERIES)], label=f"Fold {i}")
            axes[3].plot(h["val_notool_fp"], color=FIG_SERIES[i % len(FIG_SERIES)], label=f"Fold {i}")

        axes[0].set_title("Loss: train (grey) vs val (red)")
        axes[1].set_title("Val IoU");  axes[1].legend(fontsize=8)
        axes[2].set_title("Val Dice (reference)"); axes[2].legend(fontsize=8)
        axes[3].set_title("Val no-tool FP (px/frame) - ignored by val_iou")
        axes[3].legend(fontsize=8)
        for ax in axes:
            ax.set_xlabel("Epoch")

        plt.tight_layout()
        plt.savefig("/tmp/training_curves.png", dpi=150)
        plt.show()
else:
    print("Skipped: needs a completed RUN_CV pass (cv_histories not built)")


## 17. Save results
Write the run summary (config + per-fold metrics + mean±std) to `results.json`, per-image results  to `per_image_results.csv`, and training curves to this run's GCS folder.


In [ ]:
if RUN_CV:
    import json, sys
    import sklearn, albumentations, segmentation_models_pytorch

    _summary_keys = [k for k in cv_summary[0] if isinstance(cv_summary[0][k], float)]

    def _aug_spec(t):
        try:
            return albumentations.to_dict(t)
        except Exception as e:
            return {"error": f"{type(e).__name__}: {e}", "repr": repr(t)}

    def _cascade_tier_shares():
        """Share of pooled test frames each cascade tier produced."""
        if not (RUN_TRACKING_CASCADE and "cascade_tier" in viz_scores):
            return None
        t = viz_scores["cascade_tier"]
        return {k: round(t.count(k) / len(t), 4) for k in sorted(set(t))}

    def _cascade_zoom_summary():
        """Linear zoom the tool actually received relative to the baseline downsample."""
        if not (RUN_TRACKING_CASCADE and "cascade_roi_zoom" in viz_scores):
            return None
        z = np.array(viz_scores["cascade_roi_zoom"], float)
        z = z[~np.isnan(z)]
        if not len(z):
            return {"n_roi_frames": 0}
        return {"n_roi_frames": int(len(z)), "median": float(np.median(z)),
                "p25": float(np.percentile(z, 25)), "p75": float(np.percentile(z, 75)),
                "frac_no_gain": float((z <= 1.01).mean())}

    def _cascade_cover_summary():
        """How much of the tool the ROI boxes contained. Under the pre-fix "replace"
        combine every pixel outside them was a false negative no threshold could recover.
        """
        if not (RUN_TRACKING_CASCADE and "cascade_roi_cover_frac" in viz_scores):
            return None
        c = np.array(viz_scores["cascade_roi_cover_frac"], float)
        c = c[~np.isnan(c)]
        if not len(c):
            return {"n_roi_frames": 0}
        return {"n_roi_frames": int(len(c)), "median": float(np.median(c)),
                "mean": float(c.mean()), "frac_truncated": float((c < 0.99).mean())}

    def _tile_crop_bias_summary():
        """Evidence for the crop-bias trade-off noted in section 7a: pick_tile_origin
        restricts train/val crops to tool-containing tiles, so a tool-present frame's
        candidate tile set is narrower than its full geometric grid - this is how much
        narrower, in aggregate.

        No extra image reads: reuses n_tiles / n_tiles_with_tool, which evaluate_tiled
        (13a) already records per frame as part of normal test scoring. 5-fold CV rotates
        every frame through the test set exactly once, so the pooled viz_scores already
        covers the whole dataset, not just a train-fold subset - a second read pass over
        the masks would only recompute numbers that already exist.
        """
        if not (RUN_STATIC_TILING and "n_tiles" in viz_scores):
            return None
        n_tiles      = np.array(viz_scores["n_tiles"])
        n_tool_tiles = np.array(viz_scores["n_tiles_with_tool"])
        has_tool = n_tool_tiles > 0    # n_tiles_with_tool is 0 exactly for no-tool frames
        if not has_tool.any():
            return {"n_tool_frames": 0}
        coverage = n_tool_tiles[has_tool] / n_tiles[has_tool]
        return {
            "n_tool_frames": int(has_tool.sum()),
            "mean_coverage_frac": float(coverage.mean()),
            "median_coverage_frac": float(np.median(coverage)),
            "single_tile_frac": float((n_tool_tiles[has_tool] == 1).mean()),
        }

    final = {
        "run": RUN_NAME,
        "run_id": RUN_ID,
        "config": {"arch": ARCHITECTURE, "encoder": ENCODER, "encoder_weights": ENCODER_WEIGHTS,
                   "loss": LOSS, "lr": LR, "lr_source": LR_SOURCE,
                   **({"tversky_alpha": TVERSKY_ALPHA, "tversky_beta": TVERSKY_BETA,
                       "bce_tversky_lambda": BCE_TVERSKY_LAMBDA,
                       "loss_balance_sweep": globals().get("balance_out")}
                      if LOSS == "bce_tversky" else {}),
                   **({"bce_dice_lambda": BCE_DICE_LAMBDA} if LOSS == "bce_dice" else {}),
                   "batch_size": BATCH_SIZE, "num_epochs": NUM_EPOCHS,
                   "patience": PATIENCE, "min_delta": MIN_DELTA, "select_on": "val_iou",
                   "threshold": PRIMARY_THRESHOLD, "eval_thresholds": EVAL_THRESHOLDS,
                   "threshold_reeval": {
                       "ran": RUN_THRESHOLD_REEVAL,
                       "arms": THRESHOLD_REEVAL_ARMS,
                       "reference": THRESHOLD_REEVAL_REFERENCE,
                       "thresholds": REEVAL_THRESHOLDS,
                       "results": globals().get("threshold_reeval_out"),
                   } if RUN_THRESHOLD_REEVAL else None,
                   "amp": USE_AMP, "seed": SEED, "image_size": [IMAGE_HEIGHT, IMAGE_WIDTH],
                   "static_tiling": {
                       "enabled": bool(RUN_STATIC_TILING),
                       "tile_size": [IMAGE_HEIGHT, IMAGE_WIDTH],
                       "train_tile_mode": "random" if RUN_STATIC_TILING else None,
                       "val_tile_mode": "best" if RUN_STATIC_TILING else None,
                       "test_eval": "stitched_native_then_resized" if RUN_STATIC_TILING else None,
                       "tile_size_note": "grid shape is per-frame; see per_image_results.csv n_tiles / n_tiles_with_tool",
                       "train_crop_bias": _tile_crop_bias_summary(),
                   },
                   "tracking_cascade": {
                       "enabled": bool(RUN_TRACKING_CASCADE),
                       "variant": CASCADE_VARIANT if RUN_TRACKING_CASCADE else None,                       "pad_frac": CASCADE_PAD_FRAC,
                       "horizon_s": TRACK_HORIZON_S,
                       "track_downscale": TRACK_DOWNSCALE,
                       "template_match_min": TEMPLATE_MATCH_MIN,
                       "square_roi": bool(CASCADE_SQUARE_ROI),
                       "min_box_px": CASCADE_MIN_BOX_PX,
                       "max_roi_frac": CASCADE_MAX_ROI_FRAC,
                       "min_track_px": CASCADE_MIN_TRACK_PX,
                       "combine": globals().get("CASCADE_COMBINE"),
                       "blend_w": globals().get("CASCADE_BLEND_W"),
                       "max_zoom": globals().get("CASCADE_MAX_ZOOM"),
                       "max_rois": globals().get("CASCADE_MAX_ROIS"),
                       "min_component_px": globals().get("CASCADE_MIN_COMPONENT_PX"),
                       "clip_source": CLIPS_GCS,
                       "weights_from": (globals().get("CASCADE_WEIGHTS_FROM")
                                        if RESUME_FROM_GCS else "trained in this run"),
                       "n_clips": len(globals().get("clip_manifest", {})),
                       "tier_shares": _cascade_tier_shares(),
                       "roi_zoom": _cascade_zoom_summary(),
                       "roi_cover": _cascade_cover_summary(),
                   } if RUN_TRACKING_CASCADE else None,
                   "roi_finetune": {
                       "enabled": True,
                       "epochs": ROI_FT_EPOCHS,
                       "p": ROI_FT_P,
                       "neg_p": ROI_FT_NEG_P,
                       "small_w": ROI_FT_SMALL_W,
                       "pad_range": list(ROI_FT_PAD_RANGE),
                       "lr": LR * 0.1,
                       "seeded_from": globals().get("CASCADE_WEIGHTS_FROM"),
                   } if globals().get("RUN_ROI_FINETUNE") else None,
                   "synthetic_aug": {
                       "enabled": bool(RUN_SYNTHETIC_AUG),
                       "preset": AUG_PRESET if RUN_SYNTHETIC_AUG else None,
                       "preset_spec": AUG_PRESETS.get(AUG_PRESET) if RUN_SYNTHETIC_AUG else None,
                       "levels": AUG_LEVELS if RUN_SYNTHETIC_AUG else None,
                       "p_low": AUG_P_LOW, "p_std": AUG_P_STD,
                       "aug_sweep": globals().get("aug_sweep_out"),
                       "aug_ablation": globals().get("aug_ablation_out"),
                   },
                   "sampler": "WeightedRandomSampler, global presence-count weights",
                   "sampler_notool_target": NOTOOL_SAMPLE_SHARE,
                   "sampler_notool_share_per_fold": SAMPLER_NT_SHARE,
                   "thin_cut_px": THIN_CUT,
                   "augmentation": _aug_spec(train_transform),
                   "val_augmentation": _aug_spec(val_transform)},
        "data": {"n_pairs": len(all_pairs), "n_tool": int(tool.sum()), "n_notool": int((~tool).sum()),
                 # useful for checking for tools vanishing when frame is resized
                 "n_notool_eval": int(globals().get("_n_notool_eval", sum(row["n_notool"] for row in cv_summary))),
                 "n_notool_resize_vanished": int(globals().get("_n_notool_vanished", 0)),
                 "retired_statuses": list(DELETED_STATUSES),
                 "n_videos": int(len(set(video_ids))),
                 "n_groups": int(len(set(video_group))),
                 "group_key": "union(Video ID, frame directory, near-duplicate pixels)",
                 "n_sheet_mask_contradictions": n_sheet_mask_contradictions,
                 "size_class_counts": np.bincount(size_cls, minlength=5).tolist(),
                 "thin_class_counts": np.bincount(thin_cls, minlength=3).tolist()},
        "versions": {"python": sys.version.split()[0], "torch": torch.__version__,
                     "sklearn": sklearn.__version__, "albumentations": albumentations.__version__,
                     "smp": segmentation_models_pytorch.__version__},
        "folds": {str(i): sorted(set(video_group[folds[i]].tolist())) for i in range(5)},
        "per_fold": cv_summary,
        "per_fold_best": cv_best,
        "epochs_trained": [b.get("epochs_trained") for b in cv_best],
        "mean_std": {k: [float(np.nanmean([row[k] for row in cv_summary])),
                         float(np.nanstd ([row[k] for row in cv_summary], ddof=1))]
                     for k in _summary_keys},
    }
    _tcb = final["config"]["static_tiling"]["train_crop_bias"]
    if _tcb and _tcb.get("n_tool_frames"):
        print(f"Tile crop bias : {_tcb['n_tool_frames']} tool frames, mean grid coverage "
              f"{_tcb['mean_coverage_frac']:.1%} (median {_tcb['median_coverage_frac']:.1%}), "
              f"{_tcb['single_tile_frac']:.1%} have only one tool-containing tile")

    with open("/tmp/results.json", "w") as f:
        json.dump(_json_safe(final), f, indent=2)   # NaN is not valid JSON outside Python
    !gsutil -q cp /tmp/results.json {GCS_CKPT_DIR}/results.json
    !gsutil -q cp /tmp/training_curves.png {GCS_CKPT_DIR}/training_curves.png 2>/dev/null
    print("Saved run to", GCS_CKPT_DIR)
    print("Files now in this run's folder:")
    !gsutil ls {GCS_CKPT_DIR}
else:
    print("Skipped: needs a completed RUN_CV pass (cv_summary / viz_scores not built)")


In [ ]:
if RUN_CV:
    import pandas as pd

    lookup.columns = lookup.columns.str.strip()

    score_cols = list(viz_scores.keys())
    per_image = pd.DataFrame({
        "fold":  viz_fold,
        "key":   [frame_key(img) for img, _ in viz_pairs],
        "image": [img for img, _ in viz_pairs],
        **{m: viz_scores[m] for m in score_cols},
    })

    # Subgroup columns, so the weak-region analysis needs nothing but this CSV
    per_image["fg_pct"] = per_image["tool_pixels"] / (IMAGE_HEIGHT * IMAGE_WIDTH) * 100
    per_image["size_class"] = pd.Categorical(
        np.select(
            [per_image["fg_pct"] <= 0.0,
             per_image["fg_pct"] <  4.5,
             per_image["fg_pct"] <  8.0,
             per_image["fg_pct"] < 19.0],
            ["none", "tiny", "small", "med"], default="large"),
        categories=["none", "tiny", "small", "med", "large"], ordered=True)

    _width_by_key = {frame_key(img): float(w) for (img, _), w in zip(all_pairs, width_px)}
    per_image["width_px"]   = per_image["key"].map(_width_by_key)
    per_image["thin_class"] = np.where(per_image["tool_pixels"] == 0, "none",
                              np.where(per_image["width_px"] < THIN_CUT, "thin", "wide"))

    meta_cols = ["Frame ID","Video ID","Surgery Type","Video Resolution",
                 "Robotic","Annotator","Challenges","No Tools","Timestamp"]
    missing = [c for c in meta_cols if c not in lookup.columns]
    if missing:
        print("Not in sheet, skipping:", missing)
        print("Available columns:", lookup.columns.tolist())
    meta_cols = [c for c in meta_cols if c in lookup.columns]

    per_image = per_image.join(lookup[meta_cols], on="key")
    per_image = per_image.rename(columns={"No Tools": "No Tools (sheet, unreconciled)"})
    per_image.to_csv("/tmp/per_image_results.csv", index=False)
    !gsutil -q cp /tmp/per_image_results.csv {GCS_CKPT_DIR}/per_image_results.csv
    print(f"Saved {len(per_image)} rows → {GCS_CKPT_DIR}/per_image_results.csv")
    print(per_image.groupby("size_class", observed=False)[["iou","recall","precision"]].mean())
    print(per_image.groupby("thin_class", observed=False)[["iou","recall","precision"]].mean())
    per_image.head()
else:
    print("Skipped: needs a completed RUN_CV pass (cv_summary / viz_scores not built)")


## 18. Numeric best/median/worst report
Print best, median, and worst test frames per metric across all folds, with frame and video IDs.


In [ ]:
if RUN_CV:
    def _col(target):
        """Find a sheet column by name, ignoring case and spaces (sheet headers are inconsistently formatted)."""
        norm_target = target.lower().replace(" ", "")
        for c in lookup.columns:
            if c.lower().replace(" ", "") == norm_target:
                return c
        return None

    FRAME_COL = _col("Frame ID")
    VIDEO_COL = _col("Video ID")
    print("Resolved →  frame:", FRAME_COL, "| video:", VIDEO_COL)

    _has_tool = np.array(viz_scores["gt_empty"]) == 0
    _pred_empty = np.array(viz_scores["pred_empty"]) == 1

    def frame_tag(idx):
        """Human-readable label for a frame: its sheet Frame ID and Video ID if available, else its filename."""
        k = frame_key(viz_pairs[idx][0])
        if k in lookup.index and FRAME_COL and VIDEO_COL:
            return f"F{lookup.at[k, FRAME_COL]}·{lookup.at[k, VIDEO_COL]}"
        return Path(viz_pairs[idx][0]).stem

    def report_metric(m, higher_better, n=3):
        """Print the mean/median for metric `m`, plus its best-n and worst-n frames by that metric."""
        vals  = np.array(viz_scores[m], dtype=float)
        # Tool-absent frames score a free IoU/Dice of 1.0, so they are excluded here.
        ok    = np.where(~np.isnan(vals) & _has_tool)[0]
        order = ok[np.argsort(vals[ok])]
        best  = (order[::-1] if higher_better else order)[:n]
        worst = (order       if higher_better else order[::-1])[:n]
        med   = order[len(order) // 2]
        print(f"\n{m.upper():10s} mean {vals[ok].mean():7.4f} ± {vals[ok].std(ddof=1):.4f} | "
              f"median {np.median(vals[ok]):.4f}  (n={len(ok)} of {_has_tool.sum()} tool-present)")
        print("   best :", "  ".join(f"{vals[j]:.3f} [{frame_tag(j)}]" for j in best))
        print("   med  :", f"{vals[med]:.3f} [{frame_tag(med)}]")
        print("   worst:", "  ".join(f"{vals[j]:.3f} [{frame_tag(j)}]" for j in worst))

    print("="*16, "Per-image distribution (pooled test frames, all folds)", "="*16)
    print(f"Tool-present frames: {_has_tool.sum()}  |  tool-absent: {(~_has_tool).sum()}")
    print(f"Total misses (tool present, nothing predicted): {int((_has_tool & _pred_empty).sum())} "
          f"({(_has_tool & _pred_empty).mean()*100:.1f}% of tool-present frames)")
    print("\nOverlap (higher = better) - lead with IoU; Dice needs >~0.9 to mean much:")
    for m, hb in [("iou", True), ("dice", True), ("precision", True), ("recall", True)]:
        report_metric(m, hb)
    print("\nBoundary (lower = better) - prefer MSD/ASD; HD is the noisiest.")
    print("Note the n: these are undefined for a total miss, so the worst failures are excluded.")
    for m, hb in [("hd95", False), ("hd", False), ("msd", False), ("asd", False)]:
        report_metric(m, hb)

    fp_nt = np.array(viz_scores["fp_pixels"], float)[~_has_tool]
    if len(fp_nt):
        print(f"\nNO-TOOL FRAMES  n={len(fp_nt)}  clean (zero FP): {(fp_nt == 0).mean()*100:.1f}%  "
              f"| mean FP area {fp_nt.mean():.0f}px  | worst {fp_nt.max():.0f}px")
else:
    print("Skipped: needs a completed RUN_CV pass (cv_summary / viz_scores not built)")


## 19. Image visualisation
Overlay predictions on best, median, and worst test frames per metric. Each frame uses its own fold checkpoint.

Set `METRIC` and `TOP_N` in the configuration; includes a no-tool false-positive view.

All figures are also saved to `last_cell_figs/` and bundled into `last_cell_figs.zip` (the folder is wiped and rebuilt each run).

Saves every prediction mask (`.png`) and syncs it to the GCS directory. The model runs at `IMAGE_HEIGHT` × `IMAGE_WIDTH`, so each mask is resized back to its source frame's resolution with nearest-neighbour before writing, and coloured with `FG_COLOUR` from the labelmap.


In [ ]:
if RUN_CV:
    import matplotlib.pyplot as plt
    from matplotlib.patches import Patch
    import os, shutil
    import cv2
    import numpy as np
    import torch
    from pathlib import Path
    from IPython.display import FileLink
    from tqdm.auto import tqdm

    EXCLUDE_EMPTY = True   # drop no-tool (empty GT) frames from best / median / worst rankings

    # --- figure saving setup ---
    FIG_DIR = "last_cell_figs"
    shutil.rmtree(FIG_DIR, ignore_errors=True)   # start clean each run
    os.makedirs(FIG_DIR, exist_ok=True)
    _fig_i = 0

    mean_np, std_np = np.array(_mean), np.array(_std)
    _model_cache = {}

    def load_ckpt_model(path):
        """Load (and cache) the model for a fold checkpoint, so each checkpoint is only loaded once."""
        if path not in _model_cache:
            model = build_model(ARCHITECTURE, ENCODER, ENCODER_WEIGHTS).to(device)
            model.load_state_dict(
                torch.load(path, map_location=device, weights_only=True)["model_state"])
            model.eval()
            _model_cache[path] = model
        return _model_cache[path]

    def predict_frame(img_path, mask_path, ckpt_path):
        """Run one frame through its fold's model; returns the display image, ground-truth mask,
        prediction mask, and source (H, W) - all at IMAGE_HEIGHT x IMAGE_WIDTH for consistent
        plotting/scoring across arms - plus a 5th value: the native-resolution prediction for
        static tiling (None for every other arm).
        """
        image = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        if mask_path == EMPTY_MASK:                      # guard: cv2.imread warns on every miss
            mask = np.zeros(image.shape[:2], dtype=np.float32)
        else:
            mask_rgb = cv2.imread(mask_path)
            mask = (np.zeros(image.shape[:2], dtype=np.float32) if mask_rgb is None
                    else rgb_mask_to_binary(cv2.cvtColor(mask_rgb, cv2.COLOR_BGR2RGB)))
        if RUN_TRACKING_CASCADE:
            _clip, _tidx, _fps, _lead = clip_for(img_path)
            probs_native = cascade_predict(load_ckpt_model(ckpt_path), image, _clip, _tidx,
                                           _fps, CASCADE_VARIANT)
            img_disp  = cv2.resize(image, (IMAGE_WIDTH, IMAGE_HEIGHT), interpolation=cv2.INTER_AREA) / 255.0
            mask_disp = cv2.resize(mask,  (IMAGE_WIDTH, IMAGE_HEIGHT), interpolation=cv2.INTER_NEAREST)
            probs_512 = cv2.resize(probs_native, (IMAGE_WIDTH, IMAGE_HEIGHT), interpolation=cv2.INTER_AREA)
            pred = (probs_512 > PRIMARY_THRESHOLD).astype(np.float32)
            pred_native = (probs_native > PRIMARY_THRESHOLD).astype(np.uint8)
            return img_disp, mask_disp, pred, image.shape[:2], pred_native
        if RUN_STATIC_TILING:
            probs_native = stitch_predict(load_ckpt_model(ckpt_path), image)
            img_disp  = cv2.resize(image, (IMAGE_WIDTH, IMAGE_HEIGHT), interpolation=cv2.INTER_AREA) / 255.0
            mask_disp = cv2.resize(mask,  (IMAGE_WIDTH, IMAGE_HEIGHT), interpolation=cv2.INTER_NEAREST)
            probs_512 = cv2.resize(probs_native, (IMAGE_WIDTH, IMAGE_HEIGHT), interpolation=cv2.INTER_AREA)
            pred = (probs_512 > PRIMARY_THRESHOLD).astype(np.float32)
            pred_native = (probs_native > PRIMARY_THRESHOLD).astype(np.uint8)
            return img_disp, mask_disp, pred, image.shape[:2], pred_native
        aug = val_transform(image=image, mask=mask)
        input_tensor = aug["image"].unsqueeze(0).to(device)
        with torch.no_grad():
            pred = (load_ckpt_model(ckpt_path)(input_tensor).sigmoid() > PRIMARY_THRESHOLD).float().cpu()[0, 0].numpy()
        img_disp = np.clip(aug["image"].permute(1, 2, 0).numpy() * std_np + mean_np, 0, 1)
        return img_disp, aug["mask"].numpy(), pred, image.shape[:2], None

    def overlay(img, pred, alpha=0.5):
        """Blend the predicted mask over the image in red, for a quick visual check."""
        out = img.copy()
        out[pred == 1] = (1 - alpha) * out[pred == 1] + alpha * np.array([1, 0, 0])
        return np.clip(out, 0, 1)

    def confusion_rgb(pred, gt):
        """Colour each pixel by TP/FP/FN so one image shows where predictions went right or wrong."""
        # TP = green, FP = red, FN = blue (see colour key added to each figure)
        out = np.zeros((*pred.shape, 3))
        out[(pred == 1) & (gt == 1)] = [0, 1, 0]   # TP green
        out[(pred == 1) & (gt == 0)] = [1, 0, 0]   # FP red
        out[(pred == 0) & (gt == 1)] = [0, 0, 1]   # FN blue
        return out

    # Colour keys reused across every figure so the panels are self-explanatory.
    OVERLAY_KEY = [Patch(facecolor="red", edgecolor="none", label="predicted instrument")]
    CONFUSION_KEY = [
        Patch(facecolor="green", edgecolor="none", label="TP - correct"),
        Patch(facecolor="red",   edgecolor="none", label="FP - false alarm"),
        Patch(facecolor="blue",  edgecolor="none", label="FN - missed"),
    ]

    def frame_label(img_path):
        """Human-readable label for a frame - its sheet Frame/Video ID if available, else its filename."""
        k = frame_key(img_path)
        if k in lookup.index and FRAME_COL and VIDEO_COL:
            return f"F{lookup.at[k, FRAME_COL]} · {lookup.at[k, VIDEO_COL]}"
        return Path(img_path).stem

    pred_masks_tmp = PRED_MASK_DIR
    _reused = _made = 0
    for idx, (img_path, mask_path) in enumerate(tqdm(viz_pairs, desc="Saving Masks")):
        out_path = pred_mask_path(img_path)
        if out_path.exists():
            _reused += 1
            continue
        _, _, pred, (src_h, src_w), pred_native = predict_frame(img_path, mask_path, viz_ckpt[idx])
        if pred_native is None:
            pred_native = cv2.resize(pred.astype(np.uint8), (src_w, src_h),
                                     interpolation=cv2.INTER_NEAREST)
        save_pred_mask(img_path, pred_native)
        _made += 1
    print(f"Prediction masks: {_reused} cached during evaluation, {_made} generated here")

    print("Syncing prediction masks to GCS...")
    !gsutil -m -q rsync -r /tmp/PredictedMasks {GCS_CKPT_DIR}/PredictedMasks
    print(f"Successfully saved all prediction masks to {GCS_CKPT_DIR}/PredictedMasks")

    def show_cases(indices, metric, title):
        """Plot one row per frame (input / ground truth / prediction overlay / TP-FP-FN) and save the figure."""
        global _fig_i
        n = len(indices)
        if n == 0:
            print(f"(no frames for: {title})"); return
        fig, axes = plt.subplots(n, 4, figsize=(16, 4 * n))
        if n == 1: axes = axes[None, :]
        for c, t in enumerate(["Input", "Ground truth", "Prediction overlay", "TP(g)/FP(r)/FN(b)"]):
            axes[0, c].set_title(t, fontsize=10)
        for row, idx in enumerate(indices):
            img_path, mask_path = viz_pairs[idx]
            img, gt, pred, _, _ = predict_frame(img_path, mask_path, viz_ckpt[idx])
            axes[row, 0].imshow(img)
            axes[row, 1].imshow(gt, cmap="gray")
            axes[row, 2].imshow(overlay(img, pred))
            axes[row, 3].imshow(confusion_rgb(pred, gt))
            score = viz_scores[metric][idx]
            axes[row, 0].set_ylabel(f"{frame_label(img_path)}\n{metric}={score:.3f}", fontsize=9)
            for c in range(4):
                axes[row, c].set_xticks([]); axes[row, c].set_yticks([])
        # colour key on the first row of the two colour-coded columns
        axes[0, 2].legend(handles=OVERLAY_KEY,   loc="upper right", fontsize=7, framealpha=0.85)
        axes[0, 3].legend(handles=CONFUSION_KEY, loc="upper right", fontsize=7, framealpha=0.85)
        fig.suptitle(title, fontsize=13)
        plt.tight_layout()
        # --- save figure ---
        fname = f"{_fig_i:02d}_{title}.png".replace(" ", "_").replace("/", "-")
        fig.savefig(os.path.join(FIG_DIR, fname), dpi=150, bbox_inches="tight")
        _fig_i += 1
        plt.show()

    def show_metric(metric):
        """Show the worst-N, median, and best-N frames for one metric."""
        higher_better = metric in ("iou", "dice", "precision", "recall")
        vals  = np.array(viz_scores[metric], dtype=float)
        valid = ~np.isnan(vals)
        if EXCLUDE_EMPTY:
            valid &= np.array(viz_scores["tool_pixels"]) > 0   # keep only tool-present frames
        order = np.where(valid)[0]
        order = order[np.argsort(vals[order])]                 # ascending
        best   = (order[::-1] if higher_better else order)[:TOP_N]
        worst  = (order       if higher_better else order[::-1])[:TOP_N]
        median = [order[len(order) // 2]]
        show_cases(list(worst),  metric, f"Worst {TOP_N} by {metric.upper()}")
        show_cases(median,       metric, f"Median by {metric.upper()}")
        show_cases(list(best),   metric, f"Best {TOP_N} by {metric.upper()}")

    # --- best / median / worst by chosen metric(s) ---
    metrics = ["iou", "dice", "precision", "recall", "hd95", "hd", "msd", "asd"]
    for m in metrics:
        show_metric(m)

    # --- no-tool frames with the most false positives ---
    tool_px, fp_px = np.array(viz_scores["tool_pixels"]), np.array(viz_scores["fp_pixels"])
    notool = np.where(tool_px == 0)[0]
    if len(notool):
        fp_rank = notool[np.argsort(fp_px[notool])[::-1]][:TOP_N]
        show_cases(list(fp_rank), "fp_pixels", "No-tool frames: most false positives")

    # --- tool-present frames where the model predicted nothing at all ---
    gt_e, pr_e = np.array(viz_scores["gt_empty"]), np.array(viz_scores["pred_empty"])
    misses = np.where((gt_e == 0) & (pr_e == 1))[0]
    if len(misses):
        biggest = misses[np.argsort(tool_px[misses])[::-1]][:TOP_N]
        show_cases(list(biggest), "iou",
                   f"Total misses: tool present, nothing predicted ({len(misses)} frames)")
    else:
        print("No total misses in the pooled test set.")

    # --- zip everything that was saved ---
    zip_path = shutil.make_archive(FIG_DIR, "zip", FIG_DIR)
    print(f"Saved {_fig_i} figures → {zip_path}")
    FileLink(zip_path)
else:
    print("Skipped: needs a completed RUN_CV pass (cv_summary / viz_scores not built)")


## 20. Release the runtime
Upload any remaining local artefacts, verify `GCS_CKPT_DIR`, then delete the Colab Enterprise runtime (`gcloud colab runtimes delete`) so billing stops. Skips shutdown if anything is missing. Set `AUTO_SHUTDOWN = False` to report only.


In [ ]:
# === Release the runtime once everything is safely in GCS ===
AUTO_SHUTDOWN  = True   # False: report GCS status and stay up
SHUTDOWN_GRACE = 60     # seconds to interrupt before delete

import json, os, subprocess, time, urllib.request

_expected = ["folds.json"]
if RUN_SWEEP: _expected += ["sweep_results.json"]
if LOSS == "bce_tversky" and RUN_LOSS_BALANCE_SWEEP: _expected += ["loss_balance_results.json", "loss_balance_curve.png"]
if RUN_AUG_SWEEP: _expected += ["aug_sweep_results.json", "aug_sweep_curve.png", "aug_sweep_tradeoff.png"]
if RUN_AUG_ABLATION: _expected += ["aug_ablation_results.json", "aug_ablation_curve.png", "aug_ablation_tradeoff.png"]
if RUN_CV:
    _expected += [f"best_fold{i}.pt" for i in range(5)]
    _expected += [f"fold{i}_scores.json" for i in range(5)]
    _expected += ["results.json", "per_image_results.csv"]
    if globals().get("EXPORT_PRED_MASKS", True):
        _expected += ["PredictedMasks"]
if RUN_POWER: _expected += ["power_analysis.json", "power_curve.png", "power_curve_size.png"]
if RUN_THRESHOLD_REEVAL: _expected += ["threshold_reeval.json"]

if "zip_path" in globals() and os.path.exists(zip_path):
    !gsutil -q cp {zip_path} {GCS_CKPT_DIR}/last_cell_figs.zip
    _expected += ["last_cell_figs.zip"]

_ls = subprocess.run(["gsutil", "ls", GCS_CKPT_DIR], capture_output=True, text=True)
_present = {p.rstrip("/").rsplit("/", 1)[-1] for p in _ls.stdout.split()}
_missing = [f for f in _expected if f not in _present]

print("Run folder:", GCS_CKPT_DIR)
for f in _expected:
    print(f"  {'ok     ' if f in _present else 'MISSING'}  {f}")

def _release_runtime():
    """Find and delete the sole RUNNING Colab Enterprise runtime, so billing stops."""
    req = urllib.request.Request(
        "http://metadata.google.internal/computeMetadata/v1/project/project-id",
        headers={"Metadata-Flavor": "Google"},
    )
    project = urllib.request.urlopen(req, timeout=5).read().decode()
    regions = (
        "us-central1 us-east1 us-east4 us-west1 us-west4 "
        "europe-west1 europe-west2 europe-west3 europe-west4 "
        "asia-east1 asia-northeast1 asia-southeast1"
    ).split()
    hits = []
    for region in regions:
        try:
            rows = json.loads(subprocess.check_output(
                ["gcloud", "colab", "runtimes", "list",
                 f"--project={project}", f"--region={region}", "--format=json"],
                text=True, stderr=subprocess.DEVNULL,
            ) or "[]")
        except subprocess.CalledProcessError:
            continue
        for r in rows:
            if "RUNNING" in str(r.get("runtimeState") or "").upper():
                hits.append((region, (r.get("name") or "").rsplit("/", 1)[-1]))
    if len(hits) != 1:
        raise RuntimeError(f"need exactly one RUNNING runtime, found {hits}")
    region, rid = hits[0]
    print(f"Deleting {rid} in {region}")
    subprocess.check_call([
        "gcloud", "colab", "runtimes", "delete", rid,
        f"--project={project}", f"--region={region}", "--quiet",
    ])

def _status(print_msg, kind, title, lines=None):
    """Print for the Colab log and notify() with the same event, in one call."""
    print(print_msg)
    notify(kind, title, lines)

if _ls.returncode != 0:
    _status(f"\nCould not list GCS (exit {_ls.returncode}); NOT shutting down.",
            "error", "shutdown skipped · could not list GCS",
            [f"gsutil exit {_ls.returncode}", "still billing"])
elif _missing:
    _status(f"\nMissing in GCS: {_missing}. NOT shutting down.",
            "error", f"shutdown skipped · {len(_missing)} missing",
            [", ".join(_missing[:8]), "still billing"])
elif not AUTO_SHUTDOWN:
    _status("\nAll artefacts present. AUTO_SHUTDOWN=False - still billing.",
            "finish", "artefacts ok · AUTO_SHUTDOWN off", [GCS_CKPT_DIR])
else:
    _status(f"\nAll artefacts present. Releasing in {SHUTDOWN_GRACE}s (interrupt to cancel).",
            "finish", f"run complete · shutdown in {SHUTDOWN_GRACE}s", [GCS_CKPT_DIR])
    time.sleep(SHUTDOWN_GRACE)
    try:
        _release_runtime()
    except Exception as e:
        _status(f"\n*** COULD NOT RELEASE: {type(e).__name__}: {e}\n*** Still billing - stop it by hand.",
                "error", "SHUTDOWN FAILED · still billing", [f"{type(e).__name__}: {e}"])
